In [1]:
# Insert the path for the data folder.
data = '../book/data'

# Preparing Data for Analysis
## Pandas Code Structure

In [2]:
import pandas as pd

data_url = 'https://raw.githubusercontent.com/jorammutenge/learn-rust/refs/heads/main/sample_sales.csv'

df = (pd.read_csv(data_url, parse_dates=['date'])
      .drop(columns='Account Name')
      .rename(columns=lambda c: c.title().replace(' ','_'))
      .rename(columns=dict(Account_Number='Customer_ID'))
      )
df

,Customer_ID,Sku,Category,Quantity,Unit_Price,Ext_Price,Date
0,803666,HX-24728,Hat,1,98.98,98.98,2014-09-28 11:56:02
1,64898,LK-02338,Sweater,9,34.80,313.20,2014-04-24 16:51:22
2,423621,ZC-07383,Sweater,12,60.24,722.88,2014-09-17 17:26:22
3,137865,QS-76400,Sweater,5,15.25,76.25,2014-01-30 07:34:02
4,435433,RU-25060,Sweater,19,51.83,984.77,2014-08-24 06:18:12
...,...,...,...,...,...,...,...
995,29068,FT-50146,Sweater,2,46.48,92.96,2014-08-10 16:20:32
996,77116,IC-59308,Socks,19,29.25,555.75,2013-11-20 13:32:45
997,23749,IC-59308,Socks,18,54.79,986.22,2014-03-10 08:11:59
998,172519,RU-25060,Sweater,15,62.53,937.95,2014-04-11 02:50:03


In [3]:
type(df['Quantity'])

pandas.Series

In [4]:
df['Quantity'].sum()

np.int64(10565)

In [5]:
df['Quantity'].agg('sum')

np.int64(10565)

In [6]:
df.drop(columns='Quantity')

,Customer_ID,Sku,Category,Unit_Price,Ext_Price,Date
0,803666,HX-24728,Hat,98.98,98.98,2014-09-28 11:56:02
1,64898,LK-02338,Sweater,34.80,313.20,2014-04-24 16:51:22
2,423621,ZC-07383,Sweater,60.24,722.88,2014-09-17 17:26:22
3,137865,QS-76400,Sweater,15.25,76.25,2014-01-30 07:34:02
4,435433,RU-25060,Sweater,51.83,984.77,2014-08-24 06:18:12
...,...,...,...,...,...,...
995,29068,FT-50146,Sweater,46.48,92.96,2014-08-10 16:20:32
996,77116,IC-59308,Socks,29.25,555.75,2013-11-20 13:32:45
997,23749,IC-59308,Socks,54.79,986.22,2014-03-10 08:11:59
998,172519,RU-25060,Sweater,62.53,937.95,2014-04-11 02:50:03


## Profiling: Distributions
### Histograms and Frequencies

In [7]:
import pandas as pd
import hvplot.pandas

data_url = 'https://raw.githubusercontent.com/jorammutenge/learn-rust/refs/heads/main/sample_sales.csv'

clothing_store_sales = (pd.read_csv(data_url, parse_dates=['date'])
 .rename(columns=lambda c: c.title().replace(' ','_'))
 .rename(columns=dict(Account_Number='Customer_ID'))
 )
clothing_store_sales

,Customer_ID,Account_Name,Sku,Category,Quantity,Unit_Price,Ext_Price,Date
0,803666,Fritsch-Glover,HX-24728,Hat,1,98.98,98.98,2014-09-28 11:56:02
1,64898,O'Conner Inc,LK-02338,Sweater,9,34.80,313.20,2014-04-24 16:51:22
2,423621,Beatty and Sons,ZC-07383,Sweater,12,60.24,722.88,2014-09-17 17:26:22
3,137865,"Gleason, Bogisich and Franecki",QS-76400,Sweater,5,15.25,76.25,2014-01-30 07:34:02
4,435433,Morissette-Heathcote,RU-25060,Sweater,19,51.83,984.77,2014-08-24 06:18:12
...,...,...,...,...,...,...,...,...
995,29068,Brekke and Sons,FT-50146,Sweater,2,46.48,92.96,2014-08-10 16:20:32
996,77116,Lang-Wunsch,IC-59308,Socks,19,29.25,555.75,2013-11-20 13:32:45
997,23749,Bogisich and Sons,IC-59308,Socks,18,54.79,986.22,2014-03-10 08:11:59
998,172519,"Kutch, Cormier and Harber",RU-25060,Sweater,15,62.53,937.95,2014-04-11 02:50:03


In [8]:
(clothing_store_sales
 ['Category']
 .value_counts()
 )

Category
Sweater    526
Socks      297
Hat        177
Name: count, dtype: int64

In [9]:
(clothing_store_sales
 .groupby('Category')
 .size()
 )

Category
Hat        177
Socks      297
Sweater    526
dtype: int64

In [10]:
(clothing_store_sales
 .groupby('Category')
 .agg(Count=('Category','size'))
 )

,Count
Category,
Hat,177
Socks,297
Sweater,526


In [11]:
(clothing_store_sales
 .groupby('Category')
 .size()
 .sort_values(ascending=False)
 .to_frame('Count') # <1>
 .hvplot.bar(x='Category', y='Count',
             title='Clothing type',
             xlabel='', ylabel='Quantity')
 .opts(title='Frequency plot of clothing inventory')
 )

:Bars   [Category]   (Count)

In [12]:
(clothing_store_sales
 ['Quantity']
 .hvplot.hist(title='Quantity distribution',
              xlabel='',
              ylabel='No. of transactions')
 .opts(title='Transactions by quantity')
 )

:Histogram   [Quantity]   (Count)

In [13]:
(clothing_store_sales
 .groupby('Customer_ID')
 .agg(Transactions=('Date','count')) # <1>
 .reset_index()
 .groupby('Transactions')
 .agg(Customers=('Customer_ID','size'))
 )

,Customers
Transactions,
1,508
2,166
3,46
4,3
5,2


### Binning

In [14]:
import numpy as np

(clothing_store_sales
 .assign(Spend_Bin=lambda d: pd.cut(
            d['Ext_Price'],
            bins=[-np.inf, 100, 500, np.inf],
            labels=['up to 100', '100 to 500', '500+']),
         Spend_Category=lambda d: pd.cut(
            d['Ext_Price'],
            bins=[-np.inf, 100, 500, np.inf],
            labels=['Small', 'Medium', 'Large']))
 .groupby(['Spend_Bin', 'Spend_Category'], observed=True) # <1>
 .agg(Customers=('Ext_Price', 'size'))
 )

,,Customers
Spend_Bin,Spend_Category,
up to 100,Small,114
100 to 500,Medium,420
500+,Large,466


In [15]:
(clothing_store_sales
 .assign(Price_Bin=np.log(pd.col('Unit_Price')))
 .groupby('Price_Bin')
 .agg(Customers=('Customer_ID','count'))
 )

,Customers
Price_Bin,
2.303585,1
2.312535,1
2.314514,1
2.322388,1
2.326302,1
...,...
4.598649,1
4.603469,1
4.603869,1


### n-Tiles

In [16]:
df = pd.DataFrame({'Total_Spend': [29.94, 59.94, 71.94, 77.94, 89.94,
                                   119.94, 140.94, 209.94, 335.88,
                                   359.94, 539.94, 599.94]})
df

,Total_Spend
0,29.94
1,59.94
2,71.94
3,77.94
4,89.94
5,119.94
6,140.94
7,209.94
8,335.88
9,359.94


In [17]:
(df
 .sort_values('Total_Spend')
 .reset_index(drop=True) # <1>
 .rename_axis('idx') # <2>
 .reset_index() # <3>
 .assign(N_Tile=pd.col('idx').mul(10).floordiv(pd.col('idx').count()).add(1))
 .filter(['Total_Spend','N_Tile'])
)

,Total_Spend,N_Tile
0,29.94,1
1,59.94,1
2,71.94,2
3,77.94,3
4,89.94,4
5,119.94,5
6,140.94,6
7,209.94,6
8,335.88,7
9,359.94,8


In [18]:
(clothing_store_sales
 .sort_values('Ext_Price')
 .reset_index(drop=True)
 .rename_axis('idx')
 .reset_index()
 .assign(N_Tile=pd.col('idx').mul(10).floordiv(pd.col('idx').count()).add(1))
 .groupby('N_Tile')
 .agg(Lower_Bound=('Ext_Price', 'min'),
      Upper_Bound=('Ext_Price', 'max'),
      Orders=('Date', 'size'))
 .reset_index()
 .sort_values('N_Tile')
)

,N_Tile,Lower_Bound,Upper_Bound,Orders
0,1,11.13,95.05,100
1,2,95.13,172.65,100
2,3,172.85,246.84,100
3,4,247.92,349.12,100
4,5,350.00,456.04,100
5,6,456.64,572.04,100
6,7,575.82,749.20,100
7,8,750.00,964.32,100
8,9,968.16,1224.00,100
9,10,1227.00,1958.60,100


In [19]:
(clothing_store_sales
 .assign(Year=pd.col('Date').dt.year)
 .loc[pd.col('Year').eq(2014)]
 .groupby(['Sku', 'Year'], as_index=False) # <1>
 .agg(Ext_Price=('Ext_Price', 'sum'))
 .assign(Pct_Rank=(pd.col('Ext_Price').rank(method='min') - 1) / (pd.col('Ext_Price').count() - 1))
 .sort_values('Pct_Rank')
)

,Sku,Year,Ext_Price,Pct_Rank
3,HX-24728,2014,30086.75,0.000000
5,LK-02338,2014,33828.48,0.111111
2,GS-95639,2014,35141.11,0.222222
8,XX-25746,2014,38738.46,0.333333
1,FT-50146,2014,39826.58,0.444444
0,AE-95093,2014,44515.29,0.555556
7,RU-25060,2014,48418.29,0.666667
4,IC-59308,2014,48471.12,0.777778
6,QS-76400,2014,51783.84,0.888889
9,ZC-07383,2014,54538.08,1.000000


In [20]:
(clothing_store_sales
 .assign(Year=pd.col('Date').dt.year)
 .groupby(['Sku', 'Year'], as_index=False)
 .agg(Ext_Price=('Ext_Price', 'sum'))
 .assign(Pct_Rank=lambda d: (d.groupby('Year')['Ext_Price'].rank(method='min') - 1)
                             .div(d.groupby('Year')['Year'].transform('size') - 1)) # <1>
 .sort_values(['Year', 'Pct_Rank'])
)

,Sku,Year,Ext_Price,Pct_Rank
18,ZC-07383,2013,6945.68,0.000000
4,GS-95639,2013,9569.33,0.111111
6,HX-24728,2013,9822.69,0.222222
0,AE-95093,2013,12631.68,0.333333
2,FT-50146,2013,13732.11,0.444444
14,RU-25060,2013,14539.41,0.555556
12,QS-76400,2013,17602.23,0.666667
8,IC-59308,2013,18841.40,0.777778
10,LK-02338,2013,20501.30,0.888889
16,XX-25746,2013,20646.11,1.000000


## Profiling: Data Quality
### Detecting Duplicates

In [21]:
student_grades = pd.DataFrame({
    'Student': ['Spencer','Emily','Hannah','Aria','Mona','Paige','Aria'],
    'Grade': [93, 88, 73, 69, 55, 89, 69]
})
student_grades

,Student,Grade
0,Spencer,93
1,Emily,88
2,Hannah,73
3,Aria,69
4,Mona,55
5,Paige,89
6,Aria,69


In [22]:
student_grades.duplicated(keep=False) # <1>

0    False
1    False
2    False
3     True
4    False
5    False
6     True
dtype: bool

In [23]:
(student_grades
 .loc[student_grades.duplicated(keep=False)]
 )

,Student,Grade
3,Aria,69
6,Aria,69


In [24]:
(student_grades
 .loc[lambda d: d.duplicated(keep=False)] # <1>
 )

,Student,Grade
3,Aria,69
6,Aria,69


In [25]:
(student_grades
 .assign(Len=lambda d: d.groupby(['Student','Grade'])['Student'].transform('size'))
 .loc[pd.col('Len').gt(1)]
 )

,Student,Grade,Len
3,Aria,69,2
6,Aria,69,2


### Deduplication with Group By and Drop Duplicates

In [26]:
employees = pd.DataFrame({
    'Name': ['Joram Mutenge','Marion Daigle','Ashwin Bhakre','Eleonora Kiziv',
             'Sasha Crespi','Paul Lee','Oliver Haynes','David Nesbitt'],
    'Employee_ID': ['E0001','E0002','E0003','E0004','E0005','E0006','E0007','E0008']})
employees

,Name,Employee_ID
0,Joram Mutenge,E0001
1,Marion Daigle,E0002
2,Ashwin Bhakre,E0003
3,Eleonora Kiziv,E0004
4,Sasha Crespi,E0005
5,Paul Lee,E0006
6,Oliver Haynes,E0007
7,David Nesbitt,E0008


In [27]:
projects = pd.DataFrame({
    'Employee_ID': ['E0004','E0002','E0004','E0003','E0004','E0005','E0003','E0004','E0001','E0006'],
    'Contributed': ['Yes','Yes','Yes','Yes','Yes','Yes','Yes','Yes','Yes','Yes']})
projects

,Employee_ID,Contributed
0,E0004,Yes
1,E0002,Yes
2,E0004,Yes
3,E0003,Yes
4,E0004,Yes
5,E0005,Yes
6,E0003,Yes
7,E0004,Yes
8,E0001,Yes
9,E0006,Yes


In [28]:
(employees
 .merge(projects, on='Employee_ID', how='inner')
 )

,Name,Employee_ID,Contributed
0,Joram Mutenge,E0001,Yes
1,Marion Daigle,E0002,Yes
2,Ashwin Bhakre,E0003,Yes
3,Ashwin Bhakre,E0003,Yes
4,Eleonora Kiziv,E0004,Yes
5,Eleonora Kiziv,E0004,Yes
6,Eleonora Kiziv,E0004,Yes
7,Eleonora Kiziv,E0004,Yes
8,Sasha Crespi,E0005,Yes
9,Paul Lee,E0006,Yes


In [29]:
(employees
 .merge(projects, on='Employee_ID', how='inner')
 .drop_duplicates()
 )

,Name,Employee_ID,Contributed
0,Joram Mutenge,E0001,Yes
1,Marion Daigle,E0002,Yes
2,Ashwin Bhakre,E0003,Yes
4,Eleonora Kiziv,E0004,Yes
8,Sasha Crespi,E0005,Yes
9,Paul Lee,E0006,Yes


In [30]:
(employees
 .merge(projects, on='Employee_ID', how='inner')
 .pipe(lambda d: d.groupby(list(d.columns), as_index=False).first()) # <1>
 )

,Name,Employee_ID,Contributed
0,Ashwin Bhakre,E0003,Yes
1,Eleonora Kiziv,E0004,Yes
2,Joram Mutenge,E0001,Yes
3,Marion Daigle,E0002,Yes
4,Paul Lee,E0006,Yes
5,Sasha Crespi,E0005,Yes


## Preparing: Data Cleaning
### Cleaning Data with np.select and np.where

In [31]:
countries = pd.DataFrame({'Country': ['United States', 'USA', 'U.S.', 'CAN', 'JPN']})
countries

,Country
0,United States
1,USA
2,U.S.
3,CAN
4,JPN


In [32]:
(countries
 .assign(Country_Cleaned=lambda d: np.select( 
     [d['Country'].isin(['United States', 'U.S.'])],
     ['USA'],
     default=d['Country']))
 )

,Country,Country_Cleaned
0,United States,USA
1,USA,USA
2,U.S.,USA
3,CAN,CAN
4,JPN,JPN


In [33]:
survey_responses = pd.DataFrame({
    'Response_ID': [10001, 10002, 10003, 10004, 10005, 10006, 10007, 10008, 10009, 10010],
    'Gender': ['M', 'M', 'F', 'M', 'F', 'M', 'F', 'M', 'F', 'M'],
    'Country': ['USA', 'JPN', 'CAN', 'USA', 'GBR', 'AUS', 'IND', 'DEU', 'BRA', 'FRA'],
    'Premium': ['Yes', 'No', 'Yes', 'No', 'No', 'Yes', 'No', 'Yes', 'No', 'Yes'],
    'Likelihood': [5, 3, 10, 7, 9, 8, 2, 9, 5, 10]
})
survey_responses

,Response_ID,Gender,Country,Premium,Likelihood
0,10001,M,USA,Yes,5
1,10002,M,JPN,No,3
2,10003,F,CAN,Yes,10
3,10004,M,USA,No,7
4,10005,F,GBR,No,9
5,10006,M,AUS,Yes,8
6,10007,F,IND,No,2
7,10008,M,DEU,Yes,9
8,10009,F,BRA,No,5
9,10010,M,FRA,Yes,10


In [34]:
(survey_responses
 .filter(['Response_ID','Likelihood'])
 .assign(Response_Type=lambda d: np.select(
     [d['Likelihood'] <= 6, d['Likelihood'] <= 8],
     ['Detractor', 'Passive'],
     default='Promoter'))
 )

,Response_ID,Likelihood,Response_Type
0,10001,5,Detractor
1,10002,3,Detractor
2,10003,10,Promoter
3,10004,7,Passive
4,10005,9,Promoter
5,10006,8,Passive
6,10007,2,Detractor
7,10008,9,Promoter
8,10009,5,Detractor
9,10010,10,Promoter


In [35]:
(survey_responses
 .filter(['Response_ID','Likelihood'])
 .assign(Response_Type=lambda d: np.select(
     [d['Likelihood'].isin(range(0,7)), d['Likelihood'].isin([7,8])], # <1>
     ['Detractor', 'Passive'],
     default='Promoter'))
 )

,Response_ID,Likelihood,Response_Type
0,10001,5,Detractor
1,10002,3,Detractor
2,10003,10,Promoter
3,10004,7,Passive
4,10005,9,Promoter
5,10006,8,Passive
6,10007,2,Detractor
7,10008,9,Promoter
8,10009,5,Detractor
9,10010,10,Promoter


In [36]:
(survey_responses
 .assign(Response_Type=lambda d: np.select(
     [(d['Likelihood'] <= 6) & (d['Country'] == 'USA') & (d['Premium'] == 'Yes'),
      (d['Likelihood'] >= 9) & (d['Country'].isin(['AUS','DEU']) | (d['Premium'] == 'No'))],
     ['USA premium detractor', 'Some desired label'],
     default='Promoter'))
 )

,Response_ID,Gender,Country,Premium,Likelihood,Response_Type
0,10001,M,USA,Yes,5,USA premium detractor
1,10002,M,JPN,No,3,Promoter
2,10003,F,CAN,Yes,10,Promoter
3,10004,M,USA,No,7,Promoter
4,10005,F,GBR,No,9,Some desired label
5,10006,M,AUS,Yes,8,Promoter
6,10007,F,IND,No,2,Promoter
7,10008,M,DEU,Yes,9,Some desired label
8,10009,F,BRA,No,5,Promoter
9,10010,M,FRA,Yes,10,Promoter


In [37]:
(survey_responses
 .assign(Response_Type=lambda d: np.select(
     [(d['Likelihood'].le(6)) & (d['Country'].eq('USA')) & (d['Premium'].eq('Yes')),
      (d['Likelihood'].ge(9)) & (d['Country'].isin(['AUS','DEU']) | (d['Premium'] == 'No'))],
     ['USA premium detractor', 'Some desired label'],
     default='Promoter'))
 )

,Response_ID,Gender,Country,Premium,Likelihood,Response_Type
0,10001,M,USA,Yes,5,USA premium detractor
1,10002,M,JPN,No,3,Promoter
2,10003,F,CAN,Yes,10,Promoter
3,10004,M,USA,No,7,Promoter
4,10005,F,GBR,No,9,Some desired label
5,10006,M,AUS,Yes,8,Promoter
6,10007,F,IND,No,2,Promoter
7,10008,M,DEU,Yes,9,Some desired label
8,10009,F,BRA,No,5,Promoter
9,10010,M,FRA,Yes,10,Promoter


In [38]:
(survey_responses
 .assign(Is_Female=lambda d: np.where(d['Gender'] == 'F', 1, 0),
         Is_Promoter=lambda d: np.where(d['Likelihood'].isin([9,10]), 1, 0))
 )

,Response_ID,Gender,Country,Premium,Likelihood,Is_Female,Is_Promoter
0,10001,M,USA,Yes,5,0,0
1,10002,M,JPN,No,3,0,0
2,10003,F,CAN,Yes,10,1,1
3,10004,M,USA,No,7,0,0
4,10005,F,GBR,No,9,1,1
5,10006,M,AUS,Yes,8,0,0
6,10007,F,IND,No,2,1,0
7,10008,M,DEU,Yes,9,0,1
8,10009,F,BRA,No,5,1,0
9,10010,M,FRA,Yes,10,0,1


In [39]:
(survey_responses
 .filter(['Gender'])
 .pipe(pd.get_dummies) # <1>
 .astype(int) # <2>
 )

,Gender_F,Gender_M
0,0,1
1,0,1
2,1,0
3,0,1
4,1,0
5,0,1
6,1,0
7,0,1
8,1,0
9,0,1


In [40]:
streaming_subs = pd.DataFrame({
    'Customer_ID': [1001, 1002, 1003, 1004, 1005,
                    1003, 1006, 1007, 1008, 1009],
    'Subscription': ['Netflix','Hulu','Hulu','AMC','Paramount','Netflix',
                     'Hulu','Netflix','Netflix','Netflix'],
    'Months_Subscribed': [8, 7, 4, 9, 2, 5, 1, 6, 10, 2]})

streaming_subs

,Customer_ID,Subscription,Months_Subscribed
0,1001,Netflix,8
1,1002,Hulu,7
2,1003,Hulu,4
3,1004,AMC,9
4,1005,Paramount,2
5,1003,Netflix,5
6,1006,Hulu,1
7,1007,Netflix,6
8,1008,Netflix,10
9,1009,Netflix,2


In [41]:
(streaming_subs
 .assign(Netflix_Subscriber=lambda d: np.where(d['Subscription'] == 'Netflix', 1, 0),
         Hulu_Subscriber=lambda d: np.where(d['Subscription'] == 'Hulu', 1, 0))
 .groupby('Customer_ID', as_index=False)[['Netflix_Subscriber','Hulu_Subscriber']]
 .max()
 )

,Customer_ID,Netflix_Subscriber,Hulu_Subscriber
0,1001,1,0
1,1002,0,1
2,1003,1,1
3,1004,0,0
4,1005,0,0
5,1006,0,1
6,1007,1,0
7,1008,1,0
8,1009,1,0


In [42]:
(streaming_subs
 .assign(Loyal_Netflix=lambda d: np.where((d['Subscription'] == 'Netflix') & (d['Months_Subscribed'] > 5), 1, 0),
         Loyal_Hulu=lambda d: np.where((d['Subscription'] == 'Hulu') & (d['Months_Subscribed'] > 5), 1, 0))
 .groupby('Customer_ID', as_index=False)[['Loyal_Netflix','Loyal_Hulu']]
 .max()
 )

,Customer_ID,Loyal_Netflix,Loyal_Hulu
0,1001,1,0
1,1002,0,1
2,1003,0,0
3,1004,0,0
4,1005,0,0
5,1006,0,0
6,1007,1,0
7,1008,1,0
8,1009,0,0


### Type Conversions and Casting

In [43]:
customer_nums = pd.DataFrame({'Customer_ID': ['2367', '5498', '9975', '4326', '3302']})
customer_nums

,Customer_ID
0,2367
1,5498
2,9975
3,4326
4,3302


In [44]:
customer_nums.dtypes

Customer_ID    str
dtype: object

In [45]:
(customer_nums
 .astype({'Customer_ID':'int32'})
 .dtypes
 )

Customer_ID    int32
dtype: object

In [46]:
(customer_nums
 .assign(Customer_ID=lambda d: pd.to_numeric(d['Customer_ID']))
 .dtypes
 )

Customer_ID    int64
dtype: object

In [47]:
breakfast_foods = pd.DataFrame({
    'Item': ['Free Range Eggs (12 pack)','Wholemeal Bread','Orange Juice (1L)',
             'Greek Yogurt (500g)','Porridge Oats (1kg)'],
    'Price': ['£2.85','£1.20','£2.00','£1.75','£1.50']})
breakfast_foods

,Item,Price
0,Free Range Eggs (12 pack),£2.85
1,Wholemeal Bread,£1.20
2,Orange Juice (1L),£2.00
3,Greek Yogurt (500g),£1.75
4,Porridge Oats (1kg),£1.50


In [48]:
(breakfast_foods
 .assign(Price=pd.col('Price').str.replace('£','').astype('Float32'))
 )

,Item,Price
0,Free Range Eggs (12 pack),2.85
1,Wholemeal Bread,1.2
2,Orange Juice (1L),2.0
3,Greek Yogurt (500g),1.75
4,Porridge Oats (1kg),1.5


In [49]:
(clothing_store_sales
 .groupby(clothing_store_sales['Date'].dt.date)
 .size()
 )

Date
2013-09-30    1
2013-10-01    3
2013-10-03    2
2013-10-04    2
2013-10-05    2
             ..
2014-09-26    3
2014-09-27    5
2014-09-28    2
2014-09-29    3
2014-09-30    3
Length: 352, dtype: int64

In [50]:
(clothing_store_sales
 .pipe(lambda d: d.groupby(d['Date'].dt.date))
 .size()
 )

Date
2013-09-30    1
2013-10-01    3
2013-10-03    2
2013-10-04    2
2013-10-05    2
             ..
2014-09-26    3
2014-09-27    5
2014-09-28    2
2014-09-29    3
2014-09-30    3
Length: 352, dtype: int64

In [51]:
(clothing_store_sales
 .filter(['Category'])
 .drop_duplicates()
 .assign(Date_Bought=pd.Timestamp(2009,7,29))
 )

,Category,Date_Bought
0,Hat,2009-07-29
1,Sweater,2009-07-29
6,Socks,2009-07-29


In [52]:
(clothing_store_sales
 .filter(['Category'])
 .drop_duplicates()
 .assign(Date_Bought=pd.to_datetime('2012-10-18'))
 )

,Category,Date_Bought
0,Hat,2012-10-18
1,Sweater,2012-10-18
6,Socks,2012-10-18


### Reducing Memory Usage by Using Appropriate Data Types

In [53]:
online_retail_sales = pd.read_excel(f'{data}/uk_online_retail_sales.xlsx')
online_retail_sales

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365.0,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365.0,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365.0,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365.0,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365.0,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
...,...,...,...,...,...,...,...,...
541904,581587.0,22613,PACK OF 20 SPACEBOY NAPKINS,12,2011-12-09 12:50:00,0.85,12680.0,France
541905,581587.0,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France
541906,581587.0,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France
541907,581587.0,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France


In [54]:
initial_memory_usage = online_retail_sales.memory_usage(deep=True).sum() / 1_000_000 # <1>
print(initial_memory_usage)

59.038584


In [55]:
online_retail_sales.info()

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    532618 non-null  float64       
 1   StockCode    541909 non-null  str           
 2   Description  540455 non-null  str           
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[us]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  str           
dtypes: datetime64[us](1), float64(3), int64(1), str(3)
memory usage: 56.3 MB


In [56]:
final_memory_usage = (online_retail_sales
 .astype({'CustomerID':'Int16',
          'Quantity':'Int32',
          'InvoiceNo':'Int32',
          'Country':'category',
          'UnitPrice':'Float32',
          })
 .memory_usage(deep=True).sum() / 1_000_000
 )
final_memory_usage

np.float64(40.410429)

In [57]:
print(((final_memory_usage / initial_memory_usage) - 1) * 100)

-31.552509795966653


In [58]:
online_retail_sales.describe()

,InvoiceNo,Quantity,InvoiceDate,UnitPrice,CustomerID
count,532618.000000,541909.000000,541909,541909.000000,406829.000000
mean,559965.752027,9.552250,2011-07-04 13:34:57.156386,4.611114,15287.690570
min,536365.000000,-80995.000000,2010-12-01 08:26:00,-11062.060000,12346.000000
25%,547906.000000,1.000000,2011-03-28 11:34:00,1.250000,13953.000000
50%,560688.000000,3.000000,2011-07-19 17:17:00,2.080000,15152.000000
75%,571841.000000,10.000000,2011-10-19 11:27:00,4.130000,16791.000000
max,581587.000000,80995.000000,2011-12-09 12:50:00,38970.000000,18287.000000
std,13428.417281,218.081158,NaN,96.759853,1713.600303


In [59]:
import numpy as np

np.iinfo('int16')

iinfo(min=-32768, max=32767, dtype=int16)

In [60]:
np.finfo('float32')

finfo(resolution=1e-06, min=-3.4028235e+38, max=3.4028235e+38, dtype=float32)

### Dealing with Nulls and Empty Strings

In [61]:
df = pd.DataFrame({'Customer': ['David','Ashwin','Aja'],
 'Price_1': [3, 12, 14],
 'Price_2': [9, None, 7],
 'Item_1': ['Mango','Guava','Cassava'],
 'Item_2': [None, '', 'Peanuts']})
df

,Customer,Price_1,Price_2,Item_1,Item_2
0,David,3,9.0,Mango,NaN
1,Ashwin,12,NaN,Guava,
2,Aja,14,7.0,Cassava,Peanuts


In [62]:
(df
 .filter(['Customer','Price_1','Price_2'])
 .assign(Horizontal_Add=lambda d: d[['Price_1','Price_2']].sum(axis=1),
         Add=pd.col('Price_1') + pd.col('Price_2'),
         Subtract=pd.col('Price_1') - pd.col('Price_2'),
         Multiply=pd.col('Price_1') * pd.col('Price_2'),
         Divide=pd.col('Price_1') / pd.col('Price_2'))
 )

,Customer,Price_1,Price_2,Horizontal_Add,Add,Subtract,Multiply,Divide
0,David,3,9.0,12.0,12.0,-6.0,27.0,0.333333
1,Ashwin,12,NaN,12.0,NaN,NaN,NaN,NaN
2,Aja,14,7.0,21.0,21.0,7.0,98.0,2.000000


In [63]:
(df
 .filter(['Customer','Item_1','Item_2'])
 .assign(Horizontal_Add=lambda d: d.filter(['Item_1','Item_2']).sum(axis=1),
         Add=pd.col('Item_1') + pd.col('Item_2'))
 )

,Customer,Item_1,Item_2,Horizontal_Add,Add
0,David,Mango,NaN,Mango,NaN
1,Ashwin,Guava,,Guava,Guava
2,Aja,Cassava,Peanuts,CassavaPeanuts,CassavaPeanuts


In [64]:
(df
 .loc[pd.col('Item_2').notna()]
 )

,Customer,Price_1,Price_2,Item_1,Item_2
1,Ashwin,12,NaN,Guava,
2,Aja,14,7.0,Cassava,Peanuts


In [65]:
(df
 .loc[pd.col('Item_2').isna()]
 )

,Customer,Price_1,Price_2,Item_1,Item_2
0,David,3,9.0,Mango,NaN


In [66]:
(df
 .filter(['Customer','Price_2'])
 .assign(Fill_Constant=pd.col('Price_2').fillna(10),
         Fill_Avg=pd.col('Price_2').fillna(pd.col('Price_2').mean()))
 )

,Customer,Price_2,Fill_Constant,Fill_Avg
0,David,9.0,9.0,9.0
1,Ashwin,NaN,10.0,8.0
2,Aja,7.0,7.0,7.0


In [67]:
(df
 .filter(['Customer','Item_2'])
 .assign(Fill_Numeric=pd.col('Item_2').fillna(10),
         Fill_Shortest=lambda d: d['Item_2'].fillna(min(d['Item_2'].dropna(), key=len)),
         Fill_Longest=lambda d: d['Item_2'].fillna(max(d['Item_2'].dropna(), key=len)),
         Fill_String=pd.col('Item_2').fillna('Watermelon'))
 )

,Customer,Item_2,Fill_Numeric,Fill_Shortest,Fill_Longest,Fill_String
0,David,NaN,10,,Peanuts,Watermelon
1,Ashwin,,,,,
2,Aja,Peanuts,Peanuts,Peanuts,Peanuts,Peanuts


In [68]:
(df
 .filter(['Customer','Item_2'])
 .assign(Item_2=pd.col('Item_2').str.replace('','Papaya'))
 )

,Customer,Item_2
0,David,NaN
1,Ashwin,Papaya
2,Aja,PapayaPPapayaePapayaaPapayanPapayauPapayatPapa...


In [69]:
(df
 .filter(['Customer','Item_2'])
 .assign(Item_2=pd.col('Item_2').replace('', np.nan))
 )

,Customer,Item_2
0,David,NaN
1,Ashwin,NaN
2,Aja,Peanuts


## Preparing: Shaping Data
### Pivoting from Long Format to Wide Format

In [70]:
(clothing_store_sales
 .assign(Date=pd.col('Date').dt.year) # <1>
 .pivot_table(index='Date', columns='Category', values='Ext_Price', aggfunc='sum') # <2>
 .reset_index()
 .rename(lambda c: f'{c}_Amount' if c != 'Date' else 'Year', axis=1) # <3>
 )

Category,Year,Hat_Amount,Socks_Amount,Sweater_Amount
0,2013,30468.80,41042.41,73320.73
1,2014,68825.21,128127.52,228395.27


### Melting from Wide Format to Long Format

In [71]:
countries_gdp = pd.read_csv(f'{data}/countries_gdp.csv', skiprows=4, on_bad_lines='skip')

zambia_namibia_angola_gdp = (countries_gdp
 .rename(columns=lambda c: c.replace(' ','_'))
 .loc[pd.col('Country_Name').isin(['Zambia','Angola','Namibia'])]
 .filter(['Country_Name'] + list(map(str, range(2020, 2025))))
 .assign(**{c: (pd.col(c).astype('float64') / 1_000_000)
         .round()
         .astype('int32')
         for c in map(str, range(2020, 2025))})
 .rename(columns=lambda c: f'Year_{c}' if c[0].isdigit() else c)
 )
zambia_namibia_angola_gdp

,Country_Name,Year_2020,Year_2021,Year_2022,Year_2023,Year_2024
4,Angola,48502,66505,104400,84875,80397
171,Namibia,10584,12402,12569,12408,13372
264,Zambia,18138,22096,29164,27578,26326


In [72]:
(zambia_namibia_angola_gdp
 .melt(id_vars='Country_Name', value_name='GDP', var_name='Year')
 .assign(Year=pd.col('Year').str.replace('Year_','').astype('int32'))
 )

,Country_Name,Year,GDP
0,Angola,2020,48502
1,Namibia,2020,10584
2,Zambia,2020,18138
3,Angola,2021,66505
4,Namibia,2021,12402
5,Zambia,2021,22096
6,Angola,2022,104400
7,Namibia,2022,12569
8,Zambia,2022,29164
9,Angola,2023,84875


# Time Series Analysis
## Date, Datetime, and Time Manipulations
### Time Zone Conversions

In [73]:
import pandas as pd

utc_df = pd.DataFrame({
    'Date': pd.date_range(
        start='2025-09-01',
        end='2025-09-01',
        freq='MS',
        tz='UTC',
    )
})
utc_df

,Date
0,2025-09-01 00:00:00+00:00


In [74]:
(utc_df
 .assign(Date=pd.col('Date').dt.tz_convert('America/Chicago'))
)

,Date
0,2025-08-31 19:00:00-05:00


### Date and Datetime Format Conversions

In [75]:
(utc_df
 .assign(Current_Date=pd.Timestamp.now().date(),
         Current_Time=pd.Timestamp.now().time())
)

,Date,Current_Date,Current_Time
0,2025-09-01 00:00:00+00:00,2026-10-01,19:33:07.286109


In [76]:
(utc_df
 .drop(columns='Date')
 .assign(New_Date=pd.Timestamp(2025, 8, 5, 11, 32, 35))
 .assign(Truncate_Hr=pd.col('New_Date').dt.floor('h'))
)

,New_Date,Truncate_Hr
0,2025-08-05 11:32:35,2025-08-05 11:00:00


In [77]:
clothing_store_sales = (pd.read_csv(f'{data}/sample_sales.csv', parse_dates=['date'])
 .drop(columns='Account Name')
 .rename(columns=lambda c: c.title().replace(' ','_'))
 .rename(columns=dict(Account_Number='Customer_ID'))
 )
clothing_store_sales

,Customer_ID,Sku,Category,Quantity,Unit_Price,Ext_Price,Date
0,803666,HX-24728,Hat,1,98.98,98.98,2014-09-28 11:56:02
1,64898,LK-02338,Sweater,9,34.80,313.20,2014-04-24 16:51:22
2,423621,ZC-07383,Sweater,12,60.24,722.88,2014-09-17 17:26:22
3,137865,QS-76400,Sweater,5,15.25,76.25,2014-01-30 07:34:02
4,435433,RU-25060,Sweater,19,51.83,984.77,2014-08-24 06:18:12
...,...,...,...,...,...,...,...
995,29068,FT-50146,Sweater,2,46.48,92.96,2014-08-10 16:20:32
996,77116,IC-59308,Socks,19,29.25,555.75,2013-11-20 13:32:45
997,23749,IC-59308,Socks,18,54.79,986.22,2014-03-10 08:11:59
998,172519,RU-25060,Sweater,15,62.53,937.95,2014-04-11 02:50:03


In [78]:
(clothing_store_sales
 .filter(['Date'])
 .assign(Year=pd.col('Date').dt.year)
 .head(1)
 )

,Date,Year
0,2014-09-28 11:56:02,2014


In [79]:
(clothing_store_sales
 .filter(['Date'])
 .assign(Year=pd.col('Date').dt.year,
         Month=pd.col('Date').dt.month,
         Day=pd.col('Date').dt.day,
         Time=pd.col('Date').dt.time,
         Hour=pd.col('Date').dt.hour)
 .head(1)
 )

,Date,Year,Month,Day,Time,Hour
0,2014-09-28 11:56:02,2014,9,28,11:56:02,11


In [80]:
(clothing_store_sales
 .filter(['Date'])
 .assign(Day_Short=pd.col('Date').dt.strftime('%a'),
         Day_Long=pd.col('Date').dt.strftime('%A'),
         Month_Short=pd.col('Date').dt.strftime('%b'),
         Month_Long=pd.col('Date').dt.strftime('%B'))
 .drop_duplicates(subset='Day_Short')
 )

,Date,Day_Short,Day_Long,Month_Short,Month_Long
0,2014-09-28 11:56:02,Sun,Sunday,Sep,September
1,2014-04-24 16:51:22,Thu,Thursday,Apr,April
2,2014-09-17 17:26:22,Wed,Wednesday,Sep,September
5,2014-09-05 07:24:23,Fri,Friday,Sep,September
6,2014-04-28 21:51:24,Mon,Monday,Apr,April
15,2014-05-27 12:36:40,Tue,Tuesday,May,May
29,2013-12-28 01:49:48,Sat,Saturday,Dec,December


In [81]:
timestamps_df = pd.DataFrame({'Timestamp': [1698943443.890866, 1677779355.900452, 1698939958.48452]})
timestamps_df

,Timestamp
0,1.698943e+09
1,1.677779e+09
2,1.698940e+09


In [82]:
(timestamps_df
 .assign(Datetime=lambda d: pd.to_datetime(d['Timestamp'], unit='s'))
 )

,Timestamp,Datetime
0,1.698943e+09,2023-11-02 16:44:03.890866041
1,1.677779e+09,2023-03-02 17:49:15.900451899
2,1.698940e+09,2023-11-02 15:45:58.484519958


In [83]:
df = (pd.read_csv(f'{data}/sample_sales.csv')
 .filter(['date','sku','quantity'])
 .head(3)
 )
df

,date,sku,quantity
0,2014-09-28T11:56:02.000000,HX-24728,1
1,2014-04-24T16:51:22.000000,LK-02338,9
2,2014-09-17T17:26:22.000000,ZC-07383,12


In [84]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   date      3 non-null      str  
 1   sku       3 non-null      str  
 2   quantity  3 non-null      int64
dtypes: int64(1), str(2)
memory usage: 306.0 bytes


In [85]:
df = (pd.read_csv(f'{data}/sample_sales.csv', parse_dates=['date'])
 .filter(['date','sku','quantity'])
 .head(3)
 )
df

,date,sku,quantity
0,2014-09-28 11:56:02,HX-24728,1
1,2014-04-24 16:51:22,LK-02338,9
2,2014-09-17 17:26:22,ZC-07383,12


In [86]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   date      3 non-null      datetime64[us]
 1   sku       3 non-null      str           
 2   quantity  3 non-null      int64         
dtypes: datetime64[us](1), int64(1), str(1)
memory usage: 228.0 bytes


In [87]:
df = (pd.read_csv(f'{data}/tsa_passenger_volumes.csv')
 .head(3)
 )
df

,Date,Passengers
0,1/1/2019,2201765
1,1/2/2019,2424225
2,1/3/2019,2279384


In [88]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   Date        3 non-null      str  
 1   Passengers  3 non-null      int64
dtypes: int64(1), str(1)
memory usage: 204.0 bytes


In [89]:
(df
 .assign(Date=lambda d: pd.to_datetime(d['Date'], format='%m/%d/%Y'))
 .head(3)
 .info()
 )

<class 'pandas.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   Date        3 non-null      datetime64[us]
 1   Passengers  3 non-null      int64         
dtypes: datetime64[us](1), int64(1)
memory usage: 180.0 bytes


### Date Math

In [90]:
two_dates = pd.concat([
    clothing_store_sales.filter(['Date']).rename(columns={'Date': 'Date_1'}).head(3).reset_index(drop=True),
    clothing_store_sales.filter(['Date']).rename(columns={'Date': 'Date_2'}).tail(3).reset_index(drop=True)
], axis=1)
two_dates

,Date_1,Date_2
0,2014-09-28 11:56:02,2014-03-10 08:11:59
1,2014-04-24 16:51:22,2014-04-11 02:50:03
2,2014-09-17 17:26:22,2014-02-14 20:10:42


In [91]:
(two_dates
 .assign(Duration=pd.col('Date_1') - pd.col('Date_2'))
 .assign(Days=pd.col('Duration').dt.days,         # <1>
         Hours=pd.col('Duration') / pd.Timedelta(hours=1),
         Minutes=pd.col('Duration') / pd.Timedelta(minutes=1))
 )

,Date_1,Date_2,Duration,Days,Hours,Minutes
0,2014-09-28 11:56:02,2014-03-10 08:11:59,202 days 03:44:03,202,4851.734167,291104.050000
1,2014-04-24 16:51:22,2014-04-11 02:50:03,13 days 14:01:19,13,326.021944,19561.316667
2,2014-09-17 17:26:22,2014-02-14 20:10:42,214 days 21:15:40,214,5157.261111,309435.666667


In [92]:
(two_dates
 .assign(Duration=pd.col('Date_2') - pd.col('Date_1'))
 )

,Date_1,Date_2,Duration
0,2014-09-28 11:56:02,2014-03-10 08:11:59,-203 days +20:15:57
1,2014-04-24 16:51:22,2014-04-11 02:50:03,-14 days +09:58:41
2,2014-09-17 17:26:22,2014-02-14 20:10:42,-215 days +02:44:20


In [93]:
(two_dates
 .assign(Months=lambda d: (d['Date_1'].dt.year - d['Date_2'].dt.year) * 12
                          + (d['Date_1'].dt.month - d['Date_2'].dt.month))
 )

,Date_1,Date_2,Months
0,2014-09-28 11:56:02,2014-03-10 08:11:59,6
1,2014-04-24 16:51:22,2014-04-11 02:50:03,0
2,2014-09-17 17:26:22,2014-02-14 20:10:42,7


In [94]:
(clothing_store_sales
 .filter(['Date'])
 .assign(Day_English=pd.col('Date').dt.day_name(),
         Day_French=pd.col('Date').dt.day_name(locale='fr_FR'),
         Month_English=pd.col('Date').dt.month_name(),
         Month_French=pd.col('Date').dt.month_name(locale='fr_FR'))
 .drop_duplicates(subset='Day_English')
 )

,Date,Day_English,Day_French,Month_English,Month_French
0,2014-09-28 11:56:02,Sunday,Dimanche,September,Septembre
1,2014-04-24 16:51:22,Thursday,Jeudi,April,Avril
2,2014-09-17 17:26:22,Wednesday,Mercredi,September,Septembre
5,2014-09-05 07:24:23,Friday,Vendredi,September,Septembre
6,2014-04-28 21:51:24,Monday,Lundi,April,Avril
15,2014-05-27 12:36:40,Tuesday,Mardi,May,Mai
29,2013-12-28 01:49:48,Saturday,Samedi,December,Décembre


In [95]:
(clothing_store_sales
 .filter(['Date'])
 .assign(Add_7_Days=pd.col('Date') + pd.Timedelta(days=7),   # <1>
         Sub_7_Days=pd.col('Date') - pd.Timedelta(days=7))   # <2>
 .head(3)
 )

,Date,Add_7_Days,Sub_7_Days
0,2014-09-28 11:56:02,2014-10-05 11:56:02,2014-09-21 11:56:02
1,2014-04-24 16:51:22,2014-05-01 16:51:22,2014-04-17 16:51:22
2,2014-09-17 17:26:22,2014-09-24 17:26:22,2014-09-10 17:26:22


### Time Math

In [96]:
(two_dates
 .filter(['Date_1'])
 .assign(Add_2_Hours=lambda d: (d['Date_1'] + pd.Timedelta(hours=2)).dt.time,   # <1>
         Sub_5_Hrs_30_Min=lambda d: (d['Date_1'] - pd.Timedelta(hours=5, minutes=30)).dt.time)
 )

,Date_1,Add_2_Hours,Sub_5_Hrs_30_Min
0,2014-09-28 11:56:02,13:56:02,06:26:02
1,2014-04-24 16:51:22,18:51:22,11:21:22
2,2014-09-17 17:26:22,19:26:22,11:56:22


In [97]:
two_times = pd.DataFrame({
    'Time_1': two_dates['Date_1'].dt.time,
    'Time_2': two_dates['Date_2'].dt.time,
})
two_times

,Time_1,Time_2
0,11:56:02,08:11:59
1,16:51:22,02:50:03
2,17:26:22,20:10:42


In [98]:
(two_times
 .assign(Difference=lambda d: pd.to_timedelta(d['Time_1'].astype(str))
                             - pd.to_timedelta(d['Time_2'].astype(str)))
 )

,Time_1,Time_2,Difference
0,11:56:02,08:11:59,0 days 03:44:03
1,16:51:22,02:50:03,0 days 14:01:19
2,17:26:22,20:10:42,-1 days +21:15:40


In [99]:
(two_times
 .assign(Difference=lambda d: pd.to_timedelta(d['Time_1'].astype(str))
                             - pd.to_timedelta(d['Time_2'].astype(str)))
 .assign(Diff_Multiplied=pd.col('Difference') * 2,
         Diff_Divided=pd.col('Difference') / 2)
 )

,Time_1,Time_2,Difference,Diff_Multiplied,Diff_Divided
0,11:56:02,08:11:59,0 days 03:44:03,0 days 07:28:06,0 days 01:52:01.500000
1,16:51:22,02:50:03,0 days 14:01:19,1 days 04:02:38,0 days 07:00:39.500000
2,17:26:22,20:10:42,-1 days +21:15:40,-1 days +18:31:20,-1 days +22:37:50


In [100]:
int_to_time = pd.DataFrame({
    'Hour': [12,8],
    'Minute': [40,17],
})
int_to_time

,Hour,Minute
0,12,40
1,8,17


In [101]:
(int_to_time
 .assign(Time=lambda d:
     (pd.Timestamp('today').normalize() # <1>
      + pd.to_timedelta(d['Hour'], unit='h')
      + pd.to_timedelta(d['Minute'], unit='m'))
     .dt.time)
)

,Hour,Minute,Time
0,12,40,12:40:00
1,8,17,08:17:00


## Trending the Data
### Simple Trends

In [102]:
customer_inquiries = (pd.read_csv(f'{data}/customer_inquiries.csv', parse_dates=['DATE'])
 .drop(columns='WORK SITE')
 .rename(columns=lambda c: c.title())
 )

to_plot = (customer_inquiries
 .query("Channel == 'Voice In'")
 .groupby('Date', as_index=False)
 .agg(Volume=('Volume', 'sum'))
 .sort_values('Date')
 )
to_plot

,Date,Volume
0,2015-04-10,27
1,2015-04-13,21
2,2015-04-14,16
3,2015-04-15,9
4,2015-04-16,13
...,...,...
2623,2025-09-23,5
2624,2025-09-24,13
2625,2025-09-26,2
2626,2025-09-29,3


In [103]:
import hvplot.pandas

(to_plot
 .hvplot.line(x='Date', y='Volume',
              height=400,
              xlabel='', ylabel='Inquiries volume',
              title='Daily customer inquiry volume through voice in')
 )

:Curve   [Date]   (Volume)

In [104]:
to_plot = (customer_inquiries
 .query("Channel == 'Voice In'") # <1>
 .sort_values('Date')
 .set_index('Date')
 .resample('MS') # <2>
 .agg(Volume=('Volume', 'sum'))
 .reset_index()
 .rename(columns={'Date': 'Month'})
 )
to_plot

,Month,Volume
0,2015-04-01,226
1,2015-05-01,419
2,2015-06-01,401
3,2015-07-01,652
4,2015-08-01,636
...,...,...
121,2025-05-01,207
122,2025-06-01,80
123,2025-07-01,90
124,2025-08-01,103


In [105]:
(to_plot
 .hvplot.line(x='Month', y='Volume',
              height=400,
              xlabel='', ylabel='Inquiries volume',
              title='Monthly customer inquiry volume through voice in')
 )

:Curve   [Month]   (Volume)

### Comparing Components

In [106]:
(customer_inquiries
 .filter(['Channel'])
 .drop_duplicates()
 )

,Channel
0,Voice In
1,SMS Out
2,e-mail Out
88,Email - Business Hotline
90,Email - Other
207,Face2Face
211,Mail Out
5169,Mail In
5870,Voice Out
5911,SMS In


In [107]:
import numpy as np

customer_inquiries_clean = (customer_inquiries
 .assign(Channel=lambda d: np.select(
     [d['Channel'].str.contains('Email|e-mail'),       # <1>
      d['Channel'].str.contains('Voice'),
      d['Channel'].str.contains('SMS'),
      d['Channel'].str.contains('Mail In|Mail Out')],
     ['Email', 'Voice', 'SMS', 'Ordinary Mail'],
     default=d['Channel']))
 )

(customer_inquiries_clean
 .filter(['Channel'])
 .drop_duplicates()
 )

,Channel
0,Voice
1,SMS
2,Email
207,Face2Face
211,Ordinary Mail


In [108]:
to_plot = (customer_inquiries_clean
 .query("Channel in ['Face2Face','Voice','Email']")
 .assign(Year=lambda d: d['Date'].dt.year)
 .groupby(['Year', 'Channel'], as_index=False)
 .agg(Volume=('Volume', 'sum'))
 .sort_values('Year')
 )
to_plot

,Year,Channel,Volume
0,2015,Email,698
1,2015,Face2Face,28
2,2015,Voice,4687
3,2016,Email,1454
4,2016,Face2Face,8
5,2016,Voice,6534
6,2017,Email,877
7,2017,Face2Face,23
8,2017,Voice,6600
9,2018,Email,929


In [109]:
(to_plot
 .pivot(index='Year', columns='Channel', values='Volume')
 .reset_index()
 .hvplot.line(x='Year',
              y=['Face2Face','Voice','Email'],
              xlabel='', ylabel='Inquiries volume',
              legend_opts={'title':'Channel'},
              legend='top_right', color='#5e5c64',
              line_dash=['dashed','solid','dotted'])
 .opts(title="Yearly customer inquiry volume\n(voice, face2face, and email)")
 )

:NdOverlay   [Channel]
   :Curve   [Year]   (value)

In [110]:
to_plot = (customer_inquiries_clean
 .query("Channel in ['Voice','Email']")
 .sort_values('Date')
 .groupby('Channel')
 .resample('MS', on='Date')
 .agg(Volume=('Volume', 'sum'))
 .reset_index()
 .rename(columns={'Date':'Month'})
 )
to_plot

,Channel,Month,Volume
0,Email,2015-04-01,110
1,Email,2015-05-01,176
2,Email,2015-06-01,47
3,Email,2015-07-01,110
4,Email,2015-08-01,67
...,...,...,...
247,Voice,2025-05-01,207
248,Voice,2025-06-01,80
249,Voice,2025-07-01,90
250,Voice,2025-08-01,103


In [111]:
(to_plot
 .pivot(index='Month', columns='Channel', values='Volume')
 .reset_index()
 .hvplot.line(x='Month',
              y=['Voice','Email'],
              xlabel='', ylabel='Inquiries volume',
              legend_opts={'title':'Channel'},
              legend='top_right', color=['#5e5c64','#c0bfbc'])
 .opts(title="Monthly voice and email customer inquiry volume")
 )

:NdOverlay   [Channel]
   :Curve   [Month]   (value)

In [112]:
(customer_inquiries_clean
 .query("Channel in ['Voice','Email']")
 .sort_values('Date')
 .groupby('Channel')
 .resample('YS', on='Date')
 .agg(Volume=('Volume', 'sum'))
 .reset_index()
 .rename(columns={'Date':'Year'})
 .pivot(index='Year', columns='Channel', values='Volume')
 .reset_index()
 .hvplot.line(x='Year',
              y=['Voice','Email'],
              xlabel='', ylabel='Inquiries volume',
              legend_opts={'title':'Channel'},
              legend='top_right', color=['#1a1a1a','#c0bfbc'])
 .opts(title="Yearly voice and email customer inquiry volume")
 )

:NdOverlay   [Channel]
   :Curve   [Year]   (value)

In [113]:
(customer_inquiries_clean
 .query("Channel in ['Voice','Email']")
 .sort_values('Date')
 .groupby('Channel')
 .resample('YS', on='Date')
 .agg(Volume=('Volume', 'sum'))
 .reset_index()
 .pivot(index='Date', columns='Channel', values='Volume')
 .reset_index()
 .rename(columns={'Date':'Year','Voice':'Voice_Volume', 'Email':'Email_Volume'})
 )

Channel,Year,Email_Volume,Voice_Volume
0,2015-01-01,698,4687
1,2016-01-01,1454,6534
2,2017-01-01,877,6600
3,2018-01-01,929,6845
4,2019-01-01,1427,6022
5,2020-01-01,495,4861
6,2021-01-01,687,3018
7,2022-01-01,676,3201
8,2023-01-01,859,2369
9,2024-01-01,1968,1612


In [114]:
(customer_inquiries_clean
 .query("Channel in ['Voice','Email']")
 .sort_values('Date')
 .groupby('Channel')
 .resample('YS', on='Date')
 .agg(Volume=('Volume', 'sum'))
 .reset_index()
 .pivot(index='Date', columns='Channel', values='Volume')
 .reset_index()
 .rename(columns={'Date':'Year','Voice':'Voice_Volume', 'Email':'Email_Volume'})
 .assign(Voice_Sub_Email=pd.col('Voice_Volume') - pd.col('Email_Volume'),
         Email_Sub_Voice=pd.col('Email_Volume') - pd.col('Voice_Volume'))
 .drop(columns=['Voice_Volume', 'Email_Volume'])
 )

Channel,Year,Voice_Sub_Email,Email_Sub_Voice
0,2015-01-01,3989,-3989
1,2016-01-01,5080,-5080
2,2017-01-01,5723,-5723
3,2018-01-01,5916,-5916
4,2019-01-01,4595,-4595
5,2020-01-01,4366,-4366
6,2021-01-01,2331,-2331
7,2022-01-01,2525,-2525
8,2023-01-01,1510,-1510
9,2024-01-01,-356,356


In [115]:
(customer_inquiries_clean
 .query("Channel in ['Voice','Email']")
 .sort_values('Date')
 .groupby('Channel')
 .resample('YS', on='Date')
 .agg(Volume=('Volume', 'sum'))
 .reset_index()
 .pivot(index='Date', columns='Channel', values='Volume')
 .reset_index()
 .rename(columns={'Date':'Year','Voice':'Voice_Volume', 'Email':'Email_Volume'})
 .assign(Voice_Sub_Email=pd.col('Voice_Volume') - pd.col('Email_Volume'))
 .filter(['Year','Voice_Sub_Email'])
 .hvplot.line(x='Year', y='Voice_Sub_Email',
              xlabel='', ylabel='Inquiries volume')
 .opts(title="Yearly difference between voice and email customer inquiry volume")
 )

:Curve   [Year]   (Voice_Sub_Email)

In [116]:
to_plot = (customer_inquiries_clean
 .query("Channel in ['Voice','Email']")
 .sort_values('Date')
 .groupby(['Channel', pd.Grouper(key='Date', freq='YS')]) # <1>
 .agg(Volume=('Volume', 'sum'))
 .reset_index()
 .pivot(index='Date', columns='Channel', values='Volume')
 .reset_index()
 .rename(columns={'Date':'Year','Voice':'Voice_Volume', 'Email':'Email_Volume'})
 .assign(Voice_Times_of_Email=pd.col('Voice_Volume') / pd.col('Email_Volume'))
 .filter(['Year','Voice_Times_of_Email'])
 )
to_plot

,Year,Voice_Times_of_Email
0,2015-01-01,6.714900
1,2016-01-01,4.493810
2,2017-01-01,7.525656
3,2018-01-01,7.368138
4,2019-01-01,4.220042
5,2020-01-01,9.820202
6,2021-01-01,4.393013
7,2022-01-01,4.735207
8,2023-01-01,2.757858
9,2024-01-01,0.819106


In [117]:
(to_plot
 .hvplot.line(x='Year', y='Voice_Times_of_Email',
              xlabel='', ylabel='')
 .opts(title="Yearly ratio of voice to email customer inquiries volume")
 )

:Curve   [Year]   (Voice_Times_of_Email)

In [118]:
(customer_inquiries_clean
 .query("Channel in ['Voice','Email']")
 .sort_values('Date')
 .groupby(['Channel', pd.Grouper(key='Date', freq='YS')])
 .agg(Volume=('Volume', 'sum'))
 .reset_index()
 .pivot(index='Date', columns='Channel', values='Volume')
 .reset_index()
 .rename(columns={'Date':'Year','Voice':'Voice_Volume', 'Email':'Email_Volume'})
 .assign(Voice_Pct_of_Email=((pd.col('Voice_Volume') / pd.col('Email_Volume')) - 1) * 100)
 .filter(['Year','Voice_Pct_of_Email'])
 )

,Year,Voice_Pct_of_Email
0,2015-01-01,571.489971
1,2016-01-01,349.381018
2,2017-01-01,652.565564
3,2018-01-01,636.813778
4,2019-01-01,322.004205
5,2020-01-01,882.020202
6,2021-01-01,339.301310
7,2022-01-01,373.520710
8,2023-01-01,175.785797
9,2024-01-01,-18.089431


### Percent of Total Calculations

In [119]:
to_plot = (customer_inquiries_clean
 .loc[pd.col('Channel').isin(['Voice','Email'])]
 .groupby([pd.Grouper(key='Date', freq='MS'), 'Channel'], as_index=False)
 .agg(Volume=('Volume','sum'))
 .assign(Total_Volume=lambda d: d.groupby('Date')['Volume'].transform('sum')) # <1>
 .assign(Pct_Total_Volume=pd.col('Volume') / pd.col('Total_Volume') * 100)
 .sort_values('Date')
 .rename(columns=dict(Date='Month'))
 .filter(['Month','Channel','Pct_Total_Volume'])
 )
to_plot

,Month,Channel,Pct_Total_Volume
0,2015-04-01,Email,31.884058
1,2015-04-01,Voice,68.115942
2,2015-05-01,Email,28.758170
3,2015-05-01,Voice,71.241830
4,2015-06-01,Email,10.239651
...,...,...,...
247,2025-07-01,Voice,19.480519
248,2025-08-01,Email,75.417661
249,2025-08-01,Voice,24.582339
250,2025-09-01,Email,65.957447


In [120]:
(to_plot
 .pivot(index='Month', columns='Channel', values='Pct_Total_Volume')
 .reset_index()
 .hvplot.line(x='Month', y=['Voice','Email'],
              height=400, color=['#1a1a1a','#c0bfbc'],
              yformatter='%.0f%%',
              legend='top_right',
              legend_opts={'title':''}, # <1>
              legend_cols=2, # <2>
              xlabel='', ylabel='% of total')
 .opts(title="Voice and email customer inquiry volume (percent of monthly total)")
 )

:NdOverlay   [Channel]
   :Curve   [Month]   (value)

In [121]:
to_plot = (customer_inquiries_clean
 .query("Channel in ['Voice','Email']")
 .groupby([pd.Grouper(key='Date', freq='MS'), 'Channel'], as_index=False)
 .agg(Volume=('Volume','sum'))
 .rename(columns={'Date':'Month'})
 .assign(Year=pd.col('Month').dt.year)
 .assign(Total_Yearly_Volume=lambda d: d.groupby(['Year','Channel'])['Volume'].transform('sum'))
 .assign(Pct_Yearly=pd.col('Volume') / pd.col('Total_Yearly_Volume') * 100)
 .filter(['Month','Channel','Pct_Yearly'])
 .sort_values('Month')
 )
to_plot

,Month,Channel,Pct_Yearly
0,2015-04-01,Email,15.759312
1,2015-04-01,Voice,5.013868
2,2015-05-01,Email,25.214900
3,2015-05-01,Voice,9.302326
4,2015-06-01,Email,6.733524
...,...,...,...
247,2025-07-01,Voice,7.334963
248,2025-08-01,Email,14.555504
249,2025-08-01,Voice,8.394458
250,2025-09-01,Email,8.567480


In [122]:
(to_plot
 .loc[pd.col('Month').dt.year.eq(2024)]
 .pivot(index='Month', columns='Channel', values='Pct_Yearly')
 .reset_index()
 .hvplot.line(
     x='Month',
     y=['Voice','Email'],
     height=450, color=['#1a1a1a','#c0bfbc'],
     title="Percent of 2024 yearly customer inquiries volume by month\n(Voice and email)",
     legend='top_right',
     legend_opts={'title': ''},
     legend_cols=2,
     ylabel='% of yearly volume',
     yformatter='%.0f%%',
     backend_opts={
         'plot.yaxis.axis_label_text_font_style': 'bold'
     })
)

:NdOverlay   [Channel]
   :Curve   [Month]   (value)

### Indexing to Measure Percent Change Over Time

In [123]:
(customer_inquiries_clean
 .query("Channel == 'Voice'")
 .assign(Year=pd.col('Date').dt.year)
 .query("Year != 2015") # <1>
 .groupby('Year', as_index=False)
 .agg(Volume=('Volume','sum'))
 .sort_values('Year')
 .assign(Index_Volume=lambda d: d['Volume'].iloc[0])
 )

,Year,Volume,Index_Volume
0,2016,6534,6534
1,2017,6600,6534
2,2018,6845,6534
3,2019,6022,6534
4,2020,4861,6534
5,2021,3018,6534
6,2022,3201,6534
7,2023,2369,6534
8,2024,1612,6534
9,2025,1227,6534


In [124]:
(customer_inquiries_clean
 .query("Channel == 'Voice'")
 .assign(Year=pd.col('Date').dt.year)
 .query("Year != 2015")
 .groupby('Year', as_index=False)
 .agg(Volume=('Volume','sum'))
 .sort_values('Year')
 .assign(Index_Volume=lambda d: d['Volume'].iloc[0])
 .assign(Pct_from_Index=(pd.col('Volume') / pd.col('Index_Volume') - 1) * 100)
 .drop(columns='Index_Volume')
 )

,Year,Volume,Pct_from_Index
0,2016,6534,0.000000
1,2017,6600,1.010101
2,2018,6845,4.759718
3,2019,6022,-7.835935
4,2020,4861,-25.604530
5,2021,3018,-53.810836
6,2022,3201,-51.010101
7,2023,2369,-63.743496
8,2024,1612,-75.329048
9,2025,1227,-81.221304


In [125]:
to_plot = (customer_inquiries_clean
 .query("Channel in ['Voice','Email']")
 .assign(Year=pd.col('Date').dt.year)
 .query("Year != 2015")
 .groupby(['Year','Channel'], as_index=False)
 .agg(Volume=('Volume','sum'))
 .sort_values(['Channel','Year'])
 .assign(Index_Volume=lambda d: d.groupby('Channel')['Volume'].transform('first'))
 .assign(Pct_from_Index=(pd.col('Volume') / pd.col('Index_Volume') - 1) * 100)
 .drop(columns='Index_Volume')
 .sort_values('Year')
 )
to_plot

,Year,Channel,Volume,Pct_from_Index
0,2016,Email,1454,0.000000
1,2016,Voice,6534,0.000000
2,2017,Email,877,-39.683631
3,2017,Voice,6600,1.010101
4,2018,Email,929,-36.107290
5,2018,Voice,6845,4.759718
6,2019,Email,1427,-1.856946
7,2019,Voice,6022,-7.835935
8,2020,Email,495,-65.955983
9,2020,Voice,4861,-25.604530


In [126]:
(to_plot
 .drop(columns='Volume')
 .pivot(index='Year', columns='Channel', values='Pct_from_Index')
 .reset_index()
 .sort_values('Year')
 .hvplot.line(
     x='Year', y=['Voice','Email'],
     title="Voice and email customer inquiries volume indexed to 2016 volume",
     yformatter='%.0f%%',
     xlabel='', ylabel='% Change',
     height=400, color=['#1a1a1a','#c0bfbc'],
     legend='top_left',
     legend_opts={'title':'Channel'},
     backend_opts={'plot.yaxis.axis_label_text_font_style':'bold'})
 )

:NdOverlay   [Channel]
   :Curve   [Year]   (value)

## Rolling Time Windows
### Calculating Rolling Time Windows

In [127]:
to_plot = (customer_inquiries_clean
 .loc[pd.col('Channel').eq('Voice') & (pd.col('Date').dt.year.between(2016, 2024))]
 .sort_values('Date')
 .groupby('Channel')
 .resample('MS', on='Date')
 .agg(Volume=('Volume','sum'))
 .reset_index()
 .assign(Moving_Avg=pd.col('Volume').rolling(window=12, min_periods=1).mean())
 .assign(Records_Count=pd.col('Volume').notna().rolling(window=12, min_periods=1).sum())
 .loc[pd.col('Date') >= pd.Timestamp(2017, 1, 1)] # <1>
 )
to_plot

,Channel,Date,Volume,Moving_Avg,Records_Count
12,Voice,2017-01-01,404,538.666667,12.0
13,Voice,2017-02-01,519,524.250000,12.0
14,Voice,2017-03-01,862,546.500000,12.0
15,Voice,2017-04-01,392,530.333333,12.0
16,Voice,2017-05-01,570,528.583333,12.0
...,...,...,...,...,...
103,Voice,2024-08-01,141,156.500000,12.0
104,Voice,2024-09-01,118,152.416667,12.0
105,Voice,2024-10-01,100,143.416667,12.0
106,Voice,2024-11-01,68,133.833333,12.0


In [128]:
(to_plot
 .hvplot.line(
     x='Date',
     y=['Volume','Moving_Avg'],
     title="Monthly customer inquiries volume and 12-month moving average volume\n(Voice)",
     xlabel='',
     ylabel='Inquiries volume',
     legend='bottom_left',
     legend_opts={'title': None},
     height=400, color=['#1a1a1a','#c0bfbc'],
     backend_opts={'plot.yaxis.axis_label_text_font_style': 'bold'})
 )

:NdOverlay   [Variable]
   :Curve   [Date]   (value)

### Rolling Time Windows with Sparse Data

In [129]:
date_dimension = (pd.read_parquet(f'{data}/date_dimension.parquet')
 .assign(Date=lambda d: pd.to_datetime(d['Date']))
 )
date_dimension

,Date,Date_Key,Day_of_Month,Day_of_Year,Day_of_Week,Day_Name,Day_Short_Name,Week_Number,Week_of_Month,Week,...,Month_Short_Name,First_Day_of_Month,Last_Day_of_Month,Quarter_Number,Quarter_Name,First_Day_of_Quarter,Last_Day_of_Quarter,Year,Decade,Centurys
0,2000-01-01,100,1,1,6,Saturday,Sat,52,1,1999-12-27,...,Jan,2000-01-01,2000-01-31,1,Q1,2000-01-01,2000-03-31,2000,2000,20
1,2000-01-02,101,2,2,0,Sunday,Sun,52,1,1999-12-27,...,Jan,2000-01-01,2000-01-31,1,Q1,2000-01-01,2000-03-31,2000,2000,20
2,2000-01-03,102,3,3,1,Monday,Mon,1,1,2000-01-03,...,Jan,2000-01-01,2000-01-31,1,Q1,2000-01-01,2000-03-31,2000,2000,20
3,2000-01-04,103,4,4,2,Tuesday,Tue,1,1,2000-01-03,...,Jan,2000-01-01,2000-01-31,1,Q1,2000-01-01,2000-03-31,2000,2000,20
4,2000-01-05,104,5,5,3,Wednesday,Wed,1,1,2000-01-03,...,Jan,2000-01-01,2000-01-31,1,Q1,2000-01-01,2000-03-31,2000,2000,20
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11318,2030-12-27,11418,27,361,5,Friday,Fri,52,4,2030-12-23,...,Dec,2030-12-01,2030-12-31,4,Q4,2030-10-01,2030-12-31,2030,2030,21
11319,2030-12-28,11419,28,362,6,Saturday,Sat,52,4,2030-12-23,...,Dec,2030-12-01,2030-12-31,4,Q4,2030-10-01,2030-12-31,2030,2030,21
11320,2030-12-29,11420,29,363,0,Sunday,Sun,52,5,2030-12-23,...,Dec,2030-12-01,2030-12-31,4,Q4,2030-10-01,2030-12-31,2030,2030,21
11321,2030-12-30,11421,30,364,1,Monday,Mon,1,5,2030-12-30,...,Dec,2030-12-01,2030-12-31,4,Q4,2030-10-01,2030-12-31,2030,2030,21


In [130]:
(date_dimension
 .loc[pd.col('Date') == pd.col('First_Day_of_Month')] # <1>
 .loc[pd.col('Date').dt.year.between(2017, 2024)]
 .merge(customer_inquiries_clean
        .loc[(pd.col('Channel') == 'Voice')
              & (pd.col('Date').dt.year.between(2016, 2024))
              & (pd.col('Date').dt.month.isin([3, 9]))]
        .assign(Month_Volume=pd.col('Date').dt.to_period('M').dt.to_timestamp())
        .groupby('Month_Volume', as_index=False)
        .agg(Volume=('Volume', 'sum')),
        how='cross')
 .loc[pd.col('Month_Volume') <= pd.col('Date')]
 .loc[pd.col('Month_Volume') >= (pd.col('Date') - pd.DateOffset(months=11))]
 .filter(['Date', 'Month_Volume', 'Volume'])
 )

,Date,Month_Volume,Volume
0,2017-01-01,2016-03-01,595
1,2017-01-01,2016-09-01,499
18,2017-02-01,2016-03-01,595
19,2017-02-01,2016-09-01,499
37,2017-03-01,2016-09-01,499
...,...,...,...
1691,2024-10-01,2024-09-01,118
1708,2024-11-01,2024-03-01,135
1709,2024-11-01,2024-09-01,118
1726,2024-12-01,2024-03-01,135


In [131]:
(date_dimension
 .loc[pd.col('Date') == pd.col('First_Day_of_Month')]
 .loc[pd.col('Date').dt.year.between(2017, 2024)]
 .merge(customer_inquiries_clean
        .loc[(pd.col('Channel') == 'Voice')
              & (pd.col('Date').dt.year.between(2016, 2024))
              & (pd.col('Date').dt.month.isin([3, 9]))]
        .assign(Month_Volume=pd.col('Date').dt.to_period('M').dt.to_timestamp())
        .groupby('Month_Volume', as_index=False)
        .agg(Volume=('Volume', 'sum')),
        how='cross')
 .loc[pd.col('Month_Volume') <= pd.col('Date')]
 .loc[pd.col('Month_Volume') >= (pd.col('Date') - pd.DateOffset(months=11))]
 .filter(['Date', 'Month_Volume', 'Volume'])
 .groupby('Date', as_index=False)
 .agg(Moving_Avg=('Volume', 'mean'),
      Records=('Volume', 'count'))
 .sort_values('Date')
 )

,Date,Moving_Avg,Records
0,2017-01-01,547.0,2
1,2017-02-01,547.0,2
2,2017-03-01,680.5,2
3,2017-04-01,680.5,2
4,2017-05-01,680.5,2
...,...,...,...
91,2024-08-01,151.0,2
92,2024-09-01,126.5,2
93,2024-10-01,126.5,2
94,2024-11-01,126.5,2


### Calculating Cumulative Values

In [132]:
to_plot = (customer_inquiries_clean
 .query("Channel == 'Voice'")
 .sort_values('Date')
 .groupby('Channel')
 .resample('MS', on='Date')
 .agg(Volume=('Volume', 'sum'))
 .reset_index()
 .assign(Volume_YTD=lambda d: d.groupby(d['Date'].dt.year)['Volume'].cumsum())
 .filter(['Date', 'Volume', 'Volume_YTD'])
 )
to_plot

,Date,Volume,Volume_YTD
0,2015-04-01,235,235
1,2015-05-01,436,671
2,2015-06-01,412,1083
3,2015-07-01,684,1767
4,2015-08-01,647,2414
...,...,...,...
121,2025-05-01,207,858
122,2025-06-01,80,938
123,2025-07-01,90,1028
124,2025-08-01,103,1131


In [133]:
plot_data = to_plot.loc[pd.col('Date').dt.year.between(2019, 2024)]

# Line plot: left axis
line_plot = (
    plot_data.hvplot.line(
        x='Date',
        y='Volume',
        color='#1a1a1a',
        label='Monthly Volume',
        ylabel='Monthly volume'
    )
    .opts(yaxis='left')
 )

# Bar plot: right axis
bar_plot = (
    plot_data.hvplot.bar(
        x='Date',
        y='Volume_YTD',
        color='#c0bfbc',
        alpha=0.6,
        label='Volume YTD',
        ylabel='Volume YTD'
    )
    .opts(yaxis='right')
 )

# Combine charts with independent y-axes
(bar_plot * line_plot).opts(
    multi_y=True,
    legend_position='top_right',
    height=400,
    title="Monthly customer inquiries volume and cumulative annual volume\n(Voice)"
 )

:Overlay
   .Bars.Volume_YTD      :Bars   [Date]   (Volume_YTD)
   .Curve.Monthly_Volume :Curve   [Date]   (Volume)

## Analyzing with Seasonality

In [134]:
categories_chosen = ['Call Centre Services',
                     'Business Assistance Services',
                     'Food and Health Activity Control']

to_plot = (customer_inquiries_clean
 .loc[pd.col('Category').isin(categories_chosen)]
 .sort_values('Date')
 .groupby('Category')
 .resample('MS', on='Date')
 .agg(Volume=('Volume', 'sum'))
 .reset_index()
 .pivot(index='Date', columns='Category', values='Volume')
 .fillna(0)
 .reset_index()
 )
to_plot

Category,Date,Business Assistance Services,Call Centre Services,Food and Health Activity Control
0,2015-04-01,319,2,13
1,2015-05-01,522,4,18
2,2015-06-01,417,4,17
3,2015-07-01,644,7,23
4,2015-08-01,598,10,18
...,...,...,...,...
121,2025-05-01,90,11,8
122,2025-06-01,64,18,7
123,2025-07-01,83,2,2
124,2025-08-01,86,2,11


In [135]:
call_center_plot = (to_plot
 .filter(['Date', categories_chosen[0]])
 .hvplot.line(
     x='Date',
     y=categories_chosen[0],
     xlabel='',
     ylabel='Call center\nservices',
     height=210)
 .opts(xaxis=None)
 )

business_assistance_plot = (to_plot
 .filter(['Date', categories_chosen[1]])
 .hvplot.line(
     x='Date',
     y=categories_chosen[1],
     xlabel='',
     ylabel='Bussiness assistance\nservices',
     height=210)
 .opts(xaxis=None)
 )

food_health_plot = (to_plot
 .filter(['Date', categories_chosen[2]])
 .hvplot.line(
     x='Date',
     y=categories_chosen[2],
     xlabel='Month',
     ylabel='Food and health\nactivity control',
     height=210)
 )

(call_center_plot + business_assistance_plot + food_health_plot).cols(1) # <1>

:Layout
   .Curve.I   :Curve   [Date]   (Call Centre Services)
   .Curve.II  :Curve   [Date]   (Business Assistance Services)
   .Curve.III :Curve   [Date]   (Food and Health Activity Control)

### The TSA Passenger Volume Dataset

In [136]:
tsa_passengers = (pd.read_csv(f'{data}/tsa_passenger_volumes.csv')
 .assign(Date=lambda d: pd.to_datetime(d['Date'], format='%m/%d/%Y'))
 )

(tsa_passengers
 .sort_values('Date')
 .groupby(pd.Grouper(key='Date', freq='MS'), as_index=False)
 .agg(Passengers=('Passengers', 'sum'))
 .assign(Passengers=pd.col('Passengers') / 1_000_000) # <1>
 .hvplot.line(x='Date', y='Passengers',
              height=350,
              xlabel='', ylabel='Passengers (millions)',
              title='TSA checkpoint monthly passenger volume')
 )

:Curve   [Date]   (Passengers)

### Period-over-Period Comparisons: YoY and MoM

In [137]:
(tsa_passengers
 .sort_values('Date')
 .assign(Prev_Date=pd.col('Date').shift(),
         Prev_Passengers=pd.col('Passengers').shift())
 )

,Date,Passengers,Prev_Date,Prev_Passengers
0,2019-01-01,2201765,NaT,NaN
1,2019-01-02,2424225,2019-01-01,2201765.0
2,2019-01-03,2279384,2019-01-02,2424225.0
3,2019-01-04,2230078,2019-01-03,2279384.0
4,2019-01-05,2049460,2019-01-04,2230078.0
...,...,...,...,...
2552,2025-12-27,2804814,2025-12-26,2716278.0
2553,2025-12-28,2924593,2025-12-27,2804814.0
2554,2025-12-29,2811757,2025-12-28,2924593.0
2555,2025-12-30,2827572,2025-12-29,2811757.0


In [138]:
(tsa_passengers
 .sort_values('Date')
 .groupby(pd.Grouper(key='Date', freq='MS'), as_index=False)
 .agg(Passengers=('Passengers', 'sum'))
 .assign(Prev_Passengers=pd.col('Passengers').shift())
 .assign(Pct_Growth_from_Prev=(pd.col('Passengers') / pd.col('Prev_Passengers') - 1) * 100)
 )

,Date,Passengers,Prev_Passengers,Pct_Growth_from_Prev
0,2019-01-01,61694899,NaN,NaN
1,2019-02-01,58535547,61694899.0,-5.120929
2,2019-03-01,72714771,58535547.0,24.223271
3,2019-04-01,69754997,72714771.0,-4.070389
4,2019-05-01,74582398,69754997.0,6.920509
...,...,...,...,...
79,2025-08-01,80331071,85093035.0,-5.596185
80,2025-09-01,72091468,80331071.0,-10.257056
81,2025-10-01,79099091,72091468.0,9.720461
82,2025-11-01,71776970,79099091.0,-9.256897


In [139]:
to_plot = (tsa_passengers
 .sort_values('Date')
 .groupby(pd.Grouper(key='Date', freq='MS'), as_index=False)
 .agg(Passengers=('Passengers', 'sum'))
 .assign(Pct_Growth_from_Prev=pd.col('Passengers').pct_change() * 100)
 )
to_plot

,Date,Passengers,Pct_Growth_from_Prev
0,2019-01-01,61694899,NaN
1,2019-02-01,58535547,-5.120929
2,2019-03-01,72714771,24.223271
3,2019-04-01,69754997,-4.070389
4,2019-05-01,74582398,6.920509
...,...,...,...
79,2025-08-01,80331071,-5.596185
80,2025-09-01,72091468,-10.257056
81,2025-10-01,79099091,9.720461
82,2025-11-01,71776970,-9.256897


In [140]:
(tsa_passengers
 .sort_values('Date')
 .groupby(pd.Grouper(key='Date', freq='YS'), as_index=False)
 .agg(Passengers=('Passengers', 'sum'))
 .assign(Pct_Change=pd.col('Passengers').pct_change() * 100)
 .assign(Year=pd.col('Date').dt.year)
 .rename(columns={'Passengers': 'Yearly_Passengers'})
 .filter(['Year', 'Yearly_Passengers', 'Pct_Change'])
 )

,Year,Yearly_Passengers,Pct_Change
0,2019,848102043,NaN
1,2020,339774756,-59.937043
2,2021,585250987,72.246754
3,2022,760071362,29.871009
4,2023,858548196,12.956262
5,2024,904068577,5.302018
6,2025,906735976,0.295044


In [141]:
(to_plot
 .hvplot.line(x='Date', y='Pct_Growth_from_Prev',
              xlabel='Month', ylabel='% Growth',
              height=500, yformatter='%.0f%%',
              title="Percent growth from previous month for US air travel passengers")
 )

:Curve   [Date]   (Pct_Growth_from_Prev)

### Period-over-Period Comparisons: Same Month Versus Last Year

In [142]:
(tsa_passengers
 .sort_values('Date')
 .groupby(pd.Grouper(key='Date', freq='MS'), as_index=False)
 .agg(Passengers=('Passengers', 'sum'))
 .assign(Month=pd.col('Date').dt.month)
 .filter(['Date', 'Month'])
 )

,Date,Month
0,2019-01-01,1
1,2019-02-01,2
2,2019-03-01,3
3,2019-04-01,4
4,2019-05-01,5
...,...,...
79,2025-08-01,8
80,2025-09-01,9
81,2025-10-01,10
82,2025-11-01,11


In [143]:
(tsa_passengers
 .sort_values('Date')
 .groupby(pd.Grouper(key='Date', freq='MS'), as_index=False)
 .agg(Passengers=('Passengers', 'sum'))
 .assign(Month=pd.col('Date').dt.month)
 .assign(Prev_Year_Month=lambda d: d.groupby('Month')['Date'].shift(),
         Prev_Year_Passengers=lambda d: d.groupby('Month')['Passengers'].shift())
 .sort_values('Month') # <1>
 .filter(['Date', 'Passengers', 'Prev_Year_Month', 'Prev_Year_Passengers'])
 )

,Date,Passengers,Prev_Year_Month,Prev_Year_Passengers
0,2019-01-01,61694899,NaT,NaN
48,2023-01-01,60906882,2022-01-01,46196098.0
24,2021-01-01,24923269,2020-01-01,64932623.0
60,2024-01-01,64647251,2023-01-01,60906882.0
36,2022-01-01,46196098,2021-01-01,24923269.0
...,...,...,...,...
59,2023-12-01,72562140,2022-12-01,66131410.0
23,2020-12-01,27785059,2019-12-01,73087498.0
47,2022-12-01,66131410,2021-12-01,59339590.0
71,2024-12-01,77384934,2023-12-01,72562140.0


In [144]:
to_plot = (tsa_passengers
 .sort_values('Date')
 .groupby(pd.Grouper(key='Date', freq='MS'), as_index=False)
 .agg(Passengers=('Passengers', 'sum'))
 .assign(Month=pd.col('Date').dt.month)
 .assign(Prev_Passengers=lambda d: d.groupby('Month')['Passengers'].shift())
 .assign(Absolute_Diff=pd.col('Passengers') - pd.col('Prev_Passengers'),
         Pct_Diff=pd.col('Passengers') / pd.col('Prev_Passengers') - 1)
 .sort_values('Month')
 .filter(['Date', 'Passengers', 'Absolute_Diff', 'Pct_Diff'])
 )
to_plot

,Date,Passengers,Absolute_Diff,Pct_Diff
0,2019-01-01,61694899,NaN,NaN
48,2023-01-01,60906882,14710784.0,0.318442
24,2021-01-01,24923269,-40009354.0,-0.616167
60,2024-01-01,64647251,3740369.0,0.061411
36,2022-01-01,46196098,21272829.0,0.853533
...,...,...,...,...
59,2023-12-01,72562140,6430730.0,0.097242
23,2020-12-01,27785059,-45302439.0,-0.619838
47,2022-12-01,66131410,6791820.0,0.114457
71,2024-12-01,77384934,4822794.0,0.066464


In [145]:
from bokeh.models import NumeralTickFormatter

passenger_plot = (to_plot
 .hvplot.line(
     x='Date',
     y='Passengers',
     height=210)
 .opts(
     xaxis=None,
     yformatter=NumeralTickFormatter(format='0a'))
)

abs_diff_plot = (to_plot
 .hvplot.line(
     x='Date',
     y='Absolute_Diff',
     ylabel='Absolute difference',
     height=210)
 .opts(
     xaxis=None,
     yformatter=NumeralTickFormatter(format='0a'))
)

pct_diff_plot = (to_plot
 .hvplot.line(
     x='Date',
     y='Pct_Diff',
     xlabel='Month',
     ylabel='Percent difference',
     height=210)
 .opts(yformatter=NumeralTickFormatter(format='0%'))
)

(passenger_plot + abs_diff_plot + pct_diff_plot).cols(1)

:Layout
   .Curve.I   :Curve   [Date]   (Passengers)
   .Curve.II  :Curve   [Date]   (Absolute_Diff)
   .Curve.III :Curve   [Date]   (Pct_Diff)

In [146]:
to_plot = (tsa_passengers
 .sort_values('Date')
 .groupby(pd.Grouper(key='Date', freq='MS'), as_index=False)
 .agg(Passengers=('Passengers', 'sum'))
 .loc[pd.col('Date').between('2021-01-01', '2023-12-01')]
 .assign(Month_Num=pd.col('Date').dt.month,
         Month_Name=pd.col('Date').dt.strftime('%B'),
         Year=pd.col('Date').dt.year)
 .pivot_table(index=['Month_Num', 'Month_Name'],
              columns='Year', values='Passengers',
              aggfunc='max') # <1>
 .sort_index(level='Month_Num')
 .reset_index()
 .rename(columns=lambda c: f'Passengers_{c}' if isinstance(c, int) else c)
 )
to_plot

Year,Month_Num,Month_Name,Passengers_2021,Passengers_2022,Passengers_2023
0,1,January,24923269,46196098,60906882
1,2,February,24633449,48709041,58361130
2,3,March,38144734,63906622,71919462
3,4,April,42087425,63746796,70216926
4,5,May,50300959,67560029,74561756
5,6,June,57255026,68760899,77156198
6,7,July,63836626,71078972,79847051
7,8,August,57804919,67806258,75366022
8,9,September,51262899,63575705,70213668
9,10,October,57659583,67792513,75729083


In [147]:
(to_plot
 .rename(columns=lambda c: c.replace('Passengers_', ''))
 .hvplot.line(x='Month_Name', y=['2021', '2022', '2023'],
              height=400, width=800, legend='top_left',
              legend_opts={'title': 'Years'},
              xlabel='', ylabel='Passengers', color='#1a1a1a',
              line_dash=['dashed', 'solid', 'dotted'],
              title='TSA checkpoint monthly passenger volume (2021 to 2023)\nAligned by month')
 .opts(yformatter=NumeralTickFormatter(format='0a'))
 )

:NdOverlay   [Year]
   :Curve   [Month_Name]   (value)

### Comparing to Multiple Prior Periods

In [148]:
(tsa_passengers
 .sort_values('Date')
 .groupby(pd.Grouper(key='Date', freq='MS'), as_index=False)
 .agg(Passengers=('Passengers', 'sum'))
 .assign(Month_Num=pd.col('Date').dt.month)
 .assign(Prev_Passengers_1=lambda d: d.groupby('Month_Num')['Passengers'].shift(periods=1),
         Prev_Passengers_2=lambda d: d.groupby('Month_Num')['Passengers'].shift(2), # <1>
         Prev_Passengers_3=lambda d: d.groupby('Month_Num')['Passengers'].shift(3))
 .sort_values('Month_Num')
 .filter(['Date', 'Passengers', 'Prev_Passengers_1', 'Prev_Passengers_2', 'Prev_Passengers_3'])
 )

,Date,Passengers,Prev_Passengers_1,Prev_Passengers_2,Prev_Passengers_3
0,2019-01-01,61694899,NaN,NaN,NaN
48,2023-01-01,60906882,46196098.0,24923269.0,64932623.0
24,2021-01-01,24923269,64932623.0,61694899.0,NaN
60,2024-01-01,64647251,60906882.0,46196098.0,24923269.0
36,2022-01-01,46196098,24923269.0,64932623.0,61694899.0
...,...,...,...,...,...
59,2023-12-01,72562140,66131410.0,59339590.0,27785059.0
23,2020-12-01,27785059,73087498.0,NaN,NaN
47,2022-12-01,66131410,59339590.0,27785059.0,73087498.0
71,2024-12-01,77384934,72562140.0,66131410.0,59339590.0


In [149]:
(tsa_passengers
 .sort_values('Date')
 .groupby(pd.Grouper(key='Date', freq='MS'), as_index=False)
 .agg(Passengers=('Passengers', 'sum'))
 .assign(Month_Num=pd.col('Date').dt.month)
 .assign(Prev_Passengers_1=lambda d: d.groupby('Month_Num')['Passengers'].shift(1),
         Prev_Passengers_2=lambda d: d.groupby('Month_Num')['Passengers'].shift(2),
         Prev_Passengers_3=lambda d: d.groupby('Month_Num')['Passengers'].shift(3))
 .assign(Avg_Prev=lambda d: d[['Prev_Passengers_1', 'Prev_Passengers_2', 'Prev_Passengers_3']]
         .mean(axis=1, skipna=False)) # <1>
 .assign(Pct_of_3_Prev=pd.col('Passengers') / pd.col('Avg_Prev') * 100)
 .sort_values('Month_Num')
 .filter(['Date', 'Passengers', 'Pct_of_3_Prev'])
 )

,Date,Passengers,Pct_of_3_Prev
0,2019-01-01,61694899,NaN
48,2023-01-01,60906882,134.302075
24,2021-01-01,24923269,NaN
60,2024-01-01,64647251,146.896359
36,2022-01-01,46196098,91.446764
...,...,...,...
59,2023-12-01,72562140,142.040988
23,2020-12-01,27785059,NaN
47,2022-12-01,66131410,123.832202
71,2024-12-01,77384934,117.230279


In [150]:
(tsa_passengers
 .sort_values('Date')
 .groupby(pd.Grouper(key='Date', freq='MS'), as_index=False)
 .agg(Passengers=('Passengers', 'sum'))
 .assign(Month_Num=pd.col('Date').dt.month)
 .assign(Avg_Prev=lambda d: d.groupby('Month_Num')['Passengers']
         .transform(lambda s: s.rolling(window=3).mean().shift(1)))
 .assign(Pct_of_Prev_3=pd.col('Passengers') / pd.col('Avg_Prev') * 100)
 .sort_values('Month_Num')
 .filter(['Date', 'Passengers', 'Pct_of_Prev_3'])
 )

,Date,Passengers,Pct_of_Prev_3
0,2019-01-01,61694899,NaN
48,2023-01-01,60906882,134.302075
24,2021-01-01,24923269,NaN
60,2024-01-01,64647251,146.896359
36,2022-01-01,46196098,91.446764
...,...,...,...
59,2023-12-01,72562140,142.040988
23,2020-12-01,27785059,NaN
47,2022-12-01,66131410,123.832202
71,2024-12-01,77384934,117.230279


### Common Business Questions on Time Series

In [151]:
(tsa_passengers
 .sort_values('Date')
 .assign(Avg_14_Day=pd.col('Passengers').rolling(window=14, min_periods=0).mean())
 .rename(columns={'Passengers': 'Actual'})
 .hvplot.line(x='Date', y=['Actual','Avg_14_Day'],
              title='TSA daily passenger traffic',
              height=400, color=['#c0bfbc','#1a1a1a'],
              legend_cols=2, yformatter='%.0f',
              legend_opts={'title':''}, legend='top_left')
 .opts(yformatter=NumeralTickFormatter(format='0a'))
 )

:NdOverlay   [Variable]
   :Curve   [Date]   (value)

In [152]:
(tsa_passengers
 .loc[pd.col('Date').dt.year == 2024]
 .sort_values('Date')
 .groupby(pd.Grouper(key='Date', freq='MS'), as_index=False)
 .agg(Passengers=('Passengers', 'sum'))
 .assign(Pct_Change=pd.col('Passengers').pct_change() * 100)
 .assign(Date=pd.col('Date').dt.strftime('%b'))
 .assign(Color=lambda d: np.where(d['Pct_Change'] < 0, '#c0bfbc', '#1a1a1a'))
 .hvplot.bar(x='Date', y='Pct_Change', color='Color',
             yformatter='%.0f%%', xlabel='', ylabel='',
             title='Month over month percent change (2024)')
 )

:Bars   [Date]   (Pct_Change,Color)

In [153]:
(tsa_passengers
 .sort_values('Date')
 .groupby(pd.Grouper(key='Date', freq='7D'), as_index=False)
 .agg(Passengers=('Passengers', 'sum'))
 .loc[pd.col('Date').dt.year == 2024]
 .hvplot.scatter(
     x='Date', xlabel='',
     y='Passengers',
     color='Passengers', # <1>
     cmap='Blues',
     size=80,
     colorbar=False,               
     yformatter='%.0f',
     title='TSA passenger volume\n(weekly average 2024)')
 .opts(yformatter=NumeralTickFormatter(format='0a'))
)

:Scatter   [Date]   (Passengers)

In [154]:
import holoviews as hv
from datetime import datetime

actual = (tsa_passengers
 .sort_values('Date')
 .groupby(pd.Grouper(key='Date', freq='7D'), as_index=False)
 .agg(Passengers=('Passengers', 'sum'))
 .loc[pd.col('Date').dt.year == 2024]
 .hvplot.line(x='Date', y='Passengers',
              xlabel='', ylabel='',
              line_dash='solid', color='#1a1a1a')
 )

annotation = hv.Text(
    x=datetime(2024, 2, 7),
    y=17.5e6,
    text='Average',
    fontsize=10
).opts(text_align="right", text_baseline="bottom", color="grey")

avg = (tsa_passengers
 .sort_values('Date')
 .groupby(pd.Grouper(key='Date', freq='7D'), as_index=False)
 .agg(Passengers=('Passengers', 'sum'))
 .loc[pd.col('Date').dt.year == 2024]
 .assign(Avg=pd.col('Passengers').mean())
 .hvplot.line(x='Date', y='Avg', line_dash='dashed',
              color='#c0bfbc', line_width=3)
 )

(actual * avg * annotation).opts(
    title='TSA weekly average passenger volume (2024)',
    legend_cols=2,
    legend_position='top_left',
    legend_opts={'title': ''},
    yformatter=NumeralTickFormatter(format='0a')
)

:Overlay
   .Curve.I  :Curve   [Date]   (Passengers)
   .Curve.II :Curve   [Date]   (Avg)
   .Text.I   :Text   [x,y]

In [155]:
(tsa_passengers
 .loc[pd.col('Date').dt.year == 2024]
 .assign(Day=pd.col('Date').dt.day,
         Month=pd.col('Date').dt.strftime('%B'))
 .loc[pd.col('Date').dt.month.isin([10, 12])]
 .pivot_table(index='Day', columns='Month', values='Passengers')
 .reset_index()
 .assign(October=pd.col('October').cumsum(),
         December=pd.col('December').cumsum())
 .hvplot.line(x='Day', y=['October','December'],
              height=500, legend='top_left',
              legend_cols=2, legend_opts={'title':''},
              color=['#1a1a1a','#c0bfbc'],
              title='TSA passenger volume running daily total (2024)')
 .opts(yformatter=NumeralTickFormatter(format='0a'))
 )

:NdOverlay   [Month]
   :Curve   [Day]   (value)

# Text Analysis
## The YouTube Comments Dataset

In [156]:
import pandas as pd
from pathlib import Path

files = Path(f'{data}').glob('*.json')

dfs = []
for file in files:
    df = (pd.read_json(file, lines=True) # <1>
     .assign(source=file.name)
     .rename(columns=str.title)
    )
    dfs.append(df)

comments_df = pd.concat(dfs, ignore_index=True)

In [157]:
youtube_comments = (comments_df
 .assign(Date=lambda d: pd.to_datetime(d['Time_Parsed'], unit='s'))
 .filter(['Date', 'Time', 'Text', 'Author', 'Source'])
 )
youtube_comments

,Date,Time,Text,Author,Source
0,2022-12-11 16:04:44.790007114,3 years ago (edited),It took me 7 years of working as a Data Analys...,@Stefanovic92,stefanovic.json
1,2025-06-11 15:04:44.793478966,6 months ago,1:50 Fiat Multipla is indeed the best car ever...,@sasker1000,stefanovic.json
2,2025-06-11 15:04:44.794984102,6 months ago,Bro has beautiful eyes for no reason,@cpblair,stefanovic.json
3,2025-06-11 15:04:44.797435999,6 months ago (edited),"Man, I did it. I actually listed what you sugg...",@cometamítico,stefanovic.json
4,2025-03-11 15:04:44.801501036,9 months ago,I think I clicked on this video because the co...,@Hervoicehealed,stefanovic.json
...,...,...,...,...,...
4834,2025-04-11 14:49:31.572190046,8 months ago,"Like with women, the numbers game + persistenc...",@kknight1990,avery_smith.json
4835,2024-12-11 15:49:31.777411938,1 year ago,I can answer that if you want\n\nTo me both to...,@vvs1998,avery_smith.json
4836,2024-12-11 15:49:31.780677080,1 year ago (edited),@vvs1998 Tableau Public is free and doesn’t r...,@Nikopup,avery_smith.json
4837,2025-01-11 15:49:31.784425974,11 months ago (edited),​Thanks ​ @vvs1998 for your honesty,@adedapomichaelowoeye9353,avery_smith.json


## Text Characteristics

In [158]:
df = pd.DataFrame({'Text': ['👍',
  'Sample comment']})
df

,Text
0,👍
1,Sample comment


In [159]:
(df
 .assign(Chars=pd.col('Text').str.len(),
         Bytes=pd.col('Text').str.encode('utf-8').str.len())
 )

,Text,Chars,Bytes
0,👍,1,4
1,Sample comment,14,14


In [160]:
to_plot = (youtube_comments
 .assign(Length=pd.col('Text').str.len())
 .groupby('Length', as_index=False)
 .agg(Records=('Text', 'size'))
 .sort_values('Length')
)
to_plot

,Length,Records
0,1,23
1,2,10
2,3,11
3,4,20
4,5,16
...,...,...
568,1830,1
569,2555,1
570,2858,2
571,3616,1


In [161]:
import hvplot.pandas

(to_plot
 .hvplot.bar(x='Length', y='Records', height=400,
             title='Distribution of comments length')
 )

:Bars   [Length]   (Records)

In [162]:
print(youtube_comments['Time'].iloc[0])
print(youtube_comments['Time'].iloc[1])

3 years ago (edited)
6 months ago


## Text Parsing

In [163]:
(youtube_comments
 .filter(['Time'])
 .assign(Time=pd.col('Time').str.slice(-3))
 )

,Time
0,ed)
1,ago
2,ago
3,ed)
4,ago
...,...
4834,ago
4835,ago
4836,ed)
4837,ed)


In [164]:
(youtube_comments
 .filter(['Time'])
 .assign(Time=pd.col('Time').str.split(' (edited)', regex=False).str.get(0)) # <1>
 .assign(Time=pd.col('Time').str.slice(-3))
 )

,Time
0,ago
1,ago
2,ago
3,ago
4,ago
...,...
4834,ago
4835,ago
4836,ago
4837,ago


In [165]:
(youtube_comments
 .filter(['Time'])
 .assign(Time=pd.col('Time').str.split(' (edited)', regex=False).str.get(0))
 .assign(Time=pd.col('Time').str.slice(-3))
 .groupby('Time', as_index=False)
 .agg(Records=('Time', 'size'))
 )

,Time,Records
0,ago,4839


In [166]:
(youtube_comments
 .filter(['Time'])
 .assign(Time=pd.col('Time').str.split(' ', regex=False))
 .assign(Digit=pd.col('Time').str.get(0).astype('int8'),
         Duration=pd.col('Time').str[1], # <1>
         Ago=pd.col('Time').str.get(2),
         Edit=pd.col('Time').str.get(3))
 )

,Time,Digit,Duration,Ago,Edit
0,"[3, years, ago, (edited)]",3,years,ago,(edited)
1,"[6, months, ago]",6,months,ago,NaN
2,"[6, months, ago]",6,months,ago,NaN
3,"[6, months, ago, (edited)]",6,months,ago,(edited)
4,"[9, months, ago]",9,months,ago,NaN
...,...,...,...,...,...
4834,"[8, months, ago]",8,months,ago,NaN
4835,"[1, year, ago]",1,year,ago,NaN
4836,"[1, year, ago, (edited)]",1,year,ago,(edited)
4837,"[11, months, ago, (edited)]",11,months,ago,(edited)


## Text Transformations

In [167]:
(youtube_comments
 .assign(Original='Deep Analysis with pandas')
 .filter(['Original'])
 .assign(Upper=pd.col('Original').str.upper(),
         Lower=pd.col('Original').str.lower())
 .head(1)
 )

,Original,Upper,Lower
0,Deep Analysis with pandas,DEEP ANALYSIS WITH PANDAS,deep analysis with pandas


In [168]:
(youtube_comments
 .assign(Language='cLoJuRe',
         Creator='rich hickey')
 .filter(['Language', 'Creator'])
 .assign(Title_Language=pd.col('Language').str.title(),
         Title_Creator=pd.col('Creator').str.title())
 .head(1)
 )

,Language,Creator,Title_Language,Title_Creator
0,cLoJuRe,rich hickey,Clojure,Rich Hickey


In [169]:
(youtube_comments
 .filter(['Text'])
 .assign(Text=pd.col('Text').str.split(' '))
 .explode('Text')
 .loc[pd.col('Text').str.lower().eq('python')]
 .rename(columns={'Text': 'Language'})
 .assign(Language_Clean=pd.col('Language').str.title())
 .drop_duplicates('Language')
)

,Language,Language_Clean
45,Python,Python
51,python,Python
4312,PYTHON,Python


In [170]:
(youtube_comments
 .assign(Language='   Clojure   ')
 .filter(['Language'])
 .assign(Language_Stripped=pd.col('Language').str.strip())
 .head(1)
 )

,Language,Language_Stripped
0,Clojure,Clojure


In [171]:
(youtube_comments
 .assign(Language='   Clojure   ')
 .filter(['Language'])
 .assign(Strip_Start=pd.col('Language').str.lstrip(),
         Strip_End=pd.col('Language').str.rstrip())
 .head(1)
 )

,Language,Strip_Start,Strip_End
0,Clojure,Clojure,Clojure


In [172]:
(youtube_comments
 .assign(Source=pd.col('Source').str.removesuffix('.json'))
 )

,Date,Time,Text,Author,Source
0,2022-12-11 16:04:44.790007114,3 years ago (edited),It took me 7 years of working as a Data Analys...,@Stefanovic92,stefanovic
1,2025-06-11 15:04:44.793478966,6 months ago,1:50 Fiat Multipla is indeed the best car ever...,@sasker1000,stefanovic
2,2025-06-11 15:04:44.794984102,6 months ago,Bro has beautiful eyes for no reason,@cpblair,stefanovic
3,2025-06-11 15:04:44.797435999,6 months ago (edited),"Man, I did it. I actually listed what you sugg...",@cometamítico,stefanovic
4,2025-03-11 15:04:44.801501036,9 months ago,I think I clicked on this video because the co...,@Hervoicehealed,stefanovic
...,...,...,...,...,...
4834,2025-04-11 14:49:31.572190046,8 months ago,"Like with women, the numbers game + persistenc...",@kknight1990,avery_smith
4835,2024-12-11 15:49:31.777411938,1 year ago,I can answer that if you want\n\nTo me both to...,@vvs1998,avery_smith
4836,2024-12-11 15:49:31.780677080,1 year ago (edited),@vvs1998 Tableau Public is free and doesn’t r...,@Nikopup,avery_smith
4837,2025-01-11 15:49:31.784425974,11 months ago (edited),​Thanks ​ @vvs1998 for your honesty,@adedapomichaelowoeye9353,avery_smith


In [173]:
youtube_comments_clean = (youtube_comments
 .assign(Source=pd.col('Source').str.removesuffix('.json'))
 .assign(Source=pd.col('Source').str.replace('_', ' ', regex=False).str.title())
 )
youtube_comments_clean

,Date,Time,Text,Author,Source
0,2022-12-11 16:04:44.790007114,3 years ago (edited),It took me 7 years of working as a Data Analys...,@Stefanovic92,Stefanovic
1,2025-06-11 15:04:44.793478966,6 months ago,1:50 Fiat Multipla is indeed the best car ever...,@sasker1000,Stefanovic
2,2025-06-11 15:04:44.794984102,6 months ago,Bro has beautiful eyes for no reason,@cpblair,Stefanovic
3,2025-06-11 15:04:44.797435999,6 months ago (edited),"Man, I did it. I actually listed what you sugg...",@cometamítico,Stefanovic
4,2025-03-11 15:04:44.801501036,9 months ago,I think I clicked on this video because the co...,@Hervoicehealed,Stefanovic
...,...,...,...,...,...
4834,2025-04-11 14:49:31.572190046,8 months ago,"Like with women, the numbers game + persistenc...",@kknight1990,Avery Smith
4835,2024-12-11 15:49:31.777411938,1 year ago,I can answer that if you want\n\nTo me both to...,@vvs1998,Avery Smith
4836,2024-12-11 15:49:31.780677080,1 year ago (edited),@vvs1998 Tableau Public is free and doesn’t r...,@Nikopup,Avery Smith
4837,2025-01-11 15:49:31.784425974,11 months ago (edited),​Thanks ​ @vvs1998 for your honesty,@adedapomichaelowoeye9353,Avery Smith


In [174]:
(youtube_comments_clean
 .groupby('Source', as_index=False)
 .agg(len=('Source', 'size'))
 .sort_values('len')
 .hvplot.barh(x='Source',
              y='len',
              height=400,
              xlabel='',
              ylabel='',
              title='Distribution of comments by video source')
 .opts(fontsize={'xticks': 11, 'yticks': 11})
 )

:Bars   [Source]   (len)

## Finding Elements Within Larger Blocks of Text
### Wildcard Matches: contains

In [175]:
(youtube_comments_clean
 .loc[pd.col('Text').str.contains('SQL')]
 .assign(Count=pd.col('Text').nunique())
 .filter(['Count'])
 .head(1)
 )

,Count
42,125


In [176]:
(youtube_comments_clean
 .loc[pd.col('Text').str.upper().str.contains('SQL')]
 .assign(Count=pd.col('Text').nunique())
 .filter(['Count'])
 .head(1)
 )

,Count
19,178


In [177]:
(youtube_comments_clean
 .loc[pd.col('Text').str.contains('SQL|sql|Sql')]
 .assign(Count=pd.col('Text').nunique())
 .filter(['Count'])
 .head(1)
 )

,Count
19,178


In [178]:
(youtube_comments_clean
 .loc[pd.col('Text').str.contains('sql', case=False)]
 .assign(Count=pd.col('Text').nunique())
 .filter(['Count'])
 .head(1)
 )

,Count
19,178


In [179]:
(youtube_comments_clean
 .loc[~pd.col('Text').str.contains('sql', case=False)]
 .assign(Count=pd.col('Text').nunique())
 .filter(['Count'])
 .head(1)
 )

,Count
0,4452


In [180]:
(youtube_comments_clean
 .loc[pd.col('Text').str.contains('SQL') | pd.col('Text').str.contains('Python')]
 .assign(Count=pd.col('Text').nunique())
 .filter(['Count'])
 .head(1)
 )

,Count
42,189


In [181]:
(youtube_comments_clean
 .loc[pd.col('Text').str.contains('SQL|Python')]
 .assign(Count=pd.col('Text').nunique())
 .filter(['Count'])
 .head(1)
 )

,Count
42,189


In [182]:
(youtube_comments_clean
 .loc[pd.col('Text').str.contains('SQL') & pd.col('Text').str.contains('Python')]
 .assign(Count=pd.col('Text').nunique())
 .filter(['Count'])
 .head(1)
 )

,Count
42,41


#### Version 1

In [183]:
(youtube_comments_clean
 .loc[pd.col('Text').str.contains('SQL') | pd.col('Text').str.contains('Python') &
      pd.col('Text').str.contains('excel')]
 .assign(Count=pd.col('Text').nunique())
 .filter(['Count'])
 .head(1)
 )

,Count
42,130


#### Version 2

In [184]:
(youtube_comments_clean
 .loc[(pd.col('Text').str.contains('SQL') | pd.col('Text').str.contains('Python')) & 
      pd.col('Text').str.contains('excel')]
 .assign(Count=pd.col('Text').nunique())
 .filter(['Count'])
 .head(1)
 )

,Count
54,18


In [185]:
import numpy as np

(youtube_comments_clean
 .assign(Text=pd.col('Text').str.lower())
 .assign(Technology=lambda d: np.select(
     [d['Text'].str.contains('power bi|powerbi'),
      d['Text'].str.contains('python'),
      d['Text'].str.contains('sql'),
      d['Text'].str.contains('tableau'),
      d['Text'].str.contains('excel')],
     ['Power BI', 'Python', 'SQL', 'Tableau', 'Excel'],
     default=None))
 .groupby('Technology', as_index=False, dropna=False)
 .agg(Count=('Technology', 'size'))
 .sort_values('Count', ascending=False)
)

,Technology,Count
5,NaN,4360
2,Python,246
3,SQL,81
1,Power BI,75
0,Excel,65
4,Tableau,12


In [186]:
(youtube_comments_clean
 .assign(Text=pd.col('Text').str.lower())
 .assign(Power_BI=pd.col('Text').str.contains('power bi|powerbi'),
         Python=pd.col('Text').str.contains('python'),
         SQL=pd.col('Text').str.contains('sql'),
         Tableau=pd.col('Text').str.contains('tableau'),
         Excel=pd.col('Text').str.contains('excel'))
 .groupby(['Power_BI', 'Python', 'SQL', 'Tableau', 'Excel'], as_index=False)
 .agg(Count=('Text', 'size'))
 .sort_values('Count', ascending=False)
)

,Power_BI,Python,SQL,Tableau,Excel,Count
0,False,False,False,False,False,4360
8,False,True,False,False,False,174
1,False,False,False,False,True,65
4,False,False,True,False,False,51
11,False,True,True,False,False,33
17,True,False,False,True,False,18
12,False,True,True,False,True,18
5,False,False,True,False,True,15
15,True,False,False,False,False,12
2,False,False,False,True,False,11


In [187]:
to_plot = (youtube_comments_clean
 .assign(Text=pd.col('Text').str.lower())
 .assign(Power_BI=pd.col('Text').str.contains('power bi|powerbi'),
         Python=pd.col('Text').str.contains('python'),
         SQL=pd.col('Text').str.contains('sql'),
         Tableau=pd.col('Text').str.contains('tableau'),
         Excel=pd.col('Text').str.contains('excel'))
 .filter(['Power_BI', 'Python', 'SQL', 'Tableau', 'Excel'])
 .sum()
)
to_plot

Power_BI     75
Python      269
SQL         178
Tableau      79
Excel       154
dtype: int64

In [188]:
(to_plot
 .to_frame('Count')
 .sort_values('Count')
 .hvplot.barh(x='index', y='Count', 
              xlabel='', ylabel='', height=400,
              title='Frequency of technologies mentioned in comments')
 .opts(fontsize={'xticks': 11, 'yticks': 11})
 )

:Bars   [index]   (Count)

### Exact Matches: isin

In [189]:
comments_1st_word_df = (youtube_comments_clean
 .assign(First_Word=pd.col('Text').str.split(' ').str[0])
 .filter(['First_Word', 'Text'])
)

pd.concat([
    comments_1st_word_df.loc[pd.col('First_Word') == 'Thanks'],
    comments_1st_word_df.loc[pd.col('First_Word') == 'Great'],
    comments_1st_word_df.loc[pd.col('First_Word') == 'How'],
    comments_1st_word_df.loc[pd.col('First_Word') == 'Can'],
    comments_1st_word_df.loc[pd.col('First_Word') == 'This'],
    comments_1st_word_df.loc[pd.col('First_Word') == 'What'],
])

,First_Word,Text
118,Thanks,Thanks for the video I'm looking forward to be...
136,Thanks,Thanks for this content man... Greetings from ...
196,Thanks,Thanks man!
209,Thanks,Thanks you for the video
214,Thanks,Thanks but I don’t want to be in front of a la...
...,...,...
4672,What,What if my degree is a healthcare management a...
4709,What,What about statistics and mathematics?
4717,What,What projects should I do if I want to land a ...
4738,What,What about supply chain?


In [190]:
word_dfs = []
for word in ['Thanks', 'Great', 'How', 'Can', 'This', 'What']:
    word_df = comments_1st_word_df.loc[pd.col('First_Word') == word]
    word_dfs.append(word_df)
pd.concat(word_dfs)

,First_Word,Text
118,Thanks,Thanks for the video I'm looking forward to be...
136,Thanks,Thanks for this content man... Greetings from ...
196,Thanks,Thanks man!
209,Thanks,Thanks you for the video
214,Thanks,Thanks but I don’t want to be in front of a la...
...,...,...
4672,What,What if my degree is a healthcare management a...
4709,What,What about statistics and mathematics?
4717,What,What projects should I do if I want to land a ...
4738,What,What about supply chain?


In [191]:
to_plot = (youtube_comments_clean
 .assign(First_Word=pd.col('Text').str.split(' ').str[0])
 .loc[pd.col('First_Word').isin(['Thanks', 'Great', 'How', 'Can', 'This', 'What'])]
 .filter(['First_Word', 'Text'])
)
to_plot

,First_Word,Text
6,This,This is a lot mehnnnn.....
18,Great,"Great content, I hope you keep on creating con..."
21,How,How can I get a remote data analysis job? Is i...
36,Can,Can you recommend any complete course from cou...
42,This,This is exactly how I started learning Data An...
...,...,...
4780,This,This is changing fast imo
4782,How,How about qlikview?
4783,Thanks,Thanks
4796,Great,Great stuff !!


In [192]:
(to_plot
 .groupby('First_Word', as_index=False)
 .agg(len=('First_Word', 'size'))
 .sort_values('len', ascending=False)
 .hvplot.bar(x='First_Word',
             y='len',
             xlabel='',
             ylabel='Count',
             height=400,
             title='Frequency of select words used as the first word in comments')
 .opts(fontsize={'xticks': 11, 'yticks': 11})        
 )

:Bars   [First_Word]   (len)

### Regular Expressions

In [193]:
(youtube_comments_clean
 .filter(['Text'])
 .loc[pd.col('Text').str.contains(r'.')]
 )

,Text
0,It took me 7 years of working as a Data Analys...
1,1:50 Fiat Multipla is indeed the best car ever...
2,Bro has beautiful eyes for no reason
3,"Man, I did it. I actually listed what you sugg..."
4,I think I clicked on this video because the co...
...,...
4834,"Like with women, the numbers game + persistenc..."
4835,I can answer that if you want\n\nTo me both to...
4836,@vvs1998 Tableau Public is free and doesn’t r...
4837,​Thanks ​ @vvs1998 for your honesty


In [194]:
(youtube_comments_clean
 .filter(['Text'])
 .loc[~pd.col('Text').str.contains(r'^\w')]
 )

,Text
41,🚀 Revolutionize Your Data Management with the ...
90,@Stefanovic92 thanks bro!
116,@Stefanovic92 thank you😊🙏
139,@Stefanovic92 ya i wanted to get it but turn...
223,🇧🇷
...,...
4827,😂😂😂
4828,@Maubinyaachi
4833,@govinddangi9655 it’s really about who you kn...
4836,@vvs1998 Tableau Public is free and doesn’t r...


In [195]:
(pd.DataFrame({'Original': ['The data is about YouTube comments.']})
 .loc[pd.col('Original').str.contains(r'[Tt]he')]
 )

,Original
0,The data is about YouTube comments.


In [196]:
(pd.DataFrame({'Original': ['tHe data is about YouTube comments.']})
 .loc[pd.col('Original').str.contains(r'[Tt]he')]
 )

,Original


In [197]:
(youtube_comments_clean
 .filter(['Time'])
 .loc[pd.col('Time').str.contains(r'[234] month')]
 )

,Time
24,4 months ago
490,2 months ago
2487,2 months ago
2488,4 months ago
2489,4 months ago
...,...
4810,2 months ago
4814,3 months ago
4815,2 months ago
4830,3 months ago


In [198]:
(youtube_comments_clean
 .filter(['Time'])
 .loc[pd.col('Time').str.contains(r'[2-4] month')]
 )

,Time
24,4 months ago
490,2 months ago
2487,2 months ago
2488,4 months ago
2489,4 months ago
...,...
4810,2 months ago
4814,3 months ago
4815,2 months ago
4830,3 months ago


In [199]:
(pd.DataFrame({'Original': ["I've clocked 126 days as a Data Analyst."]})
 .loc[pd.col('Original').str.contains(r'[0-9][0-9][0-9] days')]
 )

,Original
0,I've clocked 126 days as a Data Analyst.


In [203]:
(pd.DataFrame({'Original': ["I've clocked 126 days as a Data Analyst."]})
 .loc[pd.col('Original').str.contains(r'[0-9]+ days')]
 )

,Original
0,I've clocked 126 days as a Data Analyst.


In [204]:
sample_text = [
    '"Should I learn Tableau?" He asked',
    'The course fee was under ^72.',
]

(pd.DataFrame({'Original': sample_text})
 .loc[pd.col('Original').str.contains(r'\?')] # <1>
 )

,Original
0,"""Should I learn Tableau?"" He asked"


In [205]:
(pd.DataFrame({'Original': sample_text})
 .loc[pd.col('Original').str.contains(r'^[0-9]+')]
 )

,Original


In [206]:
(pd.DataFrame({'Original': sample_text})
 .loc[pd.col('Original').str.contains(r'\^[0-9]+')]
 )

,Original
1,The course fee was under ^72.


In [207]:
sample_text = [
'''filtering
pivoting
and aggregating''',

'''filtering
pivoting
and aggregating''',

'''filtering pivoting''',

'''aggregating'''
]

In [208]:
sample_text[0]

'filtering\npivoting\nand aggregating'

In [209]:
(youtube_comments_clean
 .head(1)
 .assign(Comparison_1=sample_text[0],
         Comparison_2=sample_text[1],
         Comparison_3=sample_text[2],
         Comparison_4=sample_text[3])
 .filter(['Comparison_1', 'Comparison_2', 'Comparison_3', 'Comparison_4'])
 .assign(Comparison_1=pd.col('Comparison_1').str.contains(r'\n'),
         Comparison_2=pd.col('Comparison_2').str.contains(r'\s'),
         Comparison_3=pd.col('Comparison_3').str.contains(r'\s'),
         Comparison_4=pd.col('Comparison_4').str.contains(r'\s'))
 )

,Comparison_1,Comparison_2,Comparison_3,Comparison_4
0,True,True,True,False


In [210]:
sample_text = [
'valid codes follow the pattern 12a34b56c',
'the first code entered was 123a456c',
'the second code entered was 99x66y33z',
]

(youtube_comments_clean
 .head(1)
 .assign(Comparison_1=sample_text[0],
         Comparison_2=sample_text[1],
         Comparison_3=sample_text[2])
 .filter(['Comparison_1', 'Comparison_2', 'Comparison_3'])
 .assign(Comparison_1=pd.col('Comparison_1').str.contains(r'[0-9]{2}[a-z]{3}'),
         Comparison_2=pd.col('Comparison_2').str.contains(r'[0-9]{2}[a-z]{3}'),
         Comparison_3=pd.col('Comparison_3').str.contains(r'[0-9]{2}[a-z]{3}'))
 )

,Comparison_1,Comparison_2,Comparison_3
0,False,False,False


In [211]:
sample_text = [
'I love learning python for data analysis',
'Writing pythonic code requires expertise',
'I find pythonbytes.fm very informative',
'I''m a pythonista who codes every day',
]

(youtube_comments_clean
 .head(1)
 .assign(Comparison=sample_text[0])
 .filter(['Comparison'])
 .assign(Comparison=pd.col('Comparison').str.contains(r'python'))
 )

,Comparison
0,True


In [212]:
(youtube_comments_clean
 .head(1)
 .assign(Comparison_1=sample_text[0],
         Comparison_2=sample_text[1],
         Comparison_3=sample_text[2],
         Comparison_4=sample_text[3])
 .filter(['Comparison_1', 'Comparison_2', 'Comparison_3', 'Comparison_4'])
 .assign(Comparison_1=pd.col('Comparison_1').str.contains(r'python'),
         Comparison_2=pd.col('Comparison_2').str.contains(r'python'),
         Comparison_3=pd.col('Comparison_3').str.contains(r'python'),
         Comparison_4=pd.col('Comparison_4').str.contains(r'python'))
 )

,Comparison_1,Comparison_2,Comparison_3,Comparison_4
0,True,True,True,True


In [213]:
(youtube_comments_clean
 .head(1)
 .assign(Comparison_1=sample_text[0],
         Comparison_2=sample_text[1],
         Comparison_3=sample_text[2],
         Comparison_4=sample_text[3])
 .filter(['Comparison_1', 'Comparison_2', 'Comparison_3', 'Comparison_4'])
 .assign(Comparison_1=pd.col('Comparison_1').str.contains(r'\bpython\b'),
         Comparison_2=pd.col('Comparison_2').str.contains(r'\bpython\b'),
         Comparison_3=pd.col('Comparison_3').str.contains(r'\bpython\b'),
         Comparison_4=pd.col('Comparison_4').str.contains(r'\bpython\b'))
 )

,Comparison_1,Comparison_2,Comparison_3,Comparison_4
0,True,False,False,False


In [214]:
other_text = [
    'python tutorials are the best on YouTube',
    'I love learning python for data analysis',
    'My favorite tool is python',
    'The debate over R vs python is over',
]

(youtube_comments_clean
 .head(1)
 .assign(Comparison_1=other_text[0],
         Comparison_2=other_text[0])
 .filter(['Comparison_1', 'Comparison_2'])
 .assign(Comparison_1=pd.col('Comparison_1').str.contains(r'\bpython\b'),
         Comparison_2=pd.col('Comparison_2').str.contains(r' python '))
 )

,Comparison_1,Comparison_2
0,True,False


In [215]:
(youtube_comments_clean
 .head(1)
 .assign(Comparison_1=other_text[0],
         Comparison_2=other_text[1],
         Comparison_3=other_text[2],
         Comparison_4=other_text[3])
 .filter(['Comparison_1', 'Comparison_2', 'Comparison_3', 'Comparison_4'])
 .assign(Comparison_1=pd.col('Comparison_1').str.contains(r'^python\b'), # <1> 
         Comparison_2=pd.col('Comparison_2').str.contains(r'^python\b'), # <2>
         Comparison_3=pd.col('Comparison_3').str.contains(r'\bpython$'), # <3>
         Comparison_4=pd.col('Comparison_4').str.contains(r'\bpython$')) # <4>
 )

,Comparison_1,Comparison_2,Comparison_3,Comparison_4
0,True,False,True,False


### Finding and replacing with regex

In [216]:
(youtube_comments_clean
 .filter(['Author'])
 .assign(Extracted=pd.col('Author').str.extract(r'([a-zA-Z]+)\d+\b', expand=False)) # <1>
 )

,Author,Extracted
0,@Stefanovic92,Stefanovic
1,@sasker1000,sasker
2,@cpblair,NaN
3,@cometamítico,NaN
4,@Hervoicehealed,NaN
...,...,...
4834,@kknight1990,kknight
4835,@vvs1998,vvs
4836,@Nikopup,NaN
4837,@adedapomichaelowoeye9353,adedapomichaelowoeye


In [217]:
to_plot = (youtube_comments_clean
 .filter(['Text'])
 .assign(Bootcamp=pd.col('Text').str.extract(r'(?i)(\bbootcamp\b)', expand=False), # <1>
         University=pd.col('Text').str.extract(r'(?i)(\buniversity\b)', expand=False),
         College=pd.col('Text').str.extract(r'(?i)(\bcollege\b)', expand=False),
         **{'Online Course':pd.col('Text').str.extract(r'(?i)(\bonline course\b)', expand=False)} # <2>
         )
 .drop(columns='Text')
 .count() # <3>
)
to_plot

Bootcamp         46
University       47
College          67
Online Course    14
dtype: int64

In [218]:
(to_plot
 .to_frame('Count')
 .sort_values('Count', ascending=False)
 .hvplot.bar(x='index', y='Count',
             xlabel='', ylabel='',
             title='Frequency of learning types mentioned')
 )

:Bars   [index]   (Count)

In [219]:
(youtube_comments_clean
 .head(1)
 .assign(Original='crust rust code')
 .filter(['Original'])
 .assign(Normal_Replace=pd.col('Original').str.replace('rust', 'python', regex=False),
         Regex_Replace=pd.col('Original').str.replace(r'\brust\b', 'python', regex=True))
 )

,Original,Normal_Replace,Regex_Replace
0,crust rust code,cpython python code,crust python code


## Constructing and Reshaping Text
### Concatenation

In [220]:
(youtube_comments_clean
 .head(1)
 .assign(Title_1='Deep Analysis',
         Title_2='with pandas')
 .filter(['Title_1', 'Title_2'])
 .assign(Concatenate=pd.col('Title_1').str.cat(pd.col('Title_2'), sep=' '))
 )

,Title_1,Title_2,Concatenate
0,Deep Analysis,with pandas,Deep Analysis with pandas


In [221]:
(youtube_comments_clean
 .head(1)
 .assign(Title_1='Deep Analysis',
         Title_2='with pandas')
 .filter(['Title_1', 'Title_2'])
 .assign(Add=pd.col('Title_1').add(' ') + pd.col('Title_2'),
         Operator=pd.col('Title_1') + ' ' + pd.col('Title_2'))
 )

,Title_1,Title_2,Add,Operator
0,Deep Analysis,with pandas,Deep Analysis with pandas,Deep Analysis with pandas


In [222]:
with pd.option_context('display.max_colwidth', 100): # <1>
    display(youtube_comments_clean
     .assign(Char_Count=pd.col('Text').str.len())
     .nlargest(5, 'Char_Count')
     .assign(Concatenation='By: ' + pd.col('Author') + ', characters: ' +
             pd.col('Char_Count').astype(str) + ', when: ' + pd.col('Time'))
     .filter(['Concatenation'])
     )

,Concatenation
1684,"By: @amdenis, characters: 8251, when: 5 years ago"
1635,"By: @grantmurray6185, characters: 3616, when: 5 years ago"
176,"By: @patrickjane276, characters: 2858, when: 3 years ago"
2004,"By: @mwredfern, characters: 2858, when: 5 years ago"
2029,"By: @mwredfern, characters: 2555, when: 2 years ago"


### Reshaping Text

In [223]:
tech_cols = ['Powerbi', 'Tableau', 'Python', 'SQL', 'Excel']

(youtube_comments_clean
 .filter(['Text', 'Author'])
 .assign(Text=pd.col('Text').str.replace(r'(?i)power bi', 'powerbi', regex=True))
 .assign(Powerbi=pd.col('Text').str.extract(r'(?i)(\bpowerbi\b)', expand=False),
         Tableau=pd.col('Text').str.extract(r'(?i)(\btableau\b)', expand=False),
         Python=pd.col('Text').str.extract(r'(?i)(\bpython\b)', expand=False),
         SQL=pd.col('Text').str.extract(r'(?i)(\bsql\b)', expand=False),
         Excel=pd.col('Text').str.extract(r'(?i)(\bexcel\b)', expand=False))
 .assign(**{col: pd.col(col).str.title() for col in tech_cols}) # <1>
 .drop(columns='Text')
 .melt(id_vars='Author', value_vars=tech_cols, value_name='Technology')
 .drop(columns='variable')
 .dropna(subset=['Technology']) # <2>
 .drop_duplicates()
 .sort_values(['Author', 'Technology'])
 .groupby('Author', as_index=False)
 .agg(Technologies=('Technology', ', '.join))
)

,Author,Technologies
0,,"Excel, Sql"
1,@1truek269,"Powerbi, Tableau"
2,@24yroldchessimprover84,Python
3,@31123fernando,Sql
4,@9030696920,Python
...,...,...
367,@yassineelallaoui7761,"Powerbi, Python"
368,@yunitasitumorang5491,"Excel, Sql"
369,@yunz2164,"Excel, Python, Sql"
370,@zenitsu-3_3,Excel


In [224]:
(youtube_comments_clean
 .head(1)
 .filter(['Text'])
 .assign(Text=pd.col('Text').str.split(' '))
 .explode('Text')
 )

,Text
0,It
0,took
0,me
0,7
0,years
...,...
0,creating
0,a
0,life
0,of


In [225]:
(youtube_comments_clean
 .head(1)
 .assign(Original='Python, SQL, Tableau, Excel')
 .filter(['Original'])
 .assign(Original=pd.col('Original').str.split(','))
 )

,Original
0,"[Python, SQL, Tableau, Excel]"


In [226]:
(youtube_comments_clean
 .head(1)
 .assign(Original='Python, SQL, Tableau, Excel')
 .filter(['Original'])
 .assign(Original=pd.col('Original').str.split(','))
 .explode('Original')
 .groupby(level=0)['Original'].agg(list) # <1>
 .to_frame()
 )

,Original
0,"[Python, SQL, Tableau, Excel]"


In [227]:
(youtube_comments_clean
 .filter(['Text'])
 .assign(Word=pd.col('Text')
         .str.replace(r'\s+', ' ', regex=True) # <1>
         .str.lower()
         .str.split(' '))
 .explode('Word')
 .assign(Word=pd.col('Word').str.replace(r'[^A-Za-z]', '', regex=True)) # <2>
 .loc[pd.col('Word') != '']
 .groupby('Word', as_index=False)
 .agg(Frequency=('Word', 'size'))
 .sort_values('Frequency', ascending=False)
)

,Word,Frequency
3255,i,4296
6819,to,3429
0,a,2837
6697,the,2783
7512,you,2665
...,...,...
3228,hub,1
3227,httpsyoutubezakdzcxlw,1
3226,httpsyoutubexuxwqghsoosihgituzseqzuwgfu,1
3225,httpsyoutubexrtprfobc,1


In [228]:
stop_words = pd.read_parquet(f'{data}/stop_words.parquet')

to_plot = (youtube_comments_clean
 .filter(['Text'])
 .assign(Word=pd.col('Text')
         .str.replace(r'\s+', ' ', regex=True)
         .str.lower()
         .str.split(' '))
 .explode('Word')
 .assign(Word=pd.col('Word').str.replace(r'[^A-Za-z]', '', regex=True))
 .loc[pd.col('Word') != '']
 .groupby('Word', as_index=False)
 .agg(Frequency=('Word', 'size'))
 .loc[~pd.col('Word').isin(stop_words['Stop_Words'])]
 .sort_values('Frequency', ascending=False)
)
to_plot

,Word,Frequency
1617,data,1841
3292,im,734
5821,science,704
6688,thanks,664
7189,video,611
...,...,...
3182,httpswwwyoutubecomwatchvighqzafpo,1
3181,httpswwwyoutubecomwatchvifceyulgzyabchannelkenjee,1
3180,httpswwwyoutubecomwatchvifbjdiexcgabchannelkenjee,1
3179,httpswwwyoutubecomwatchvfnpwffc,1


In [229]:
(to_plot
 .head(10)
 .iloc[::-1] # <1>
 .hvplot.barh(x='Word', y='Frequency',
             xlabel='', height=450)
 .opts(fontsize={'xticks': 11, 'yticks': 11},
       title='Top 10 most common words in comments\n(without stop words)')
 )

:Bars   [Word]   (Frequency)

# Anomaly Detection

## Detecting Outliers
### Sorting to Find Anomalies

In [230]:
import pandas as pd

new_cols = ['Invoice','Date','Store_Num','Store','Address','City','Zip_Code',
 'Location','County_Num','County','Category_Num','Category','Vendor_Num','Vendor',
 'Item_Num','Item','Pack','Bottle_Vol_ml','Cost','Retail','Bottles',
 'Dollars','Volume_Ltr','Volume_Gal']

liquor_sales = (pd.read_parquet(f'{data}/iowa_liquor_sales.parquet')
 .set_axis(new_cols, axis=1)
 .assign(Date=lambda d: pd.to_datetime(d['Date']))
 .loc[pd.col('Date').dt.year.ne(2025)]
 .astype({'Store_Num': 'Int16', 'Vendor_Num': 'Int16', 'Volume_Gal': 'Float64'})
 .assign(Bottle_Vol_ml=pd.col('Bottle_Vol_ml').str.replace(',', '', regex=False).astype('Int32'),
         Bottles=pd.col('Bottles').str.replace(',', '', regex=False).astype('Int32'),
         Cost=pd.col('Cost').str.replace(r'[$,]', '', regex=True).astype('Float32'),
         Retail=pd.col('Retail').str.replace(r'[$,]', '', regex=True).astype('Float32'),
         Dollars=pd.col('Dollars').str.replace(r'[$,]', '', regex=True).astype('Float32'),
         Volume_Ltr=pd.col('Volume_Ltr').str.replace(',', '', regex=False).astype('Float32'))
 )

In [231]:
(liquor_sales
 .filter(['Volume_Gal'])
 .sort_values('Volume_Gal', ascending=False)
 )

,Volume_Gal
103808,998.57
30558079,998.57
19648349,998.57
2597148,998.57
16620360,998.57
...,...
29529204,-95.1
31313400,-120.46
789686,-166.42
30953679,-171.18


In [232]:
(liquor_sales
 .filter(['Volume_Gal'])
 .loc[pd.col('Volume_Gal').ge(0)]
 .sort_values('Volume_Gal', ascending=False)
 )

,Volume_Gal
11378343,998.57
3695699,998.57
2340470,998.57
3294542,998.57
30527869,998.57
...,...
3671027,0.0
31219690,0.0
28125647,0.0
10276439,0.0


In [233]:
(liquor_sales
 .loc[pd.col('Volume_Gal').ge(0)]
 .groupby('Volume_Gal', as_index=False)
 .agg(Transactions=('Invoice', 'count'))
 .assign(Pct_Transactions=(pd.col('Transactions') * 100 / pd.col('Transactions').sum()).round(8))
 .sort_values('Volume_Gal', ascending=False)
)

,Volume_Gal,Transactions,Pct_Transactions
1914,998.57,36,0.000117
1913,997.97,1,0.000003
1912,993.02,1,0.000003
1911,974.79,1,0.000003
1910,972.42,1,0.000003
...,...,...,...
4,0.04,540,0.001756
3,0.03,130137,0.423189
2,0.02,308874,1.004420
1,0.01,585115,1.902721


In [234]:
(liquor_sales
 .loc[pd.col('Volume_Gal').ge(0)]
 .groupby('Volume_Gal', as_index=False)
 .agg(Transactions=('Invoice', 'count'))
 .assign(Pct_Transactions=(pd.col('Transactions') * 100 / pd.col('Transactions').sum()).round(8))
 .sort_values('Volume_Gal')
)

,Volume_Gal,Transactions,Pct_Transactions
0,0.0,651,0.002117
1,0.01,585115,1.902721
2,0.02,308874,1.004420
3,0.03,130137,0.423189
4,0.04,540,0.001756
...,...,...,...
1910,972.42,1,0.000003
1911,974.79,1,0.000003
1912,993.02,1,0.000003
1913,997.97,1,0.000003


In [235]:
(liquor_sales
 .loc[pd.col('Volume_Gal').ge(0) & pd.col('City').eq('DES MOINES')]
 .groupby(['City','Volume_Gal'], as_index=False)
 .agg(Count=('Invoice', 'size'))
 .sort_values(['City','Volume_Gal'], ascending=[True, False])
)

,City,Volume_Gal,Count
781,DES MOINES,951.02,2
780,DES MOINES,951.01,1
779,DES MOINES,912.98,1
778,DES MOINES,871.77,3
777,DES MOINES,871.76,2
...,...,...,...
4,DES MOINES,0.04,25
3,DES MOINES,0.03,10256
2,DES MOINES,0.02,27825
1,DES MOINES,0.01,45050


### Calculating Percentiles and Standard Deviations to Find Anomalies

In [236]:
(liquor_sales
 .loc[pd.col('Volume_Gal').ge(0) & pd.col('City').eq('DES MOINES')]
 .assign(Percentile=lambda d: (d.groupby('City')['Volume_Gal'].rank(method='min') - 1)
         / (d.groupby('City')['Volume_Gal'].transform('count') - 1))
 .groupby(['City','Volume_Gal','Percentile'], as_index=False)
 .agg(Count=('Volume_Gal', 'size'))
 .sort_values(['City','Volume_Gal'], ascending=[True, False])
)

,City,Volume_Gal,Percentile,Count
781,DES MOINES,951.02,1.0,2
780,DES MOINES,951.01,0.999999,1
779,DES MOINES,912.98,0.999999,1
778,DES MOINES,871.77,0.999998,3
777,DES MOINES,871.76,0.999997,2
...,...,...,...,...
4,DES MOINES,0.04,0.031797,25
3,DES MOINES,0.03,0.027875,10256
2,DES MOINES,0.02,0.017237,27825
1,DES MOINES,0.01,0.000013,45050


In [237]:
import numpy as np

(liquor_sales
 .loc[pd.col('Volume_Gal').ge(0) & pd.col('City').eq('DECORAH')]
 .assign(Buckets=lambda d: d.groupby('City')['Volume_Gal']
         .rank(method='first')
         .mul(100) # <1>
         .div(d.groupby('City')['Volume_Gal'].transform('size'))
         .pipe(np.ceil) # <2>
         .astype('Int64'))
 .sort_values(['City', 'Volume_Gal'], ascending=[True, False])
 .filter(['City', 'Volume_Gal', 'Buckets'])
)

,City,Volume_Gal,Buckets
28893753,DECORAH,271.83,100
17834052,DECORAH,269.06,100
21643307,DECORAH,269.06,100
27866346,DECORAH,269.06,100
2336543,DECORAH,269.05,100
...,...,...,...
32556093,DECORAH,0.01,1
32556288,DECORAH,0.01,1
32563871,DECORAH,0.01,1
164544,DECORAH,0.0,1


In [238]:
(liquor_sales
 .loc[pd.col('Volume_Gal').ge(0) & pd.col('City').eq('DECORAH')]
 .assign(Buckets=lambda d: d.groupby('City')['Volume_Gal']
         .rank(method='first')
         .mul(4)
         .div(d.groupby('City')['Volume_Gal'].transform('size'))
         .pipe(np.ceil)
         .astype('Int64'))
 .groupby(['City','Buckets'], as_index=False)
 .agg(Minimum=('Volume_Gal', 'min'),
      Maximum=('Volume_Gal', 'max'))
 .sort_values(['City','Buckets'], ascending=[True, False])
)

,City,Buckets,Minimum,Maximum
3,DECORAH,4,2.77,271.83
2,DECORAH,3,2.38,2.77
1,DECORAH,2,0.79,2.38
0,DECORAH,1,0.0,0.79


In [239]:
(liquor_sales
 .loc[pd.col('Volume_Gal').ge(0) & pd.col('City').eq('DECORAH')]
 .filter(['Volume_Gal'])
 .quantile([0.25, 0.5, 0.75], interpolation='linear')
 .rename({0.25: 'Pct_25', 0.50: 'Pct_50', 0.75: 'Pct_75'})
 .T # <1>
)

,Pct_25,Pct_50,Pct_75
Volume_Gal,0.79,2.38,2.77


In [240]:
(liquor_sales
 .loc[pd.col('Volume_Gal').ge(0) & pd.col('City').eq('DECORAH')]
 .filter(['Volume_Gal', 'Dollars'])
 .quantile(0.25, interpolation='linear')
 .rename({'Volume_Gal': 'Pct_25_Volume_Gal',
          'Dollars': 'Pct_25_Dollars'})
 .to_frame()
 .T
)

,Pct_25_Volume_Gal,Pct_25_Dollars
0.25,0.79,60.720001


In [241]:
(liquor_sales
 .loc[pd.col('Volume_Gal').ge(0) & pd.col('City').isin(['DECORAH', 'DUBUQUE'])]
 .astype({'Dollars': 'Float32'})
 .groupby('City', as_index=False)[['Volume_Gal', 'Dollars']]
 .quantile(0.25, interpolation='linear')
 .rename(columns={'Volume_Gal': 'Pct_25_Volume_Gal', 'Dollars': 'Pct_25_Dollars'})
)

,City,Pct_25_Volume_Gal,Pct_25_Dollars
0,DECORAH,0.79,60.720001
1,DUBUQUE,0.46,38.98


In [242]:
with pd.option_context('display.precision', 10):
    display(liquor_sales
    .loc[pd.col('Volume_Gal').ge(0)]
    .agg(Std_Pop_Volume_Gal=('Volume_Gal', lambda s: s.std(ddof=0)),
         Std_Samp_Volume_Gal=('Volume_Gal', lambda s: s.std(ddof=1)))
    )

,Volume_Gal
Std_Pop_Volume_Gal,8.6846913101
Std_Samp_Volume_Gal,8.6846914513


In [243]:
(liquor_sales
 .loc[pd.col('Volume_Gal').ge(0)]
 .assign(Avg_Volume_Gal=pd.col('Volume_Gal').mean(),
         Std_Dev=pd.col('Volume_Gal').std(ddof=0))
 .assign(Z_Score=(pd.col('Volume_Gal') - pd.col('Avg_Volume_Gal')) / pd.col('Std_Dev'))
 .filter(['City','Volume_Gal','Avg_Volume_Gal','Std_Dev','Z_Score'])
 .sort_values('Volume_Gal', ascending=False)
)

,City,Volume_Gal,Avg_Volume_Gal,Std_Dev,Z_Score
11378343,WEST DES MOINES,998.57,2.40437,8.684691,114.703631
3695699,CORALVILLE,998.57,2.40437,8.684691,114.703631
2340470,CORALVILLE,998.57,2.40437,8.684691,114.703631
3294542,CORALVILLE,998.57,2.40437,8.684691,114.703631
30527869,WEST DES MOINES,998.57,2.40437,8.684691,114.703631
...,...,...,...,...,...
3671027,CEDAR RAPIDS,0.0,2.40437,8.684691,-0.276851
31219690,COLUMBUS JUNCTION,0.0,2.40437,8.684691,-0.276851
28125647,WATERLOO,0.0,2.40437,8.684691,-0.276851
10276439,BURLINGTON,0.0,2.40437,8.684691,-0.276851


### Graphing to Find Anomalies Visually

In [244]:
to_plot = (liquor_sales
 .loc[pd.col('Volume_Gal').ge(0)]
 .groupby('Volume_Gal', as_index=False)
 .agg(Count=('Volume_Gal', 'size'))
 .sort_values('Volume_Gal')
 )
to_plot

,Volume_Gal,Count
0,0.0,651
1,0.01,585115
2,0.02,308874
3,0.03,130137
4,0.04,540
...,...,...
1910,972.42,1
1911,974.79,1
1912,993.02,1
1913,997.97,1


In [245]:
import hvplot.pandas
from bokeh.models import NumeralTickFormatter

(to_plot
 .loc[pd.col('Volume_Gal').ge(0)]
 .hvplot.bar(x='Volume_Gal', y='Count',
             xlabel='Volume',
             title='Distribution of liquor volumes (2012 to 2024)')
 .opts(yformatter=NumeralTickFormatter(format='0a'))
 )

:Bars   [Volume_Gal]   (Count)

In [246]:
(to_plot
 .loc[pd.col('Volume_Gal').ge(700)]
 .hvplot.bar(x='Volume_Gal', y='Count', height=400, xlabel='Volume')
 .opts(title='A zoomed-in view of the distribution of liquor volumes\nfocused on the highest magnitudes')
 )

:Bars   [Volume_Gal]   (Count)

In [247]:
to_plot = (liquor_sales
 .loc[pd.col('Volume_Gal').ge(0)]
 .groupby(['Volume_Gal','Dollars'], as_index=False)
 .agg(Counts=('Volume_Gal', 'size'))
 )
to_plot

,Volume_Gal,Dollars,Counts
0,0.0,7.4,352
1,0.0,7.5,41
2,0.0,8.45,207
3,0.0,13.83,23
4,0.0,14.25,16
...,...,...,...
93082,998.57,28188.0,14
93083,998.57,28944.0,3
93084,998.57,30888.0,1
93085,998.57,42940.800781,3


In [248]:
(to_plot
 .hvplot.scatter(x='Volume_Gal', y='Dollars', height=500,
                 xlabel='Volume', ylabel='Sales ($)', alpha=0.6,
                 yformatter=NumeralTickFormatter(format='0,0'),
                 title='Scatter plot of liquor volume and sales (2012 to 2024)')
 )

:Scatter   [Volume_Gal]   (Dollars)

In [249]:
to_plot = (liquor_sales
 .loc[pd.col('Volume_Gal').ge(0) & pd.col('County').eq('VAN BUREN')]
 .filter(['Volume_Gal'])
 .sort_values('Volume_Gal')
 )
to_plot

,Volume_Gal
2051019,0.0
2827763,0.0
9130265,0.01
31852936,0.01
31854411,0.01
...,...
23194968,13.87
21238748,13.87
11337218,13.87
24335112,13.87


In [250]:
(to_plot
 .hvplot.box(width=300, height=500,
             ylabel='Volume')
 .opts(title='Liquor volume (2012 to 2024)\nVan Buren county',
       fontsize={'ylabel':11, 'yticks': 11})
 )

:BoxWhisker   (Volume_Gal)

In [251]:
(liquor_sales
 .loc[pd.col('Volume_Gal').ge(0) & pd.col('County').eq('VAN BUREN')]
 .filter(['Volume_Gal'])
 .quantile([0.25, 0.5, 0.75], interpolation='linear')
 .rename({0.25: 'Percentile_25', 0.50: 'Median', 0.75: 'Percentile_75'})
 .T
 .assign(IQR=(pd.col('Percentile_75') - pd.col('Percentile_25')) * 1.5)
 .assign(Lower_Whisker=pd.col('Percentile_25') - pd.col('IQR'),
         Upper_Whisker=pd.col('Percentile_75') + pd.col('IQR'))
)

,Percentile_25,Median,Percentile_75,IQR,Lower_Whisker,Upper_Whisker
Volume_Gal,0.5,1.18,2.77,3.405,-2.905,6.175


In [252]:
to_plot = (liquor_sales
 .loc[pd.col('Volume_Gal').ge(0) & pd.col('County').eq('VAN BUREN')]
 .assign(Year=pd.col('Date').dt.year)
 .filter(['Year', 'Volume_Gal'])
 .sort_values(['Year', 'Volume_Gal'])
 )
to_plot

,Year,Volume_Gal
12264626,2012,0.08
22395146,2012,0.08
11970404,2012,0.1
22227244,2012,0.1
12087888,2012,0.13
...,...,...
31068659,2024,7.13
32212236,2024,7.13
32223449,2024,7.13
32223786,2024,7.13


In [253]:
(to_plot
 .astype({'Volume_Gal': 'float32'})
 .hvplot.box(y='Volume_Gal', by='Year', height=400,
             xlabel='', ylabel='Volume',
             title='Box plot of liquor volumes in Van Buren county by year')
 .opts(fontsize={'ylabel':10, 'ticks': 10})
 )

:BoxWhisker   [Year]   (Volume_Gal)

## Forms of Anomalies
### Anomalous Values

In [254]:
(liquor_sales
 .loc[pd.col('Dollars').gt(65_000)]
 .groupby('City', as_index=False)
 .agg(Count=('City', 'size'))
 )

,City,Count
0,DAVENPORT,1
1,DES MOINES,10
2,IOWA CITY,2
3,TEMPLETON,1
4,URBANDALE,20
5,WEST DES MOINES,3


In [255]:
(liquor_sales
 .loc[pd.col('Dollars') > 65_000]
 .groupby('Item', as_index=False)
 .agg(Count=('Item', 'size'))
 )

,Item,Count
0,ABSOLUT W/4-50MLS,1
1,BAILEY'S ORIGINAL IRISH CREAM,1
2,CROWN ROYAL,6
3,CROWN ROYAL CANADIAN WHISKY,2
4,CROWN ROYAL PEACH,1
5,CROWN ROYAL REGAL APPLE,1
6,CROWN ROYAL W/NFL JERSEY BAG,1
7,JACK DANIELS OLD #7 BLACK LBL,5
8,JAMESON,3
9,JOHNNIE WALKER BLUE,1


### Anomalous Counts or Frequencies

In [256]:
(liquor_sales
 .loc[pd.col('Volume_Gal').ge(0)]
 .assign(Transaction_Year=pd.col('Date').dt.to_period('Y').dt.to_timestamp())
 .groupby('Transaction_Year', as_index=False)
 .agg(Transactions=('Transaction_Year', 'size'))
 .sort_values('Transaction_Year')
)

,Transaction_Year,Transactions
0,2012-01-01,2080569
1,2013-01-01,2062145
2,2014-01-01,2096013
3,2015-01-01,2182757
4,2016-01-01,2278554
5,2017-01-01,2291253
6,2018-01-01,2355546
7,2019-01-01,2380326
8,2020-01-01,2614339
9,2021-01-01,2622689


In [257]:
to_plot = (liquor_sales
 .loc[pd.col('Volume_Gal').ge(0)]
 .assign(Transaction_Month=pd.col('Date').dt.to_period('M').dt.to_timestamp())
 .groupby('Transaction_Month', as_index=False)
 .agg(Transactions=('Transaction_Month', 'size'))
 .sort_values('Transaction_Month')
)
to_plot

,Transaction_Month,Transactions
0,2012-01-01,147684
1,2012-02-01,156583
2,2012-03-01,160434
3,2012-04-01,164840
4,2012-05-01,192146
...,...,...
151,2024-08-01,213749
152,2024-09-01,200336
153,2024-10-01,223504
154,2024-11-01,216874


In [258]:
(to_plot
 .hvplot.line(x='Transaction_Month', y='Transactions',
              xlabel='Month', height=400)
 .opts(yformatter=NumeralTickFormatter(format='0a'),
       title='Number of liquor transactions per month')
 )

:Curve   [Transaction_Month]   (Transactions)

In [259]:
df = (pd.read_csv(f'{data}/iowa_county_population.csv')
 .assign(County=pd.col('County').str.upper())
 )

population = (df
 .assign(Pop_Bucket=pd.qcut(pd.col('Population'), [0, 1/3, 2/3, 1],
                            labels=['Low', 'Medium', 'High']))
 )
population

,County,Population,Area_Sqr_Mile,Density,Pop_Bucket
0,WRIGHT,12714,580,22,Medium
1,WORTH,7292,400,18,Low
2,WOODBURY,107970,873,124,High
3,WINNESHIEK,19718,690,29,High
4,WINNEBAGO,10203,400,26,Low
...,...,...,...,...,...
94,AUDUBON,5591,443,13,Low
95,APPANOOSE,12083,497,24,Medium
96,ALLAMAKEE,14200,639,22,Medium
97,ADAMS,3659,423,9,Low


In [260]:
to_plot = (liquor_sales
 .loc[pd.col('Volume_Gal').ge(0)]
 .assign(Transaction_Month=pd.col('Date').dt.to_period('M').dt.to_timestamp())
 .filter(['County', 'Transaction_Month'])
 .merge(population.filter(['County','Pop_Bucket']), on='County', how='inner')
 .groupby(['Transaction_Month', 'Pop_Bucket'], as_index=False)
 .agg(Transactions=('Transaction_Month', 'size'))
 .sort_values('Transaction_Month')
 )
to_plot

,Transaction_Month,Pop_Bucket,Transactions
0,2012-01-01,Low,10682
1,2012-01-01,Medium,23754
2,2012-01-01,High,112944
3,2012-02-01,Low,11108
4,2012-02-01,Medium,23504
...,...,...,...
462,2024-11-01,Low,13794
464,2024-11-01,High,172325
466,2024-12-01,Medium,34033
465,2024-12-01,Low,14133


In [261]:
(to_plot
 .pivot(index='Transaction_Month', columns='Pop_Bucket', values='Transactions')
 .hvplot.line(x='Transaction_Month',
              y=['Low','Medium','High'],
              xlabel='Month',
              ylabel='Transactions',
              height=500,
              color=['#77767b','#c0bfbc','#1a1a1a'],
              legend='top_left',
              legend_opts={'title':''},
              legend_cols=3)
 .opts(title='Number of transactions per month\n(split by population bucket)',
       yformatter=NumeralTickFormatter(format='0a'))
 )

:NdOverlay   [Pop_Bucket]
   :Curve   [Transaction_Month]   (value)

### Anomalies from the Absence of Data

In [262]:
(liquor_sales
 .loc[pd.col('Volume_Gal').ge(700)
      & pd.col('County').ne('EL PASO')
      & pd.col('County').notna()]
 .assign(Next_Date=lambda d: d.groupby('City')['Date'].shift(-1),
         Latest=lambda d: d.groupby('City')['Date'].transform('max'))
 .assign(Gap=pd.col('Next_Date') - pd.col('Date'))
 .assign(Days_Since_Latest=(pd.Timestamp(2024, 12, 31) - pd.col('Latest')).dt.days)
 .groupby(['City','Days_Since_Latest'], as_index=False)
 .agg(Transactions=('City', 'size'),
      Avg_Gap=('Gap', lambda s: s.mean().days),
      Max_Gap=('Gap', lambda s: s.max().days))
)

,City,Days_Since_Latest,Transactions,Avg_Gap,Max_Gap
0,AMES,1070,2,490.0,490.0
1,ANKENY,223,8,80.0,623.0
2,CARROLL,4470,1,NaN,NaN
3,CEDAR RAPIDS,363,4,289.0,2938.0
4,CORALVILLE,36,61,-1.0,1623.0
5,COUNCIL BLUFFS,1000,3,-1489.0,-560.0
6,DAVENPORT,140,21,17.0,1372.0
7,DENISON,3505,2,-385.0,-385.0
8,DES MOINES,796,12,-92.0,2565.0
9,DUBUQUE,907,2,658.0,658.0


## Handling Anomalies
### Removal

In [263]:
(liquor_sales
 .loc[pd.col('Volume_Gal').ge(0)]
 .head()
 )

,Invoice,Date,Store_Num,Store,Address,City,Zip_Code,Location,County_Num,County,...,Item_Num,Item,Pack,Bottle_Vol_ml,Cost,Retail,Bottles,Dollars,Volume_Ltr,Volume_Gal
0,INV-71393800027,2024-06-19,5187,KWIK STOP LIQUOR & GROCERIES AMES,125 6TH ST,AMES,50010.0,POINT (-93.611679978 42.027185979),NaN,STORY,...,89199,JOSE CUERVO ESPECIAL REPOSADO SQUARE,12,375,6.53,9.8,12,117.599998,4.5,1.18
1,INV-71387600015,2024-06-19,3045,BRITT FOOD CENTER,8 2ND ST NW,BRITT,50423.0,POINT (-93.802630964 43.09902101),NaN,HANCOCK,...,42079,UV CAKE,12,1000,7.5,11.25,2,22.5,2.0,0.52
2,INV-71392700079,2024-06-19,4129,CYCLONE LIQUORS,626 LINCOLN WAY,AMES,50010.0,POINT (-93.618248037 42.021326987),NaN,STORY,...,69637,DR MCGILLICUDDYS CHERRY,12,1000,11.25,16.879999,12,202.559998,12.0,3.17
3,INV-71420400062,2024-06-20,5889,SUPER QUICK LIQUOR / WDM,1800 22ND ST,WEST DES MOINES,50266.0,POINT (-93.736764041 41.599012004),NaN,POLK,...,64000,ABSENTE,12,750,24.629999,36.950001,2,73.900002,1.5,0.39
4,INV-71346200003,2024-06-18,5413,LIL' CHUBS CORNER STOP,712 S OAK ST,INWOOD,51240.0,POINT (-96.430479985 43.302142016),NaN,LYON,...,10550,BLACK VELVET TOASTED CARAMEL,12,750,6.0,9.0,1,9.0,0.75,0.19


In [264]:
(liquor_sales
 .agg(Avg_Volume=('Volume_Gal', 'mean'),
      Avg_Volume_Adjusted=('Volume_Gal', lambda s: s[s.ge(0)].mean()))
 )

,Volume_Gal
Avg_Volume,2.40326
Avg_Volume_Adjusted,2.40437


In [265]:
(liquor_sales
 .loc[pd.col('County').eq('SCOTT')]
 .agg(Avg_Volume=('Volume_Gal', 'mean'),
      Avg_Volume_Adjusted=('Volume_Gal', lambda s: s[s.ge(0)].mean()))
 )

,Volume_Gal
Avg_Volume,2.689077
Avg_Volume_Adjusted,2.690168


### Replacement with Alternate Values

In [266]:
(liquor_sales
 .groupby('Store', as_index=False)
 .size()
 .sort_values('size')
 .tail(10)
 )

,Store,size
1702,HY-VEE FOOD STORE #5 / CEDAR RAPIDS,133072
1818,HY-VEE WINE AND SPIRITS / IOWA CITY,141550
1555,HY-VEE #4 / WDM,141890
118,BENZ DISTRIBUTING,142636
1558,HY-VEE #7 / CEDAR RAPIDS,152449
1808,HY-VEE WINE AND SPIRITS / BETTENDORF,159464
1724,HY-VEE FOOD STORE / CEDAR FALLS,162727
945,"CENTRAL CITY LIQUOR, INC.",187547
944,CENTRAL CITY 2,219737
1549,HY-VEE #3 / BDI / DES MOINES,251312


In [267]:
import numpy as np

(liquor_sales
 .assign(Store=pd.col('Store').where(
         ~pd.col('Store').str.contains('HY-VEE', na=False),
         'HY-VEE'))
 .groupby('Store', as_index=False)
 .size()
 .sort_values('size')
 .tail(10)
)

,Store,size
2027,LOT-A-SPIRITS,86356
2317,QUICK SHOP / CLEAR LAKE,88724
955,"CHARLIE'S WINE AND SPIRITS,",99741
1444,HAPPY'S WINE & SPIRITS,103747
3021,WILKIE LIQUORS,105234
1067,CYCLONE LIQUORS,127086
118,BENZ DISTRIBUTING,142636
945,"CENTRAL CITY LIQUOR, INC.",187547
944,CENTRAL CITY 2,219737
1537,HY-VEE,9758981


In [268]:
(liquor_sales
 .filter(['Volume_Gal'])
 .quantile([0.95, 0.05], interpolation='linear')
 .rename({0.95: 'Percentile_95', 0.05: 'Percentile_05'})
 .T
)

,Percentile_95,Percentile_05
Volume_Gal,5.55,0.1


In [269]:
(liquor_sales
 .assign(Percentile_95=pd.col('Volume_Gal').quantile(0.95, interpolation='linear'),
         Percentile_05=pd.col('Volume_Gal').quantile(0.05, interpolation='linear'))
 .assign(Volume_Winsorized=pd.col('Volume_Gal').clip(
         lower=pd.col('Percentile_05'),
         upper=pd.col('Percentile_95')))
 .filter(['Date','City','Volume_Gal','Volume_Winsorized'])
)

,Date,City,Volume_Gal,Volume_Winsorized
0,2024-06-19,AMES,1.18,1.18
1,2024-06-19,BRITT,0.52,0.52
2,2024-06-19,AMES,3.17,3.17
3,2024-06-20,WEST DES MOINES,0.39,0.39
4,2024-06-18,INWOOD,0.19,0.19
...,...,...,...,...
32593198,2024-06-19,OSKALOOSA,3.17,3.17
32593199,2024-06-19,WINDSOR HEIGHTS,2.77,2.77
32593200,2024-06-19,EMMETSBURG,0.01,0.1
32593201,2024-06-19,SPIRIT LAKE,0.01,0.1


### Rescaling

In [270]:
to_plot = (liquor_sales
 .assign(Volume_Gal=pd.col('Volume_Gal').round(1))
 .loc[pd.col('Volume_Gal').gt(0)]
 .assign(Log_Volume=np.log10(pd.col('Volume_Gal')))
 .groupby(['Volume_Gal','Log_Volume'], as_index=False)
 .agg(Transactions=('Volume_Gal', 'size'))
)
to_plot

,Volume_Gal,Log_Volume,Transactions
0,0.1,-1.0,921216
1,0.2,-0.69897,2409885
2,0.3,-0.522879,1072624
3,0.4,-0.39794,2467227
4,0.5,-0.30103,891821
...,...,...,...
1183,972.4,2.987845,1
1184,974.8,2.988916,1
1185,993.0,2.996949,1
1186,998.0,2.999131,1


In [271]:
(to_plot
 .astype({'Volume_Gal': 'float32'})
 .hvplot.bar(x='Volume_Gal', y='Transactions',
             xlabel='Volume', ylabel='Transactions',
             title='Distribution of positive volumes for liquor transactions\n(2012 to 2024)')
 .opts(yformatter=NumeralTickFormatter(format='0a'))
 )

:Bars   [Volume_Gal]   (Transactions)

In [272]:
(to_plot
 .astype({'Volume_Gal': 'float32'})
 .hvplot.bar(x='Log_Volume', y='Transactions',
             xlabel='Volume (log scale)', ylabel='Transactions',
             title='Distribution of positive volumes for liquor transactions\n(2012 to 2024)')
 .opts(yformatter=NumeralTickFormatter(format='0a'))
 )

:Bars   [Log_Volume]   (Transactions)

# Experiment Analysis
## The Dataset

In [273]:
import pandas as pd

students = pd.read_csv(f'{data}/students.csv', parse_dates=True)
students.sample(3)

,User_ID,Enroll_Date,Country
39025,40025,2024-01-21,Germany
2756,3756,2024-01-03,Canada
7101,8099,2024-01-07,Canada


In [274]:
actions = pd.read_csv(f'{data}/actions.csv', parse_dates=True)
actions.sample(3)

,User_ID,Action,Date
94006,33740,Email opt-in,2024-01-20
91142,28880,Email opt-in,2024-01-16
39837,54351,Tutorial complete,2024-01-28


In [275]:
payments = pd.read_csv(f'{data}/payments.csv', parse_dates=['Pay_Date'])
payments.sample(3)

,User_ID,Pay_Date,Amount
631,9621,2024-01-10,10.0
9746,100998,2024-02-22,10.0
5985,58304,2024-02-04,25.0


In [276]:
experiments = pd.read_csv(f'{data}/experiments.csv', parse_dates=['Date'])
experiments.sample(3)

,Experiment,User_ID,Date,Variant
27295,Welcome tutorial,28295,2024-01-16,Control
53486,Welcome tutorial,54486,2024-01-28,Control
33730,Welcome tutorial,34730,2024-01-20,Control


## Types of Experiments
### Experiments with Binary Outcomes: The Chi-Squared Test

In [277]:
contingency_df = (experiments
 .loc[pd.col('Experiment') == 'Welcome tutorial']
 .merge(actions.query('Action == "Tutorial complete"'),
        on='User_ID',
        how='left')
 .groupby('Variant', as_index=False)
 .agg(Total=('User_ID', 'size'),
      Completed=('Action', 'count'))
 .assign(Not_Completed=pd.col('Total') - pd.col('Completed'))
 .filter(['Variant', 'Completed', 'Not_Completed'])
)
contingency_df

,Variant,Completed,Not_Completed
0,Control,36268,13629
1,Variant 1,38280,11995


In [278]:
contingency_table = (contingency_df
 .melt(id_vars='Variant', var_name='Status', value_name='Count')
 .pipe(lambda d: pd.crosstab(
     d['Variant'],
     d['Status'],
     values=d['Count'],
     aggfunc='sum',
     margins=True,
     margins_name='Total'))
 .rename(columns={'Completed': 'Yes', 'Not_Completed': 'No'})
)
contingency_table

Status,Yes,No,Total
Variant,,,
Control,36268,13629,49897
Variant 1,38280,11995,50275
Total,74548,25624,100172


In [279]:
last_row = contingency_table.index[-1]

(contingency_table
 .style
 .set_caption('Completed tutorial?')
 .format({'Yes': '{:,.0f}', 'No': '{:,.0f}', 'Total': '{:,.0f}'})
 .set_properties(subset=['Total'], **{'font-weight': 'bold'})
 .apply(lambda row: ['font-weight: bold' if row.name == last_row else '' for _ in row], axis=1)
 .set_table_styles([{'selector': 'th.col_heading.level0.col3',
                      'props': [('font-weight', 'bold')]}])
)

Status,Yes,No,Total
Variant,,,
Control,"36,268","13,629","49,897"
Variant 1,"38,280","11,995","50,275"
Total,"74,548","25,624","100,172"


In [280]:
(experiments
 .query('Experiment == "Welcome tutorial"')
 .merge(actions.query('Action == "Tutorial complete"'),
        on='User_ID',
        how='left')
 .groupby('Variant', as_index=False)
 .agg(Total_Cohorted=('User_ID', 'size'),
      Completions=('Action', 'count'))
 .assign(Pct_Completed=pd.col('Completions') / pd.col('Total_Cohorted'))
)

,Variant,Total_Cohorted,Completions,Pct_Completed
0,Control,49897,36268,0.726857
1,Variant 1,50275,38280,0.761412


### Experiments with Continuous Outcomes: The t-Test

In [281]:
(experiments
 .query('Experiment == "Welcome tutorial"')
 .merge(payments, on='User_ID', how='left')
 .assign(Amount=pd.col('Amount').fillna(0))
 .groupby(['Variant', 'User_ID'], as_index=False)
 .agg(Amount=('Amount', 'sum'))
 .groupby('Variant', as_index=False)
 .agg(Total_Cohorted=('User_ID', 'count'),
      Mean_Amount=('Amount', 'mean'),
      Std_Amount=('Amount', 'std'))
)

,Variant,Total_Cohorted,Mean_Amount,Std_Amount
0,Control,49897,3.781218,18.940378
1,Variant 1,50275,3.687589,19.220194


In [282]:
(experiments
 .query('Experiment == "Welcome tutorial"')
 .merge(actions.query('Action == "Tutorial complete"'),
        on='User_ID',
        how='inner')
 .merge(payments, on='User_ID', how='left')
 .assign(Amount=pd.col('Amount').fillna(0))
 .groupby(['Variant', 'User_ID'], as_index=False)
 .agg(Amount=('Amount', 'sum'))
 .groupby('Variant', as_index=False)
 .agg(Total_Cohorted=('User_ID', 'count'),
      Mean_Amount=('Amount', 'mean'),
      Std_Amount=('Amount', 'std'))
)

,Variant,Total_Cohorted,Mean_Amount,Std_Amount
0,Control,36268,5.202146,22.048994
1,Variant 1,38280,4.843091,21.899284


## Outliers

In [283]:
(experiments
 .query('Experiment == "Welcome tutorial"')
 .merge(actions.query('Action == "Tutorial complete"'),
        on='User_ID',
        how='inner')
 .merge(payments, on='User_ID', how='left')
 .assign(Amount=pd.col('Amount').fillna(0))
 .groupby(['Variant', 'User_ID'], as_index=False)
 .agg(Amount=('Amount', 'sum'))
 .groupby('Variant', as_index=False)
 .agg(Total_Cohorted=('User_ID', 'count'),
      Buyers=('Amount', lambda s: (s > 0).sum()))
 .assign(Pct_Buyers=pd.col('Buyers') / pd.col('Total_Cohorted'))
)

,Variant,Total_Cohorted,Buyers,Pct_Buyers
0,Control,36268,4988,0.137532
1,Variant 1,38280,4981,0.130120


### Time Boxing

In [284]:
(experiments
 .query('Experiment == "Welcome tutorial"')
 .merge(payments, on='User_ID', how='left')
 .assign(Valid_Amount=pd.col('Amount').where(
     pd.col('Pay_Date') <= pd.col('Date') + pd.Timedelta(days=7), 0))
 .groupby(['Variant', 'User_ID'], as_index=False)
 .agg(Amount=('Valid_Amount', 'sum'))
 .groupby('Variant', as_index=False)
 .agg(Total_Cohorted=('User_ID', 'count'),
      Mean_Amount=('Amount', 'mean'),
      Std_Amount=('Amount', 'std'))
)

,Variant,Total_Cohorted,Mean_Amount,Std_Amount
0,Control,49897,1.369382,5.766338
1,Variant 1,50275,1.351688,5.612986


### Pre-/Post-Analysis

In [285]:
import numpy as np

(students
 .loc[pd.col('Enroll_Date').between('2024-01-13', '2024-02-09')]
 .assign(Variant=lambda d: np.where(d['Enroll_Date'] <= '2024-01-26', 'pre', 'post'))
 .merge(actions.query('Action == "Email opt-in"'), on='User_ID', how='left')
 .groupby('Variant', as_index=False)
 .agg(Cohorted=('User_ID', 'nunique'),
      Opted_In=('Action', 'count'),
      Days=('Enroll_Date', 'nunique'))
 .assign(Pct_Optin=pd.col('Opted_In') / pd.col('Cohorted'))
)

,Variant,Cohorted,Opted_In,Days,Pct_Optin
0,post,27617,11220,14,0.406271
1,pre,24662,14489,14,0.587503


### Natural Experiment Analysis

In [286]:
(students
 .loc[pd.col('Country').isin(['United States', 'Canada'])]
 .merge(payments, on='User_ID', how='left')
 .assign(Amount=pd.col('Amount').fillna(0))
 .groupby(['Country', 'User_ID'], as_index=False)
 .agg(Amount=('Amount', 'sum'))
 .groupby('Country', as_index=False)
 .agg(Total_Cohorted=('User_ID', 'count'),
      Purchasers=('Amount', lambda s: (s > 0).sum()))
 .assign(Pct_Purchased=pd.col('Purchasers') / pd.col('Total_Cohorted'))
)

,Country,Total_Cohorted,Purchasers,Pct_Purchased
0,Canada,20179,5011,0.248327
1,United States,45012,4958,0.110148


# Creating Complex Datasets for Analysis
## Organizing Computations
### Using Intermediate Variables

In [287]:
store_df = (liquor_sales
 .groupby('Store', as_index=False)
 .agg(Dollars=('Dollars', 'sum'))
 .assign(Category=None, Vendor=None)
 .filter(['Store', 'Category', 'Vendor', 'Dollars'])
)

category_df = (liquor_sales
 .groupby('Category', as_index=False)
 .agg(Dollars=('Dollars', 'sum'))
 .assign(Store=None, Vendor=None)
 .filter(['Store', 'Category', 'Vendor', 'Dollars'])
)

vendor_df = (liquor_sales
 .groupby('Vendor', as_index=False)
 .agg(Dollars=('Dollars', 'sum'))
 .assign(Category=None, Store=None)
 .filter(['Store', 'Category', 'Vendor', 'Dollars'])
)

pd.concat([store_df, category_df, vendor_df])

,Store,Category,Vendor,Dollars
0,"""DOUBLE """"D"""" LIQUOR STORE""",None,None,416938.25
1,'DA BOOZE BARN / WEST BEND,None,None,633370.875
2,10TH HOLE INN & SUITE / GIFT SHOP,None,None,35399.371094
3,16TH AVE BP / CEDAR RAPIDS,None,None,26917.820312
4,173 CRAFT DISTILLERY,None,None,15768.0
...,...,...,...,...
626,None,None,"WYCH DR. DISTILLING, LLC",14340.0
627,None,None,XAMAY GLOBAL SPIRITS LLC,1012.799988
628,None,None,YAHARA BAY DISTILLERS INC,114261.773438
629,None,None,"YAHARA BAY DISTILLERS, INC",434237.75


In [288]:
(pd.concat([
    (liquor_sales
     .groupby('Store', as_index=False)
     .agg(Dollars=('Dollars', 'sum'))
     .assign(Category=None, Vendor=None)
     .filter(['Store', 'Category', 'Vendor', 'Dollars'])
    ),
    (liquor_sales
     .groupby('Category', as_index=False)
     .agg(Dollars=('Dollars', 'sum'))
     .assign(Store=None, Vendor=None)
     .filter(['Store', 'Category', 'Vendor', 'Dollars'])
    ),
    (liquor_sales
     .groupby('Vendor', as_index=False)
     .agg(Dollars=('Dollars', 'sum'))
     .assign(Category=None, Store=None)
     .filter(['Store', 'Category', 'Vendor', 'Dollars'])
    )])
)

,Store,Category,Vendor,Dollars
0,"""DOUBLE """"D"""" LIQUOR STORE""",None,None,416938.25
1,'DA BOOZE BARN / WEST BEND,None,None,633370.875
2,10TH HOLE INN & SUITE / GIFT SHOP,None,None,35399.371094
3,16TH AVE BP / CEDAR RAPIDS,None,None,26917.820312
4,173 CRAFT DISTILLERY,None,None,15768.0
...,...,...,...,...
626,None,None,"WYCH DR. DISTILLING, LLC",14340.0
627,None,None,XAMAY GLOBAL SPIRITS LLC,1012.799988
628,None,None,YAHARA BAY DISTILLERS INC,114261.773438
629,None,None,"YAHARA BAY DISTILLERS, INC",434237.75


In [289]:
(pd.concat([
    liquor_sales
    .groupby(col, as_index=False)
    .agg(Dollars=('Dollars', 'sum'))
    .assign(**{c: None for c in ['Store', 'Category', 'Vendor'] if c != col})
    .filter(['Store', 'Category', 'Vendor', 'Dollars'])
    for col in ['Store', 'Category', 'Vendor']])
)

,Store,Category,Vendor,Dollars
0,"""DOUBLE """"D"""" LIQUOR STORE""",None,None,416938.25
1,'DA BOOZE BARN / WEST BEND,None,None,633370.875
2,10TH HOLE INN & SUITE / GIFT SHOP,None,None,35399.371094
3,16TH AVE BP / CEDAR RAPIDS,None,None,26917.820312
4,173 CRAFT DISTILLERY,None,None,15768.0
...,...,...,...,...
626,None,None,"WYCH DR. DISTILLING, LLC",14340.0
627,None,None,XAMAY GLOBAL SPIRITS LLC,1012.799988
628,None,None,YAHARA BAY DISTILLERS INC,114261.773438
629,None,None,"YAHARA BAY DISTILLERS, INC",434237.75


In [290]:
import pandas as pd

new_cols = ['Title', 'Author', 'Call_Num', 'ID', 'Item', 'Status',
 'Lang', 'Age', 'Library', 'Date']

library_checkouts = (pd.read_parquet(f'{data}/brisbane_library.parquet')
 .pipe(lambda d: d.rename(columns=dict(zip(d.columns, new_cols))))
 )

top_2_checkouts_library = (library_checkouts
 .loc[pd.col('ID') != "REFRESH"] # <1>
 .groupby('Library', as_index=False).size()
 .sort_values('size')
 ['Library']
 .tail(2)
 .tolist()
 )
top_2_checkouts_library

['SBK', 'CDE']

In [291]:
(library_checkouts
 .loc[pd.col('Date').dt.year == 2024]
 .loc[pd.col('Library').isin(top_2_checkouts_library)]
 .filter(['Author'])
 .drop_duplicates()
 )

,Author
2318985,"Penman, Sharon Kay,"
2318987,NaN
2319002,"Scrace, Carolyn,"
2319003,"Goldman, Ken,"
2319039,"Adams, Wes,"
...,...
2898701,"Turnbull, Bryn,"
2898736,"Macfarlane, Robert,"
2898737,"Szymanski, Kris,"
2898738,"Colloca, Silvia."


## Managing Dataset Size and Privacy Concerns
### Sampling with Sample

In [292]:
clothing_store_sales = (pd.read_csv(f'{data}/sample_sales.csv', parse_dates=['date'])
 .drop(columns='Account Name')
 .rename(columns=lambda c: c.title().replace(' ','_'))
 .rename(columns=dict(Account_Number='Customer_ID'))
 )

(clothing_store_sales
 .sample(500)
 )

,Customer_ID,Sku,Category,Quantity,Unit_Price,Ext_Price,Date
330,823932,QS-76400,Sweater,12,63.35,760.20,2014-01-22 09:20:45
883,949740,XX-25746,Hat,19,35.26,669.94,2013-10-27 02:38:00
763,227563,LK-02338,Sweater,10,60.47,604.70,2014-02-08 17:25:35
12,185412,AE-95093,Socks,10,10.93,109.30,2014-07-11 17:20:23
642,585977,AE-95093,Socks,19,67.25,1277.75,2014-06-27 17:55:08
...,...,...,...,...,...,...,...
435,282942,ZC-07383,Sweater,3,86.57,259.71,2014-04-03 18:50:51
717,957458,ZC-07383,Sweater,20,29.20,584.00,2014-01-26 22:43:55
380,729939,LK-02338,Sweater,5,13.73,68.65,2014-07-27 11:01:12
632,221265,IC-59308,Socks,18,64.86,1167.48,2014-05-17 11:25:26


In [293]:
(clothing_store_sales
 .sample(frac=1/2)
 )

,Customer_ID,Sku,Category,Quantity,Unit_Price,Ext_Price,Date
888,374609,GS-95639,Socks,19,29.26,555.94,2014-05-27 04:25:31
121,40033,GS-95639,Socks,2,26.45,52.90,2014-07-30 20:10:37
899,423670,LK-02338,Sweater,3,41.40,124.20,2014-01-18 03:19:31
59,145616,ZC-07383,Sweater,14,25.75,360.50,2013-12-07 23:40:46
670,141320,ZC-07383,Sweater,2,34.87,69.74,2013-11-12 09:37:07
...,...,...,...,...,...,...,...
908,215844,QS-76400,Sweater,5,24.97,124.85,2014-08-15 13:32:45
946,486457,LK-02338,Sweater,15,11.51,172.65,2014-02-17 19:35:47
884,870007,ZC-07383,Sweater,18,41.53,747.54,2014-09-20 02:42:43
86,555482,IC-59308,Socks,16,12.41,198.56,2014-03-27 08:01:56


In [294]:
(clothing_store_sales
 ['Customer_ID'].sample(frac=1/2)
 )

372    385821
771    435915
783    742526
548    655756
778     64410
        ...  
610    104104
65     115558
42     879335
23     172430
829    335539
Name: Customer_ID, Length: 500, dtype: int64

In [295]:
(clothing_store_sales
 .sample(frac=1/2, random_state=42)
 )

,Customer_ID,Sku,Category,Quantity,Unit_Price,Ext_Price,Date
521,830285,QS-76400,Sweater,11,23.36,256.96,2014-09-28 16:01:29
737,68120,QS-76400,Sweater,8,70.48,563.84,2014-05-14 10:53:31
740,740581,IC-59308,Socks,7,95.05,665.35,2013-12-19 20:21:33
660,719100,GS-95639,Socks,10,51.65,516.50,2013-12-03 23:39:39
411,372790,FT-50146,Sweater,14,17.25,241.50,2013-10-24 11:10:18
...,...,...,...,...,...,...,...
178,726918,HX-24728,Hat,18,32.67,588.06,2014-05-25 04:13:27
444,50180,XX-25746,Hat,17,91.37,1553.29,2014-09-01 12:18:52
416,911360,GS-95639,Socks,13,14.35,186.55,2014-02-14 18:54:41
870,195565,GS-95639,Socks,14,31.34,438.76,2014-07-02 11:06:29


### Reducing Dimensionality

In [296]:
import numpy as np

county_population = (pd.read_csv(f'{data}/iowa_county_population.csv')
 .filter(['County','Population'])
 .assign(County=pd.col('County').str.upper())
 )

(liquor_sales
 .merge(county_population, on='County', how='inner')
 .assign(County_Group=lambda d: np.where(
         d['County'].isin(['POLK','LINN','SCOTT','JOHNSON','BLACK HAWK']),
         d['County'], 'Other'))
 .groupby('County_Group', as_index=False)
 .agg(Count=('County_Group','count'))
 .sort_values('Count', ascending=False)
 )

,County_Group,Count
3,Other,17267124
4,POLK,5719277
2,LINN,2559418
5,SCOTT,1838667
0,BLACK HAWK,1709832
1,JOHNSON,1505461


In [297]:
(liquor_sales
 .loc[pd.col('County') != 'EL PASO'] # <1>
 .merge(liquor_sales
        .loc[pd.col('County') != 'EL PASO']
        .groupby('County', as_index=False)
        .agg(Invoice=('Invoice', 'nunique'))
        .assign(Rank=pd.col('Invoice').rank(method='min', ascending=False)),
        on='County')
 .assign(County_Group=lambda d: np.where(d['Rank'] <= 5, d['County'], 'Other'))
 .groupby('County_Group', as_index=False)
 .agg(Transactions=('Invoice_x', 'nunique'))
 .sort_values('Transactions', ascending=False)
)

,County_Group,Transactions
3,Other,17267124
4,POLK,5719277
2,LINN,2559418
5,SCOTT,1838667
0,BLACK HAWK,1709832
1,JOHNSON,1505461


In [298]:
(library_checkouts
 .loc[pd.col('ID') != "REFRESH"]
 .groupby('ID', as_index=False)
 .agg(Checkouts=('ID','size'))
 .assign(Two_Checkouts_Flag=pd.col('Checkouts') >= 2)
 .groupby('Two_Checkouts_Flag', as_index=False)
 .agg(Items=('Checkouts', 'size'))
)

,Two_Checkouts_Flag,Items
0,False,542858
1,True,826199


In [299]:
(library_checkouts
 .loc[pd.col('ID') != "REFRESH"]
 .groupby('ID', as_index=False)
 .agg(Checkouts=('ID','size'))
 .assign(Checkouts_Level=lambda d: np.select(
     [d['Checkouts'] >= 10, d['Checkouts'] >= 2],
     ['10+', '2 - 9'],
     default='1'))
 .groupby('Checkouts_Level', as_index=False)
 .agg(Items=('Checkouts', 'size'))
)

,Checkouts_Level,Items
0,1,542858
1,10+,7266
2,2 - 9,818933


### PII and Data Privacy

In [300]:
emails_df = pd.DataFrame({
    'Email': ['eleonora@memail.com', 'mae@memail.com',
    'grace@memail.com', 'mae@memail.com', 'elsa@memail.com'],
})
emails_df

,Email
0,eleonora@memail.com
1,mae@memail.com
2,grace@memail.com
3,mae@memail.com
4,elsa@memail.com


In [301]:
(emails_df
 .sort_values('Email') # <1>
 .assign(Anonymized=pd.col('Email').rank(method='dense'))
)

,Email,Anonymized
0,eleonora@memail.com,1.0
4,elsa@memail.com,2.0
2,grace@memail.com,3.0
1,mae@memail.com,4.0
3,mae@memail.com,4.0


In [302]:
(emails_df
 .assign(Anonymized=lambda d: pd.util.hash_pandas_object(d['Email']))
)

,Email,Anonymized
0,eleonora@memail.com,9700798426834329500
1,mae@memail.com,1722221469318118824
2,grace@memail.com,6641798629316670494
3,mae@memail.com,3743392290627978901
4,elsa@memail.com,7047493077789107928


# Reporting Your Analysis
## Writing to Excel for Stakeholders
### Writing a Basic Excel File

In [303]:
import pandas as pd

candidates = pd.read_parquet(f'{data}/candidates.parquet')
candidates

,Series,ID,Name,Full_Name,Background,Age
0,16,100,Aaron,Aaron Willis,Flight operations instructor,38
1,16,101,Akeem,Akeem Bundu-Kamara,Financial firm strategy manager,29
2,16,102,Akshay,Akshay Thakrar,Digital marketing agency owner,28
3,16,103,Alex,Alex Short,Commercial cleaning company owner,27
4,19,104,Amber-Rose,Amber-Rose Badrudin,Convenience store owner,24
...,...,...,...,...,...,...
81,18,181,Steve,Steve Darken,Management consultant,37
82,15,182,Thomas,Thomas Skinner,"Owner, Pillow Company",28
83,18,183,Tre,Tre Lowe,Musician & wellness entrepreneur,48
84,17,184,Victoria,Victoria Goulbourne,Online sweet business owner,28


In [304]:
(candidates
 .loc[pd.col('Age') > pd.col('Age').mean()]
 .filter(['Full_Name','Age'])
 .sort_values('Age', ascending=False)
 .rename(columns={'Full_Name': 'Candidate Name'})
 .to_excel(f'{data}/processed/candidates_above_average_age.xlsx', index=False)
 )

### Designing Stakeholder-Friendly Sheets

In [305]:
with pd.ExcelWriter(f'{data}/processed/candidates_above_average_age.xlsx', engine='xlsxwriter') as writer:
    report = (candidates
     .loc[pd.col('Age') > pd.col('Age').mean()]
     .filter(['Full_Name','Age'])
     .sort_values('Age', ascending=False)
     .rename(columns={'Full_Name': 'Candidate Name'})
     )

    report.to_excel(writer,
                    sheet_name='Candidates Age', # <1>
                    index=False,
                    autofilter=True,
                    freeze_panes=(1, 0)) # <2>

    workbook = writer.book
    worksheet = writer.sheets['Candidates Age']
    header_format = workbook.add_format({'bold': True}) # <3>
    for col_num, value in enumerate(report.columns):
        worksheet.write(0, col_num, value, header_format)

## Conditional Formatting in Excel
### Highlighting Performance

In [306]:
performance = pd.read_parquet(f'{data}/performance.parquet')
performance

,ID,Series,Week,Result
0,100,16,1,IN
1,101,16,1,BR
2,102,16,1,LOSE
3,103,16,1,IN
4,104,19,1,IN
...,...,...,...,...
1027,181,18,12,NaN
1028,182,15,12,NaN
1029,183,18,12,NaN
1030,184,17,12,NaN


In [307]:
to_report = (performance
 .loc[pd.col('Series') == 15]
 .merge(candidates.filter(['ID','Name']), on='ID', how='left')
 .drop(columns='ID')
 .assign(Days=lambda d: d.groupby('Name')['Result'].transform('count')) # <1>
 .drop_duplicates('Name')
 .filter(['Name','Days'])
 )
to_report

,Name,Days
0,Carina,12
1,Dean,10
2,Iasha,6
3,Jemelin,7
4,Kenna,2
5,Lewis,11
6,Lottie,11
7,Lubna,4
8,Marianne,9
9,Pamela,11


In [308]:
with pd.ExcelWriter(f'{data}/processed/days_in_the_show.xlsx', engine='xlsxwriter') as writer:
    to_report.to_excel(writer, sheet_name='Sheet1', index=False,
                       autofilter=True, freeze_panes=(1, 0))

    workbook = writer.book
    worksheet = writer.sheets['Sheet1']

    red_format = workbook.add_format({'bg_color': '#F44769'}) # <1>
    green_format = workbook.add_format({'bg_color': '#44F69D'})
    header_format = workbook.add_format({'bg_color': '#2C2D2D', 'bold': True, 'font_color': '#FFFFFF'}) # <2>

    days_col = to_report.columns.get_loc('Days')
    n_rows = len(to_report)

    worksheet.conditional_format(1, days_col, n_rows, days_col,
                                  {'type': 'cell', 'criteria': '<', 'value': 6, 'format': red_format})
    worksheet.conditional_format(1, days_col, n_rows, days_col,
                                  {'type': 'cell', 'criteria': '>=', 'value': 6, 'format': green_format})

    for col_num, value in enumerate(to_report.columns):
        worksheet.write(0, col_num, value, header_format)

### Color Scales and Data Bars

In [309]:
with pd.ExcelWriter(f'{data}/processed/days_in_the_show_color_scale.xlsx', engine='xlsxwriter') as writer:
    sorted_report = to_report.sort_values('Days') # <1>
    sorted_report.to_excel(writer, sheet_name='Sheet1', index=False,
                           autofilter=True, freeze_panes=(1, 0))

    workbook = writer.book
    worksheet = writer.sheets['Sheet1']
    header_format = workbook.add_format({'bold': True})

    days_col = sorted_report.columns.get_loc('Days')
    n_rows = len(sorted_report)

    worksheet.conditional_format(1, days_col, n_rows, days_col, {'type': '3_color_scale'})

    for col_num, value in enumerate(sorted_report.columns):
        worksheet.write(0, col_num, value, header_format)

In [310]:
with pd.ExcelWriter(f'{data}/processed/days_in_the_show_data_bar.xlsx', engine='xlsxwriter') as writer:
    to_report.to_excel(writer, sheet_name='Sheet1', index=False,
                       autofilter=True, freeze_panes=(1, 0))

    workbook = writer.book
    worksheet = writer.sheets['Sheet1']
    header_format = workbook.add_format({'bold': True})

    days_col = to_report.columns.get_loc('Days')
    n_rows = len(to_report)

    worksheet.conditional_format(1, days_col, n_rows, days_col,
                                  {'type': 'data_bar', 'bar_color': '#E990F8'})

    for col_num, value in enumerate(to_report.columns):
        worksheet.write(0, col_num, value, header_format)

### Advanced Conditional Formatting

In [311]:
(performance
 .filter(['Result'])
 .drop_duplicates()
 )

,Result
0,IN
1,BR
2,LOSE
21,FIRED
23,WIN
107,NaN
160,LEFT
953,RUNNER-UP
959,HIRED


In [312]:
import numpy as np

(performance
 .loc[pd.col('Series').eq(15)]
 .merge(candidates[['ID', 'Name']], on='ID', how='left')
 .drop(columns='ID')
 .assign(Week=pd.col('Week').astype(str))
 .assign(_count=lambda d: d.groupby('Name')['Result'].transform('count'))
 .sort_values('_count', ascending=False)
 .drop(columns='_count')
 .pivot_table(index=['Series', 'Name'], columns='Week', values='Result',
              aggfunc='first', sort=False)
 .reset_index()
 .rename_axis(None, axis=1)
 .reindex(columns=['Series', 'Name', *map(str, range(1, 13))])
 .assign(index=lambda d: np.select([d['12'].eq('HIRED'),    # <1>
                                    d['12'].eq('RUNNER-UP')],
                                   [0, 1],
                                   default=np.arange(len(d))))
 .sort_values('index')
 .drop(columns='index')
)

,Series,Name,1,2,3,4,5,6,7,8,9,10,11,12
0,15,Carina,IN,WIN,IN,IN,IN,WIN,IN,IN,IN,WIN,IN,HIRED
1,15,Scarlett,WIN,IN,IN,IN,IN,LOSE,IN,IN,IN,IN,IN,RUNNER-UP
2,15,Lottie,IN,IN,BR,IN,IN,BR,IN,WIN,IN,IN,FIRED,NaN
3,15,Pamela,IN,IN,WIN,IN,IN,IN,IN,IN,BR,LOSE,FIRED,NaN
4,15,Lewis,LOSE,IN,IN,IN,IN,IN,IN,IN,LOSE,BR,FIRED,NaN
5,15,Dean,BR,BR,IN,IN,IN,BR,WIN,IN,WIN,FIRED,NaN,NaN
6,15,Marianne,IN,IN,IN,BR,WIN,IN,LOSE,BR,FIRED,NaN,NaN,NaN
7,15,Thomas,IN,IN,IN,LOSE,IN,IN,IN,BR,FIRED,NaN,NaN,NaN
8,15,Ryan-Mark,IN,BR,IN,IN,BR,IN,BR,FIRED,NaN,NaN,NaN,NaN
9,15,Jemelin,IN,IN,IN,IN,LOSE,IN,FIRED,NaN,NaN,NaN,NaN,NaN


In [313]:
to_report = (performance
 .loc[pd.col('Series').between(15, 19)]
 .merge(candidates[['ID', 'Name']], on='ID', how='left')
 .drop(columns='ID')
 .assign(Week=pd.col('Week').astype(str))
 .assign(_count=lambda d: d.groupby(['Series', 'Name'])['Result'].transform('count'))
 .sort_values(['Series', '_count'], ascending=[True, False])
 .drop(columns='_count')
 .pivot_table(index=['Series', 'Name'], columns='Week', values='Result',
              aggfunc='first', sort=False)
 .reset_index()
 .rename_axis(None, axis=1)
 .reindex(columns=['Series', 'Name', *map(str, range(1, 13))])
 .assign(_index=lambda d: d.groupby('Series').cumcount())
 .assign(_index=lambda d: np.select([d['12'].eq('HIRED'),
                                     d['12'].eq('RUNNER-UP')],
                                    [0, 1],
                                    default=d['_index']))
 .sort_values(['Series', '_index'])
 .drop(columns='_index')
)
to_report

,Series,Name,1,2,3,4,5,6,7,8,9,10,11,12
0,15,Carina,IN,WIN,IN,IN,IN,WIN,IN,IN,IN,WIN,IN,HIRED
1,15,Scarlett,WIN,IN,IN,IN,IN,LOSE,IN,IN,IN,IN,IN,RUNNER-UP
2,15,Lewis,LOSE,IN,IN,IN,IN,IN,IN,IN,LOSE,BR,FIRED,NaN
3,15,Lottie,IN,IN,BR,IN,IN,BR,IN,WIN,IN,IN,FIRED,NaN
4,15,Pamela,IN,IN,WIN,IN,IN,IN,IN,IN,BR,LOSE,FIRED,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
81,19,Jana,IN,IN,IN,BR,LEFT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
82,19,Nadia,IN,BR,IN,FIRED,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
83,19,Carlo,BR,IN,FIRED,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
84,19,Aoibheann,IN,FIRED,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [314]:
week_cols = [str(i) for i in range(1, 13)] # <1>

result_colors = {
    'FIRED': {'bg_color': '#DC143C'}, # <2>
    'HIRED': {'bg_color': '#00FF7F'},
    'WIN': {'bg_color': '#6495ED'},
    'LOSE': {'bg_color': '#FFA500'},
    'RUNNER-UP': {'bg_color': '#40E0D0'},
    'LEFT': {'bg_color': '#696969', 'font_color': '#FFFFFF'},
    'BR': {'bg_color': '#FFC0CB'},
}

with pd.ExcelWriter(f'{data}/processed/performance_grid.xlsx', engine='xlsxwriter') as writer:
    to_report.to_excel(writer, sheet_name='Sheet1', index=False, freeze_panes=(1, 0))

    workbook = writer.book
    worksheet = writer.sheets['Sheet1']

    header_format = workbook.add_format({'bold': True})
    name_format = workbook.add_format({'bold': True}) # <3>
    n_rows = len(to_report)

    name_col = to_report.columns.get_loc('Name')
    worksheet.set_column(name_col, name_col, None, name_format)

    for col in week_cols:
        col_idx = to_report.columns.get_loc(col)
        for value, fmt_props in result_colors.items():
            worksheet.conditional_format(1, col_idx, n_rows, col_idx,
                {'type': 'cell', 'criteria': '==', 'value': f'"{value}"', # <2>
                 'format': workbook.add_format(fmt_props)})

    for col_num, value in enumerate(to_report.columns):
        worksheet.write(0, col_num, value, header_format)

## Writing Reports as Text Files
### Generating Summary Narratives from Pandas

In [315]:
tasks = (pd.read_parquet(f'{data}/tasks.parquet')
 .filter(['Series','Episode','Task','Boardroom'])
 )
tasks

,Series,Episode,Task,Boardroom
0,19,Week 1: Austria Tours,"Plan a tour of Innsbrück, Austria. Most profit...","Anisa, Carlo, Dean, Emma R, Keir, Melica"
1,19,Week 2: Virtual Pop Star,To create a virtual pop star and licence it ou...,"Amber-Rose, Aoibheann, Nadia"
2,19,Week 3: Discount Buying,Purchase a list of nine items associated with ...,"Max, Carlo, Liam"
3,19,Week 4: Crops to Cash,Create products using assigned produce to sell...,"Nadia , Jana, Keir"
4,19,Week 5: Easter Eggs,Crate an Easter egg that will stand out in a c...,None
5,19,Week 6: Turkey Corporate Hospitality,Host a corporate hospitality event in Turkey. ...,"Chisola , Frederick, Jonny"
6,19,Week 7: Kids' Banking,Create a banking app and accompanying money bo...,"Anisa , Dean , Keir , Melica , Mia"
7,19,Week 8: Hot Sauce,"Develop a new hot sauce, create an advertising...","Parallel: Amber-Rose , Dean , Melica . Ascenda..."
8,19,Week 9: TV Selling,Sell products during a primetime slot on a sho...,"Max , Chisola, Dean"
9,19,Week 10: Fashion,Launch an environmentally conscious fashion br...,"Jordan , Anisa , Mia"


In [316]:
most_in_br = (tasks
 .filter(['Series','Boardroom'])
 .assign(Boardroom=pd.col('Boardroom').str.split(', ')) # <1>
 .explode('Boardroom') # <2>
 .groupby(['Series','Boardroom'], as_index=False).size() # <3>
 .loc[pd.col('size') > 3] # <4>
 .sort_values(['Series','size'], ascending=[True, False])
 .rename(columns={'Boardroom': 'Name'})
 )
most_in_br

,Series,Name,size
0,15,Dean,4
6,15,Marianne,4
18,16,Akshay,6
25,16,Kathryn,5
58,18,Phil,4
63,18,Virdi,4


In [317]:
series_list = (most_in_br['Series']
 .drop_duplicates()
 .sort_values()
 .astype(str)
 .pipe(lambda s: ', '.join(s)) # <1>
 .rsplit(', ', 1) # <2>
)
series_list = ' and '.join(series_list) # <3>
series_list

'15, 16 and 18'

In [318]:
most_in_br_name = (most_in_br
 .loc[pd.col('size') == pd.col('size').max()]
 ['Name']
 .iloc[0]
 )
most_in_br_name

'Akshay'

In [319]:
(most_in_br
 .rename(columns={'size': 'Times in Boardroom'})
 .assign(Series=lambda d: d['Series'].where(
         d.groupby('Series').cumcount().eq(0),'')) # <1>
)

,Series,Name,Times in Boardroom
0,15,Dean,4
6,,Marianne,4
18,16,Akshay,6
25,,Kathryn,5
58,18,Phil,4
63,,Virdi,4


In [320]:
from datetime import date

today = date.today()
print(f"Report Generated on {today.strftime('%d/%m/%Y')}\n")

print(f'The candidates who were in the boardroom the most are from series {series_list}.')
print(f'{most_in_br_name} has been in the boardroom most times.\n')
print(f'The table below shows the No. of times each of the {len(most_in_br)} candidates were in the boardrom.\n')

report_table = (most_in_br
                .rename(columns={'size': 'Times in Boardroom'})
                .assign(Series=lambda d: d['Series'].where(
                        d.groupby('Series').cumcount().eq(0),''))
                )
print(report_table.to_string(index=False))

Report Generated on 01/10/2026

The candidates who were in the boardroom the most are from series 15, 16 and 18.
Akshay has been in the boardroom most times.

The table below shows the No. of times each of the 6 candidates were in the boardrom.

Series     Name  Times in Boardroom
    15     Dean                   4
       Marianne                   4
    16   Akshay                   6
        Kathryn                   5
    18     Phil                   4
          Virdi                   4


In [321]:
from datetime import date

with open(f'{data}/processed/boardroom_summary.txt', 'w+', encoding='utf-8') as fp:

    today = date.today()
    print(f"Report Generated on {today.strftime('%d/%m/%Y')}\n", file=fp)

    print(
        f'The candidates who were in the boardroom the most are from series {series_list}.',
        file=fp
    )

    print(
        f'{most_in_br_name} has been in the boardroom most times.\n',
        file=fp
    )

    print(
        f'The table below shows the No. of times each of the {len(most_in_br)} candidates were in the boardroom.\n',
        file=fp
    )

    report_table = (most_in_br
                    .rename(columns={'size': 'Times in Boardroom'})
                    .assign(Series=lambda d: d['Series'].where(
                            d.groupby('Series').cumcount().eq(0),''))
                    )
    print(report_table.to_string(index=False), file=fp)

In [322]:
pm_performance = (performance
 .loc[pd.col('Result').isin(['WIN','LOSE','FIRED','LEFT'])] # <1>
 )
pm_performance

,ID,Series,Week,Result
2,102,16,1,LOSE
7,107,19,1,LOSE
21,121,17,1,FIRED
22,122,19,1,FIRED
23,123,19,1,WIN
...,...,...,...,...
920,160,15,11,FIRED
922,162,18,11,FIRED
940,180,16,11,FIRED
943,183,18,11,FIRED


In [323]:
the_tasks = (tasks
 .filter(['Series','Episode'])
 .assign(Week=pd.col('Episode').str.extract(r'(\d+)', expand=False).astype('int8'))
 .assign(Task=pd.col('Episode').str.split(': ').str[1])
 .drop(columns='Episode')
 )
the_tasks

,Series,Week,Task
0,19,1,Austria Tours
1,19,2,Virtual Pop Star
2,19,3,Discount Buying
3,19,4,Crops to Cash
4,19,5,Easter Eggs
5,19,6,Turkey Corporate Hospitality
6,19,7,Kids' Banking
7,19,8,Hot Sauce
8,19,9,TV Selling
9,19,10,Fashion


In [324]:
teams = pd.read_parquet(f'{data}/teams.parquet')

name_replace = {
    'Rose':'Amber-Rose','Joe':'Joseph',
    'Mark':'Ryan-Mark',
}

project_managers = (teams
 .melt(id_vars='Series', var_name='Week', value_name='Team')
 .assign(Week=pd.col('Week').astype('int8'),
         Team=pd.col('Team').str.replace('Project Manager','PM', regex=False))
 .assign(PM=pd.col('Team').str.extract(r'(\w+)\s+\(PM\)', expand=False))
 .loc[pd.col('PM').notna()]
 .assign(Team=pd.col('Team').str.split(':').str[0],
         PM=pd.col('PM').replace(name_replace)) # <1>
 .merge(the_tasks, on=['Series','Week'], how='left')
 .merge(candidates.filter(['ID','Name']), left_on='PM', right_on='Name', how='left')
 .merge(pm_performance, on=['ID','Series','Week'], how='left')
 .drop(columns='ID')
 )
project_managers

,Series,Week,Team,PM,Task,Name,Result
0,15,1,Men,Lewis,South Africa,Lewis,LOSE
1,16,1,Men,Akshay,Cruise Ship,Akshay,LOSE
2,17,1,Men,Joseph,Antigua Tourism,Joseph,WIN
3,18,1,Men,Virdi,Highlands Corporate Away Day,Virdi,LOSE
4,19,1,Forest,Anisa,Austria Tours,Anisa,LOSE
5,15,2,Men,Kenna,Ice Lollies,Kenna,FIRED
6,16,2,Men,Aaron,Toothbrush,Aaron,LOSE
7,17,2,Men,Bradley,Bao Buns,Bradley,LOSE
8,18,2,Men,Phil,Cheesecakes,Phil,LOSE
9,19,2,Forest,Anisa,Virtual Pop Star,Anisa,WIN


In [325]:
total_pm = project_managers['PM'].nunique()
total_pm

37

In [326]:
most_pm = (project_managers
 .groupby('PM', as_index=False).size()
 .loc[pd.col('size') == pd.col('size').max()]
 )
most_pm

,PM,size
6,Bradley,3


In [327]:
(project_managers
 .loc[pd.col('PM') == 'Bradley']
 .assign(Task='- ' + pd.col('Task'))
 .filter(['Task'])
 )

,Task
7,- Bao Buns
22,- Electric Motorbike Advertising
42,- Male Beauty


In [328]:
result_replace = {
    'WIN':'  % of PMs who won',
    'LOSE':'  % of PMs who lost',
}

won_lost_pct = (project_managers
 .assign(Result=lambda d: np.where(d['Result'] != 'WIN', 'LOSE', d['Result']))
 .groupby('Result', as_index=False).size()
 .assign(Percent=pd.col('size') / pd.col('size').sum() * 100)
 .drop(columns='size')
 .assign(Result=pd.col('Result').replace(result_replace))
 )
won_lost_pct

,Result,Percent
0,% of PMs who lost,68.0
1,% of PMs who won,32.0


In [329]:
lost_fired = (project_managers
 .loc[pd.col('Result').isin(['FIRED','LOSE'])] # <1>
 .groupby('Result', as_index=False).size()
 .assign(Percent=(pd.col('size') / pd.col('size').sum() * 100).round(2))
 .loc[pd.col('Result') == 'FIRED'] # <2>
 ['Percent']
 .tolist()[0]
 )
lost_fired

39.39

In [330]:
print(f"Report Generated on {today.strftime('%d/%m/%Y')}\n")
print(f'The total number of candidates who were project managers (PMs) is {total_pm}.\n')
print(f"{most_pm['PM'].iloc[0]} was PM the most holding the role {most_pm['size'].iloc[0]} times.")
print('He was PM for the following tasks:')
print(project_managers
      .loc[pd.col('PM') == 'Bradley']
      .assign(Task='- ' + pd.col('Task'))
      .filter(['Task'])
      .to_string(index=False, header=False)
     )

print('\nThe following are the percentages for PMs who won and lost the tasks:')
print(won_lost_pct.to_string(index=False, header=False))

print(f'\nThe proportion of PMs who lost and were fired is {lost_fired}')

Report Generated on 01/10/2026

The total number of candidates who were project managers (PMs) is 37.

Bradley was PM the most holding the role 3 times.
He was PM for the following tasks:
                      - Bao Buns
- Electric Motorbike Advertising
                   - Male Beauty

The following are the percentages for PMs who won and lost the tasks:
  % of PMs who lost 68.0
   % of PMs who won 32.0

The proportion of PMs who lost and were fired is 39.39
